# Model Development and Internal Validation — Clean Unweighted Workflow

This notebook is the consolidated primary-analysis workflow. It can be run in order from top to bottom, or with **Run All** after restarting the kernel.

- All four candidate models are fixed to `class_weight=None`.
- Hyperparameters are tuned using inner-fold PR-AUC.
- Classification thresholds are derived only from inner training data.
- Every fitting/search operation uses `n_jobs=1` for stability.
- Existing `unweighted_...csv` checkpoints are preserved and resumed automatically.
- The earlier mixed-weight analysis and endpoint-audit cells are intentionally excluded from this execution notebook.

Do not delete the five `unweighted_...csv` checkpoint files unless you deliberately want to discard all completed evaluations and restart at evaluation 1.


In [ ]:
# ============================================================
# Import libraries and define reproducibility settings
# ============================================================

from pathlib import Path
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import set_config


# ------------------------------------------------------------
# General settings
# ------------------------------------------------------------

RANDOM_SEED = 2026

np.random.seed(RANDOM_SEED)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    200
)

set_config(
    display="diagram"
)

sns.set_theme(
    style="whitegrid",
    context="notebook"
)


# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_DIRECTORY = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

PROCESSED_DATA_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Data"
    / "Processed"
)

RESULTS_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Results"
)

TABLES_DIRECTORY = (
    RESULTS_DIRECTORY
    / "Tables"
)

FIGURES_DIRECTORY = (
    RESULTS_DIRECTORY
    / "Figures"
)

MODEL_RESULTS_DIRECTORY = (
    RESULTS_DIRECTORY
    / "Model_Results"
)

FINAL_MODELING_FILE = (
    PROCESSED_DATA_DIRECTORY
    / "final_modeling_dataset.xlsx"
)


for directory in [
    TABLES_DIRECTORY,
    FIGURES_DIRECTORY,
    MODEL_RESULTS_DIRECTORY
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Verify required input
# ------------------------------------------------------------

if not FINAL_MODELING_FILE.exists():
    raise FileNotFoundError(
        "The final modelling dataset was not found:\n"
        f"{FINAL_MODELING_FILE}"
    )


# ------------------------------------------------------------
# Record software versions
# ------------------------------------------------------------

software_versions = pd.DataFrame({
    "Software": [
        "Python",
        "pandas",
        "NumPy",
        "SciPy",
        "scikit-learn",
        "Matplotlib",
        "Seaborn"
    ],
    "Version": [
        platform.python_version(),
        pd.__version__,
        np.__version__,
        scipy.__version__,
        sklearn.__version__,
        matplotlib.__version__,
        sns.__version__
    ]
})


print("MODEL-DEVELOPMENT ENVIRONMENT INITIALIZED")
print("-----------------------------------------")
print("Random seed:", RANDOM_SEED)
print("Python executable:", sys.executable)
print("Input file:", FINAL_MODELING_FILE)

print("\nSoftware versions:")
display(software_versions)

In [ ]:
# ============================================================
# Load and verify the final modelling dataset
# ============================================================

analysis_data = pd.read_excel(
    FINAL_MODELING_FILE
)


# ------------------------------------------------------------
# Define variables in their exact Excel-file order
# ------------------------------------------------------------

expected_file_columns = [
    "Age_at_Diagnosis",
    "Marital_Status",
    "Nationality",
    "Menopause_Status",
    "Breast_Cancer_Family_History",
    "Primary_Site_Group",
    "Laterality",
    "Histology_Group",
    "Tumor_Grade",
    "Tumor_Size_mm",
    "Summary_Stage",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Surgery",
    "Chemotherapy",
    "Hormone_Therapy",
    "Recurrence"
]


continuous_predictors = [
    "Age_at_Diagnosis",
    "Tumor_Size_mm"
]


categorical_predictors = [
    "Marital_Status",
    "Nationality",
    "Menopause_Status",
    "Breast_Cancer_Family_History",
    "Primary_Site_Group",
    "Laterality",
    "Histology_Group",
    "Tumor_Grade",
    "Summary_Stage",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Surgery",
    "Chemotherapy",
    "Hormone_Therapy"
]


# Preserve the predictor order used in the final Excel file.
candidate_predictors = [
    column
    for column in expected_file_columns
    if column != "Recurrence"
]


outcome_variable = "Recurrence"


# ------------------------------------------------------------
# Structural validation
# ------------------------------------------------------------

assert analysis_data.shape == (1370, 18), (
    "Expected 1,370 patients and 18 variables, but found "
    f"{analysis_data.shape[0]:,} patients and "
    f"{analysis_data.shape[1]} variables."
)

assert analysis_data.columns.tolist() == expected_file_columns, (
    "The variables or their order differ from the verified "
    "final modelling dataset.\n\n"
    f"Expected:\n{expected_file_columns}\n\n"
    f"Observed:\n{analysis_data.columns.tolist()}"
)

assert len(candidate_predictors) == 17, (
    "Expected exactly 17 candidate predictors."
)

assert len(continuous_predictors) == 2, (
    "Expected two continuous predictors."
)

assert len(categorical_predictors) == 15, (
    "Expected 15 categorical predictors."
)

assert set(candidate_predictors) == set(
    continuous_predictors + categorical_predictors
), (
    "The continuous and categorical predictor lists do not "
    "reconcile with the 17 candidate predictors."
)


# ------------------------------------------------------------
# Outcome validation
# ------------------------------------------------------------

observed_outcome_counts = (
    analysis_data[outcome_variable]
    .value_counts(dropna=False)
    .to_dict()
)


expected_outcome_counts = {
    "No": 1297,
    "Yes": 73
}


assert observed_outcome_counts == expected_outcome_counts, (
    "The recurrence distribution differs from the verified "
    "final dataset.\n"
    f"Expected: {expected_outcome_counts}\n"
    f"Observed: {observed_outcome_counts}"
)

assert (
    analysis_data[outcome_variable]
    .isna()
    .sum()
    == 0
), "Missing recurrence outcomes were detected."


# ------------------------------------------------------------
# Predictor validation
# ------------------------------------------------------------

assert (
    analysis_data["Age_at_Diagnosis"]
    .isna()
    .sum()
    == 0
), "Unexpected missing values were found in age."

assert (
    analysis_data["Tumor_Size_mm"]
    .isna()
    .sum()
    == 174
), "The tumor-size missing count differs from the expected value."

assert (
    analysis_data[
        categorical_predictors
    ]
    .isna()
    .sum()
    .sum()
    == 0
), "Standard missing values were found in categorical predictors."

assert (
    analysis_data.duplicated().sum()
    == 0
), "Completely duplicated records were detected."


# Check for blank categorical strings.
blank_categorical_values = (
    analysis_data[
        categorical_predictors
    ]
    .apply(
        lambda column: (
            column.astype(str)
            .str.strip()
            .eq("")
            .sum()
        )
    )
)


assert (
    blank_categorical_values.sum()
    == 0
), "Blank categorical predictor values were detected."


# ------------------------------------------------------------
# Create predictor matrix and binary outcome
# ------------------------------------------------------------

X = analysis_data[
    candidate_predictors
].copy()


y = (
    analysis_data[outcome_variable]
    .map({
        "No": 0,
        "Yes": 1
    })
    .astype(int)
)


assert X.shape == (1370, 17), (
    "The predictor matrix has an unexpected shape."
)

assert y.shape[0] == 1370, (
    "The outcome vector has an unexpected length."
)

assert y.value_counts().to_dict() == {
    0: 1297,
    1: 73
}, "Binary outcome encoding is incorrect."


# ------------------------------------------------------------
# Create an internal identifier for held-out predictions
# ------------------------------------------------------------

patient_id = pd.Series(
    np.arange(1, len(analysis_data) + 1),
    index=analysis_data.index,
    name="Modeling_Patient_ID"
)


# ------------------------------------------------------------
# Display verification summary
# ------------------------------------------------------------

dataset_verification = pd.DataFrame({
    "Measure": [
        "Patients",
        "Candidate predictors",
        "Continuous predictors",
        "Categorical predictors",
        "Recurrence events",
        "Non-recurrence observations",
        "Tumor-size missing values",
        "Completely duplicated records"
    ],
    "Result": [
        len(analysis_data),
        len(candidate_predictors),
        len(continuous_predictors),
        len(categorical_predictors),
        int(y.sum()),
        int((y == 0).sum()),
        int(
            analysis_data["Tumor_Size_mm"]
            .isna()
            .sum()
        ),
        int(
            analysis_data.duplicated()
            .sum()
        )
    ]
})


print("FINAL MODELLING DATASET LOADED AND VERIFIED")
print("-------------------------------------------")

display(dataset_verification)

print("\nPredictor matrix shape:", X.shape)
print("Outcome vector shape:", y.shape)

print(
    "Event percentage:",
    f"{y.mean() * 100:.2f}%"
)

In [ ]:
# ============================================================
# Estimate encoded model dimensionality
# ============================================================

categorical_dimension_records = []


for predictor in categorical_predictors:

    observed_categories = sorted(
        analysis_data[predictor]
        .astype(str)
        .unique()
        .tolist()
    )

    number_of_categories = len(
        observed_categories
    )

    # Reference-category encoding produces k - 1 coefficients.
    number_of_dummy_coefficients = (
        number_of_categories - 1
    )

    categorical_dimension_records.append({
        "Predictor": predictor,
        "Number_of_Categories": number_of_categories,
        "Reference_Coded_Coefficients": (
            number_of_dummy_coefficients
        ),
        "Categories": ", ".join(
            observed_categories
        )
    })


categorical_dimension_summary = pd.DataFrame(
    categorical_dimension_records
)


number_of_continuous_coefficients = len(
    continuous_predictors
)

number_of_categorical_coefficients = int(
    categorical_dimension_summary[
        "Reference_Coded_Coefficients"
    ].sum()
)

total_predictor_coefficients = (
    number_of_continuous_coefficients
    + number_of_categorical_coefficients
)

total_parameters_with_intercept = (
    total_predictor_coefficients + 1
)

number_of_events = int(
    y.sum()
)

events_per_predictor_coefficient = (
    number_of_events
    / total_predictor_coefficients
)


print("CATEGORICAL ENCODING DIMENSION")
print("------------------------------")

display(categorical_dimension_summary)


model_dimension_summary = pd.DataFrame({
    "Measure": [
        "Original candidate predictors",
        "Continuous coefficients",
        "Categorical coefficients",
        "Total predictor coefficients",
        "Intercept",
        "Total parameters including intercept",
        "Recurrence events",
        "Events per predictor coefficient"
    ],
    "Result": [
        len(candidate_predictors),
        number_of_continuous_coefficients,
        number_of_categorical_coefficients,
        total_predictor_coefficients,
        1,
        total_parameters_with_intercept,
        number_of_events,
        round(
            events_per_predictor_coefficient,
            2
        )
    ]
})


print("\nMODEL DIMENSION SUMMARY")
print("-----------------------")

display(model_dimension_summary)


assert total_predictor_coefficients > 17, (
    "Expected categorical encoding to produce more "
    "coefficients than the number of original predictors."
)

assert number_of_events == 73, (
    "The expected 73 recurrence events were not found."
)

## Prespecified Model-Development and Validation Protocol

### Prediction objective

The objective is to predict whether breast cancer recurrence is recorded
within five years of diagnosis using information available after initial
treatment.

The outcome does not represent confirmed recurrence-free survival because
the registry extract does not contain dates of last contact, death, loss to
follow-up, or censoring. Patients classified as negative therefore have no
recurrence recorded within the defined endpoint window; they cannot
necessarily be confirmed as continuously recurrence-free for five years.

No survival model, Kaplan-Meier analysis, median follow-up estimate, or
number-at-risk table will be produced because the required censoring and
follow-up information is unavailable.

### Candidate predictors

The complete model contains 17 clinically selected candidate predictors.
Predictors were selected before model fitting and were not chosen using
univariable p-values.

Reference-category encoding produces 38 predictor coefficients from only
73 recurrence events. Consequently, an ordinary unpenalized regression
model containing all coefficients will not be treated as a primary model.

### Candidate models

Four candidate models will be evaluated under the same internal-validation
framework:

1. Clinically prespecified ridge logistic regression.
2. Elastic-net logistic regression using all 17 candidate predictors.
3. Random forest using all 17 candidate predictors.
4. Gradient boosting using all 17 candidate predictors.

The clinically prespecified ridge logistic-regression model will include:

- Age at diagnosis
- Tumor size
- Tumor grade
- Summary stage
- Estrogen receptor status
- HER2 status
- Surgery
- Chemotherapy
- Hormone therapy

This model provides a clinically interpretable comparator with fewer
predictors. Progesterone receptor status is not included in the core model
because of its strong relationship with estrogen receptor status, but it
remains available to the full elastic-net and machine-learning models.

The comparison is intentionally limited to four model families because the
cohort contains only 73 recurrence events. The study does not claim to
propose a new machine-learning algorithm. Its contribution is the rigorous
development and internal validation of recurrence-prediction models using
regional real-world clinical data.

### Nested internal validation

Internal validation will use repeated nested stratified cross-validation:

- Outer validation: five stratified folds repeated ten times.
- Inner tuning: four stratified folds.
- Total outer-fold evaluations: 50.
- Random seed: 2026.
- Every patient will appear in one held-out outer fold during each
  repetition.
- Hyperparameter tuning and model selection will occur only within the
  corresponding outer-training data.
- No training-set performance will be presented as evidence of model
  generalizability.

The primary model-selection metric within the inner cross-validation will
be average precision, representing the area under the precision-recall
curve. This metric is prioritized because recurrence occurs in only 5.33%
of patients. ROC-AUC will be treated as an important secondary
discrimination measure.

### Fold-specific preprocessing

All preprocessing operations that learn information from the data will be
performed separately within each training fold.

For continuous predictors:

- Tumor size will be imputed using the median calculated from the
  corresponding training fold.
- A tumor-size missingness indicator will be included.
- Continuous predictors will be standardized when required by the model.
- Imputation and scaling parameters will never be estimated using held-out
  observations.

For categorical predictors:

- Unknown will remain an explicit category.
- Categorical predictors will be one-hot encoded.
- Previously unseen categories in a held-out fold will be ignored safely
  during transformation.
- Reference-category handling will be documented for regression models.

All preprocessing will be implemented within model pipelines to prevent
information leakage.

### Class imbalance

Class weighting will be the primary strategy for addressing the low
recurrence prevalence. Unweighted and class-balanced alternatives will be
evaluated as hyperparameter options within the inner cross-validation.

Synthetic oversampling will not be used in the primary analysis. Applying
ordinary SMOTE after one-hot encoding could generate fractional or
clinically implausible combinations of categorical predictors. If a
mixed-data oversampling method is examined as a sensitivity analysis, it
will be applied only to outer-training data and within the inner
cross-validation pipeline.

Accuracy will not be used as the primary measure of model quality because
a model predicting no recurrence for every patient would achieve high
accuracy in this imbalanced cohort.

### Probability thresholds

Threshold-independent measures will be calculated directly from held-out
predicted probabilities.

For threshold-dependent measures, the classification threshold for each
outer fold will be selected using predictions generated entirely within
the corresponding outer-training data. No held-out outer-fold outcomes
will be used to select thresholds.

Performance at the conventional probability threshold of 0.50 will also
be reported as a secondary analysis. The procedure used to select the
training-derived threshold will be applied identically to all candidate
models.

### Performance measures

Model discrimination will be evaluated using:

- Area under the precision-recall curve
- Area under the receiver operating characteristic curve

Threshold-dependent performance will be evaluated using:

- Sensitivity
- Specificity
- Precision or positive predictive value
- Negative predictive value
- F1 score
- Balanced accuracy
- Confusion matrix

Probability accuracy and calibration will be evaluated using:

- Brier score
- Calibration intercept
- Calibration slope
- Calibration plot
- Comparison between observed and predicted recurrence prevalence

Performance measures will be calculated separately from the held-out
predictions produced during each outer repetition and summarized across
the ten repetitions.

### Confidence intervals and model comparisons

Performance estimates will be accompanied by 95% confidence intervals
obtained using 2,000 stratified patient-level bootstrap resamples of the
held-out predictions.

Repeated predictions belonging to the same patient will remain grouped
during bootstrap resampling. Within each bootstrap sample, the performance
measure will be calculated for every outer repetition and then averaged
across repetitions.

Candidate models will be compared using paired bootstrap differences in:

- Precision-recall AUC
- ROC-AUC
- Brier score

The same bootstrap patient sample will be used for both models in each
comparison. A model will not be considered superior solely because its
point estimate is numerically higher.

### Feature selection and model interpretation

No feature selection will be performed using the complete dataset.

For elastic-net logistic regression:

- Coefficients will be estimated separately within the outer-training
  folds.
- Nonzero coefficient frequencies will be summarized across outer fits.
- Predictor-group stability will be reported because categorical
  predictors generate multiple encoded coefficients.
- Standardized coefficient direction and magnitude will be examined.

For all candidate models:

- Permutation importance will be calculated using held-out outer-fold
  observations.
- Importance values will be summarized across folds and repetitions.
- Model-specific impurity importance from tree-based models will not be
  used as the primary cross-model explanation.
- Predictor importance will not be interpreted as a causal effect.

### Prespecified ablation analyses

The following predictor configurations will be evaluated:

1. Full model using all 17 candidate predictors.
2. Model without tumor grade, ER, PR, and HER2 status.
3. Model without surgery, chemotherapy, and hormone therapy.
4. Clinical disease model containing age, tumor size, histology, tumor
   grade, summary stage, ER, PR, and HER2 status.
5. Model without marital status and nationality.

The ablation analyses will use the same validation framework as the
primary models. Their purpose is to assess the incremental contribution
of clinically related predictor groups rather than to search for the
best-performing predictor combination.

### Sensitivity analyses

Planned sensitivity analyses will:

- Exclude patients with distant disease at diagnosis.
- Exclude the two records with inconsistent histology and summary-stage
  classifications.
- Remove grade and receptor predictors whose availability changed
  substantially over time.
- Evaluate 2019 as an exploratory temporal validation period.
- Examine the handling of recurrence recorded after the five-year
  endpoint.
- Examine approximate diagnosis or recurrence dates close to endpoint
  boundaries.
- Compare primary median tumor-size imputation with an alternative
  fold-specific imputation method, if supported by the available data.
- Consider mixed-data oversampling only as a secondary analysis.

The temporal validation analysis will be interpreted cautiously because
the 2019 cohort contains only eight recurrence events.

### Reporting and reproducibility

The model-development pipeline will be illustrated in a methodological
diagram showing outer validation, inner tuning, preprocessing, class
imbalance handling, threshold selection, and held-out evaluation.

The following will be saved and reported:

- Held-out patient-level predictions
- Fold and repetition identifiers
- Training-derived thresholds
- Selected hyperparameters
- Hyperparameter search spaces
- Random seeds
- Software and package versions
- Elastic-net selection frequencies
- Held-out permutation importance
- Calibration results
- Ablation and sensitivity-analysis results

Code will be made available where permitted. Patient-level clinical data
will not be made publicly available if prohibited by institutional
confidentiality and data-governance requirements. Any restrictions will
be stated explicitly.

Because no external validation cohort is available, all reported
performance will represent internal validation. Results will not be
presented as evidence of external generalizability, clinical readiness,
or suitability for deployment.

In [ ]:
# ============================================================
# Define and audit the outer cross-validation design
# ============================================================

from sklearn.model_selection import RepeatedStratifiedKFold


OUTER_FOLDS = 5
OUTER_REPEATS = 10
INNER_FOLDS = 4


outer_cross_validation = RepeatedStratifiedKFold(
    n_splits=OUTER_FOLDS,
    n_repeats=OUTER_REPEATS,
    random_state=RANDOM_SEED
)


outer_fold_audit_records = []


for split_number, (
    training_indices,
    validation_indices
) in enumerate(
    outer_cross_validation.split(X, y),
    start=1
):

    repetition_number = (
        (split_number - 1) // OUTER_FOLDS
    ) + 1

    fold_number = (
        (split_number - 1) % OUTER_FOLDS
    ) + 1

    y_training = y.iloc[
        training_indices
    ]

    y_validation = y.iloc[
        validation_indices
    ]

    outer_fold_audit_records.append({
        "Repetition": repetition_number,
        "Outer_Fold": fold_number,
        "Training_Patients": len(
            training_indices
        ),
        "Training_Events": int(
            y_training.sum()
        ),
        "Training_Non_Events": int(
            (y_training == 0).sum()
        ),
        "Validation_Patients": len(
            validation_indices
        ),
        "Validation_Events": int(
            y_validation.sum()
        ),
        "Validation_Non_Events": int(
            (y_validation == 0).sum()
        )
    })


outer_fold_audit = pd.DataFrame(
    outer_fold_audit_records
)


# ------------------------------------------------------------
# Validate the resampling structure
# ------------------------------------------------------------

expected_outer_splits = (
    OUTER_FOLDS * OUTER_REPEATS
)


assert len(outer_fold_audit) == expected_outer_splits, (
    "Unexpected number of outer validation splits."
)

assert (
    outer_fold_audit[
        "Validation_Events"
    ].min()
    > 0
), (
    "At least one validation fold contains no "
    "recurrence events."
)

assert (
    outer_fold_audit[
        "Training_Events"
    ].min()
    >= INNER_FOLDS
), (
    "There are insufficient training events for the "
    "specified inner cross-validation."
)

assert (
    outer_fold_audit[
        "Validation_Patients"
    ].sum()
    == len(analysis_data) * OUTER_REPEATS
), (
    "Every patient should appear once in validation "
    "during every repetition."
)

assert (
    outer_fold_audit[
        "Validation_Events"
    ].sum()
    == int(y.sum()) * OUTER_REPEATS
), (
    "The held-out recurrence-event counts do not "
    "reconcile across repetitions."
)

assert (
    outer_fold_audit[
        "Validation_Non_Events"
    ].sum()
    == int((y == 0).sum()) * OUTER_REPEATS
), (
    "The held-out non-event counts do not reconcile "
    "across repetitions."
)


# ------------------------------------------------------------
# Summarize the cross-validation design
# ------------------------------------------------------------

outer_event_summary = (
    outer_fold_audit[
        [
            "Training_Patients",
            "Training_Events",
            "Validation_Patients",
            "Validation_Events"
        ]
    ]
    .agg([
        "min",
        "max",
        "mean"
    ])
    .round(2)
)


print("OUTER CROSS-VALIDATION DESIGN")
print("-----------------------------")
print("Outer folds:", OUTER_FOLDS)
print("Outer repetitions:", OUTER_REPEATS)
print("Total outer splits:", expected_outer_splits)
print("Inner tuning folds:", INNER_FOLDS)

print("\nOUTER-FOLD SIZE AND EVENT SUMMARY")
display(outer_event_summary)

print("\nFIRST TEN OUTER SPLITS")
display(
    outer_fold_audit.head(10)
)

In [ ]:
# ============================================================
# Define leakage-safe preprocessing pipelines
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)


# ------------------------------------------------------------
# Define the clinically prespecified core predictors
# ------------------------------------------------------------

core_predictors = [
    "Age_at_Diagnosis",
    "Tumor_Size_mm",
    "Tumor_Grade",
    "Summary_Stage",
    "ER_Status",
    "HER2_Status",
    "Surgery",
    "Chemotherapy",
    "Hormone_Therapy"
]


core_continuous_predictors = [
    predictor
    for predictor in continuous_predictors
    if predictor in core_predictors
]


core_categorical_predictors = [
    predictor
    for predictor in categorical_predictors
    if predictor in core_predictors
]


assert len(core_predictors) == 9, (
    "Expected nine prespecified core predictors."
)

assert set(core_predictors) == set(
    core_continuous_predictors
    + core_categorical_predictors
), (
    "The core continuous and categorical predictors "
    "do not reconcile."
)


# ------------------------------------------------------------
# Create a reusable preprocessing constructor
# ------------------------------------------------------------

def create_preprocessor(
    selected_predictors,
    drop_reference_category
):
    """
    Construct a preprocessing pipeline without fitting it.

    Continuous predictors:
    - training-fold median imputation;
    - missingness indicator when missing values occur;
    - standardization.

    Categorical predictors:
    - one-hot encoding;
    - Unknown retained as an ordinary recorded category;
    - unseen held-out categories safely ignored.
    """

    selected_continuous = [
        predictor
        for predictor in continuous_predictors
        if predictor in selected_predictors
    ]

    selected_categorical = [
        predictor
        for predictor in categorical_predictors
        if predictor in selected_predictors
    ]


    continuous_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    categorical_pipeline = Pipeline(
        steps=[
            (
                "encoder",
                OneHotEncoder(
                    drop=(
                        "first"
                        if drop_reference_category
                        else None
                    ),
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )


    preprocessor = ColumnTransformer(
        transformers=[
            (
                "continuous",
                continuous_pipeline,
                selected_continuous
            ),
            (
                "categorical",
                categorical_pipeline,
                selected_categorical
            )
        ],
        remainder="drop",
        verbose_feature_names_out=True
    )


    return preprocessor


# ------------------------------------------------------------
# Regression preprocessors
# ------------------------------------------------------------

core_regression_preprocessor = create_preprocessor(
    selected_predictors=core_predictors,
    drop_reference_category=True
)


full_regression_preprocessor = create_preprocessor(
    selected_predictors=candidate_predictors,
    drop_reference_category=True
)


# ------------------------------------------------------------
# Tree-model preprocessor
# ------------------------------------------------------------

full_tree_preprocessor = create_preprocessor(
    selected_predictors=candidate_predictors,
    drop_reference_category=False
)


# ------------------------------------------------------------
# Confirm that no preprocessing object has been fitted
# ------------------------------------------------------------

for preprocessor_name, preprocessor in {
    "Core regression": core_regression_preprocessor,
    "Full regression": full_regression_preprocessor,
    "Full tree": full_tree_preprocessor
}.items():

    if hasattr(
        preprocessor,
        "transformers_"
    ):
        raise RuntimeError(
            f"{preprocessor_name} preprocessing was fitted "
            "outside cross-validation."
        )


# ------------------------------------------------------------
# Summarize the preprocessing definitions
# ------------------------------------------------------------

preprocessing_summary = pd.DataFrame({
    "Pipeline": [
        "Core regression",
        "Full regression",
        "Full tree models"
    ],
    "Predictor_Count": [
        len(core_predictors),
        len(candidate_predictors),
        len(candidate_predictors)
    ],
    "Continuous_Predictors": [
        len(core_continuous_predictors),
        len(continuous_predictors),
        len(continuous_predictors)
    ],
    "Categorical_Predictors": [
        len(core_categorical_predictors),
        len(categorical_predictors),
        len(categorical_predictors)
    ],
    "Reference_Category_Dropped": [
        "Yes",
        "Yes",
        "No"
    ],
    "Tumor_Size_Imputation": [
        "Training-fold median",
        "Training-fold median",
        "Training-fold median"
    ],
    "Missingness_Indicator": [
        "Yes",
        "Yes",
        "Yes"
    ]
})


print("LEAKAGE-SAFE PREPROCESSING DEFINED")
print("----------------------------------")

display(preprocessing_summary)


print("\nCORE PREDICTORS")

for predictor in core_predictors:
    print(f"- {predictor}")

In [ ]:
# ============================================================
# Define primary unweighted candidate models and hyperparameter spaces
# ============================================================

from sklearn.base import clone
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. Clinically prespecified ridge logistic regression
# ------------------------------------------------------------

core_ridge_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(
                core_regression_preprocessor
            )
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                solver="liblinear",
                max_iter=5000,
                random_state=RANDOM_SEED
            )
        )
    ]
)


core_ridge_parameter_space = {
    "model__C": np.logspace(
        -3,
        3,
        9
    ).tolist()
}


# ------------------------------------------------------------
# 2. Full elastic-net logistic regression
# ------------------------------------------------------------

elastic_net_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(
                full_regression_preprocessor
            )
        ),
        (
            "model",
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                max_iter=10000,
                tol=1e-4,
                random_state=RANDOM_SEED
            )
        )
    ]
)


elastic_net_parameter_space = {
    "model__C": np.logspace(
        -3,
        2,
        8
    ).tolist(),

    "model__l1_ratio": [
        0.00,
        0.25,
        0.50,
        0.75,
        1.00
    ],

}


# ------------------------------------------------------------
# 3. Random forest
# ------------------------------------------------------------

random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(
                full_tree_preprocessor
            )
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_SEED,
                n_jobs=1
            )
        )
    ]
)


random_forest_parameter_space = {
    "model__n_estimators": [
        300,
        600,
        900
    ],

    "model__max_depth": [
        3,
        5,
        8,
        None
    ],

    "model__min_samples_leaf": [
        5,
        10,
        20,
        40
    ],

    "model__max_features": [
        "sqrt",
        0.50,
        1.00
    ],


    "model__bootstrap": [
        True
    ]
}


# ------------------------------------------------------------
# 4. Histogram-based gradient boosting
# ------------------------------------------------------------

gradient_boosting_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(
                full_tree_preprocessor
            )
        ),
        (
            "model",
            HistGradientBoostingClassifier(
                early_stopping=False,
                random_state=RANDOM_SEED
            )
        )
    ]
)


gradient_boosting_parameter_space = {
    "model__learning_rate": [
        0.03,
        0.05,
        0.10
    ],

    "model__max_iter": [
        100,
        200,
        300
    ],

    "model__max_leaf_nodes": [
        7,
        15,
        31
    ],

    "model__min_samples_leaf": [
        10,
        20,
        40
    ],

    "model__l2_regularization": [
        0.0,
        1.0,
        10.0
    ]
}


# ------------------------------------------------------------
# Collect model specifications
# ------------------------------------------------------------

model_specifications = {
    "Core_Ridge_Logistic": {
        "pipeline": core_ridge_pipeline,
        "parameter_space": (
            core_ridge_parameter_space
        ),
        "search_method": "grid",
        "search_iterations": None,
        "predictors": core_predictors
    },

    "Elastic_Net_Logistic": {
        "pipeline": elastic_net_pipeline,
        "parameter_space": (
            elastic_net_parameter_space
        ),
        "search_method": "randomized",
        "search_iterations": 30,
        "predictors": candidate_predictors
    },

    "Random_Forest": {
        "pipeline": random_forest_pipeline,
        "parameter_space": (
            random_forest_parameter_space
        ),
        "search_method": "randomized",
        "search_iterations": 30,
        "predictors": candidate_predictors
    },

    "Gradient_Boosting": {
        "pipeline": gradient_boosting_pipeline,
        "parameter_space": (
            gradient_boosting_parameter_space
        ),
        "search_method": "randomized",
        "search_iterations": 30,
        "predictors": candidate_predictors
    }
}


# ------------------------------------------------------------
# Validate model definitions
# ------------------------------------------------------------

assert len(model_specifications) == 4, (
    "Expected four candidate model specifications."
)


for model_name, specification in (
    model_specifications.items()
):

    assert isinstance(
        specification["pipeline"],
        Pipeline
    ), (
        f"{model_name} is not defined as a pipeline."
    )

    assert (
        "preprocessor"
        in specification["pipeline"].named_steps
    ), (
        f"{model_name} does not contain preprocessing."
    )

    assert (
        "model"
        in specification["pipeline"].named_steps
    ), (
        f"{model_name} does not contain a classifier."
    )

    fitted_attributes = [
        attribute
        for attribute in [
            "classes_",
            "coef_",
            "feature_importances_"
        ]
        if hasattr(
            specification[
                "pipeline"
            ].named_steps["model"],
            attribute
        )
    ]

    assert not fitted_attributes, (
        f"{model_name} appears to have been fitted."
    )


# ------------------------------------------------------------
# Summarize model specifications
# ------------------------------------------------------------

model_definition_summary = pd.DataFrame({
    "Model": list(
        model_specifications.keys()
    ),

    "Predictor_Count": [
        len(
            specification[
                "predictors"
            ]
        )
        for specification
        in model_specifications.values()
    ],

    "Search_Method": [
        specification[
            "search_method"
        ]
        for specification
        in model_specifications.values()
    ],

    "Randomized_Search_Iterations": [
        specification[
            "search_iterations"
        ]
        for specification
        in model_specifications.values()
    ],

    "Class_Weight_Tuned": [
        (
            "model__class_weight"
            in specification[
                "parameter_space"
            ]
        )
        for specification
        in model_specifications.values()
    ]
})


print("CANDIDATE MODELS DEFINED")
print("------------------------")

display(model_definition_summary)


print("\nHYPERPARAMETER SPACES")

for model_name, specification in (
    model_specifications.items()
):

    print(f"\n{model_name}")

    for parameter_name, values in (
        specification[
            "parameter_space"
        ].items()
    ):
        print(
            f"  {parameter_name}: {values}"
        )


In [ ]:
# ============================================================
# Define threshold-selection and performance functions
# ============================================================

import os

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score,
    roc_curve
)


# ------------------------------------------------------------
# Silence the harmless joblib CPU-detection warning
# ------------------------------------------------------------

LOGICAL_CPU_COUNT = (
    os.cpu_count()
    if os.cpu_count() is not None
    else 1
)

os.environ[
    "LOKY_MAX_CPU_COUNT"
] = str(LOGICAL_CPU_COUNT)


# ------------------------------------------------------------
# Select a threshold using outer-training predictions only
# ------------------------------------------------------------

def select_youden_threshold(
    observed_outcomes,
    predicted_probabilities
):
    """
    Select the probability threshold maximizing Youden's J.

    Youden's J = sensitivity + specificity - 1.

    If several thresholds have the same maximum value,
    select the lowest threshold among them to favor
    sensitivity.
    """

    observed_outcomes = np.asarray(
        observed_outcomes,
        dtype=int
    )

    predicted_probabilities = np.asarray(
        predicted_probabilities,
        dtype=float
    )


    false_positive_rate, (
        true_positive_rate
    ), thresholds = roc_curve(
        observed_outcomes,
        predicted_probabilities
    )


    finite_threshold = np.isfinite(
        thresholds
    )

    false_positive_rate = (
        false_positive_rate[
            finite_threshold
        ]
    )

    true_positive_rate = (
        true_positive_rate[
            finite_threshold
        ]
    )

    thresholds = thresholds[
        finite_threshold
    ]


    youden_j = (
        true_positive_rate
        - false_positive_rate
    )

    maximum_youden_j = np.max(
        youden_j
    )

    tied_thresholds = thresholds[
        np.isclose(
            youden_j,
            maximum_youden_j
        )
    ]


    selected_threshold = float(
        np.min(tied_thresholds)
    )


    return selected_threshold


# ------------------------------------------------------------
# Estimate calibration intercept and slope
# ------------------------------------------------------------

def calculate_calibration_parameters(
    observed_outcomes,
    predicted_probabilities
):
    """
    Estimate calibration intercept and slope by fitting
    a logistic recalibration model:

        logit(Y) = intercept + slope * logit(predicted risk)
    """

    observed_outcomes = np.asarray(
        observed_outcomes,
        dtype=int
    )

    predicted_probabilities = np.asarray(
        predicted_probabilities,
        dtype=float
    )


    clipped_probabilities = np.clip(
        predicted_probabilities,
        1e-6,
        1 - 1e-6
    )


    predicted_log_odds = np.log(
        clipped_probabilities
        / (
            1 - clipped_probabilities
        )
    ).reshape(-1, 1)


    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=5000
    )


    calibration_model.fit(
        predicted_log_odds,
        observed_outcomes
    )


    calibration_intercept = float(
        calibration_model.intercept_[0]
    )

    calibration_slope = float(
        calibration_model.coef_[0, 0]
    )


    return (
        calibration_intercept,
        calibration_slope
    )


# ------------------------------------------------------------
# Calculate complete performance measures
# ------------------------------------------------------------

def calculate_binary_performance(
    observed_outcomes,
    predicted_probabilities,
    classification_threshold
):
    """
    Calculate discrimination, probability accuracy,
    calibration, and threshold-dependent performance.
    """

    observed_outcomes = np.asarray(
        observed_outcomes,
        dtype=int
    )

    predicted_probabilities = np.asarray(
        predicted_probabilities,
        dtype=float
    )


    predicted_classes = (
        predicted_probabilities
        >= classification_threshold
    ).astype(int)


    true_negative, false_positive, (
        false_negative
    ), true_positive = confusion_matrix(
        observed_outcomes,
        predicted_classes,
        labels=[0, 1]
    ).ravel()


    sensitivity = (
        true_positive
        / (true_positive + false_negative)
        if (
            true_positive + false_negative
        ) > 0
        else np.nan
    )


    specificity = (
        true_negative
        / (true_negative + false_positive)
        if (
            true_negative + false_positive
        ) > 0
        else np.nan
    )


    negative_predictive_value = (
        true_negative
        / (true_negative + false_negative)
        if (
            true_negative + false_negative
        ) > 0
        else np.nan
    )


    calibration_intercept, (
        calibration_slope
    ) = calculate_calibration_parameters(
        observed_outcomes,
        predicted_probabilities
    )


    return {
        "PR_AUC": average_precision_score(
            observed_outcomes,
            predicted_probabilities
        ),

        "ROC_AUC": roc_auc_score(
            observed_outcomes,
            predicted_probabilities
        ),

        "Brier_Score": brier_score_loss(
            observed_outcomes,
            predicted_probabilities
        ),

        "Calibration_Intercept": (
            calibration_intercept
        ),

        "Calibration_Slope": (
            calibration_slope
        ),

        "Threshold": float(
            classification_threshold
        ),

        "Sensitivity": sensitivity,

        "Specificity": specificity,

        "Precision": precision_score(
            observed_outcomes,
            predicted_classes,
            zero_division=0
        ),

        "Negative_Predictive_Value": (
            negative_predictive_value
        ),

        "F1_Score": f1_score(
            observed_outcomes,
            predicted_classes,
            zero_division=0
        ),

        "Balanced_Accuracy": (
            balanced_accuracy_score(
                observed_outcomes,
                predicted_classes
            )
        ),

        "True_Negative": int(
            true_negative
        ),

        "False_Positive": int(
            false_positive
        ),

        "False_Negative": int(
            false_negative
        ),

        "True_Positive": int(
            true_positive
        )
    }


# ------------------------------------------------------------
# Unit-test the functions
# ------------------------------------------------------------

test_outcomes = np.array([
    0, 0, 0, 0, 1, 1
])

test_probabilities = np.array([
    0.05,
    0.10,
    0.30,
    0.40,
    0.60,
    0.85
])


test_threshold = select_youden_threshold(
    test_outcomes,
    test_probabilities
)


test_performance = (
    calculate_binary_performance(
        test_outcomes,
        test_probabilities,
        test_threshold
    )
)


assert (
    0 < test_threshold < 1
), "The threshold-selection function failed."

assert set(
    [
        "PR_AUC",
        "ROC_AUC",
        "Brier_Score",
        "Calibration_Intercept",
        "Calibration_Slope",
        "Sensitivity",
        "Specificity",
        "Precision",
        "Negative_Predictive_Value",
        "F1_Score",
        "Balanced_Accuracy"
    ]
).issubset(
    test_performance.keys()
), "The performance function is incomplete."


print("PERFORMANCE FUNCTIONS DEFINED AND TESTED")
print("----------------------------------------")
print("Logical CPU count:", LOGICAL_CPU_COUNT)
print(
    "Test threshold:",
    round(test_threshold, 4)
)

display(
    pd.DataFrame(
        [test_performance]
    ).round(4)
)

In [ ]:
# ============================================================
# Configure and verify the resumable unweighted analysis
# ============================================================

import warnings


# ------------------------------------------------------------
# Define unweighted checkpoint files
# ------------------------------------------------------------

FOLD_PERFORMANCE_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_nested_fold_performance.csv"
)

PREDICTIONS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_nested_held_out_predictions.csv"
)

HYPERPARAMETERS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_nested_selected_hyperparameters.csv"
)

PERMUTATION_IMPORTANCE_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_nested_permutation_importance.csv"
)

COEFFICIENTS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_nested_logistic_coefficients.csv"
)


checkpoint_files = [
    FOLD_PERFORMANCE_FILE,
    PREDICTIONS_FILE,
    HYPERPARAMETERS_FILE,
    PERMUTATION_IMPORTANCE_FILE,
    COEFFICIENTS_FILE,
]


# ------------------------------------------------------------
# Verify model configurations
# ------------------------------------------------------------

for model_name, specification in model_specifications.items():

    pipeline_parameters = (
        specification["pipeline"]
        .get_params(deep=True)
    )

    parameter_space = specification[
        "parameter_space"
    ]

    assert (
        "model__class_weight"
        not in parameter_space
    ), (
        f"{model_name} still tunes class_weight."
    )

    if (
        "model__class_weight"
        in pipeline_parameters
    ):
        assert (
            pipeline_parameters[
                "model__class_weight"
            ]
            is None
        ), (
            f"{model_name} is not unweighted."
        )

    if model_name == "Random_Forest":
        assert (
            pipeline_parameters[
                "model__n_jobs"
            ]
            == 1
        ), (
            "Random_Forest does not have "
            "n_jobs=1."
        )


# ------------------------------------------------------------
# Suppress harmless encoder warnings
# ------------------------------------------------------------

warnings.filterwarnings(
    "ignore",
    message=(
        "Found unknown categories in columns .* "
        "during transform.*"
    ),
    category=UserWarning,
    module="sklearn.preprocessing._encoders"
)


# ------------------------------------------------------------
# Report readiness and checkpoint status
# ------------------------------------------------------------

print("PRIMARY UNWEIGHTED RUN: READY")
print("-----------------------------")

print("All estimators are unweighted.")

print(
    "Random forest and all search operations "
    "use n_jobs=1."
)

print(
    "Harmless unknown-category warnings "
    "are suppressed."
)

print("\nCheckpoint status:")


for checkpoint_file in checkpoint_files:

    print(
        "-",
        checkpoint_file.name,
        "| exists:",
        checkpoint_file.exists()
    )


print(
    "\nExisting checkpoints will be resumed "
    "automatically; completed evaluations "
    "will not be repeated."
)

## Execution

Use **Kernel → Restart Kernel and Run All Cells**. The validation cell will read the existing unweighted checkpoints, skip every completed model-fold evaluation, and resume at the first unfinished evaluation.

If 61 evaluations were previously saved, the validation output should report `Previously completed model-fold evaluations: 61` and continue with evaluation 62.


In [ ]:
# ============================================================
# Run repeated nested cross-validation
# ============================================================

import json
import time

from sklearn.inspection import permutation_importance
from sklearn.model_selection import (
    cross_val_predict,
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
)


# Set to True to continue an interrupted run.
RESUME_EXISTING_RUN = True


def load_checkpoint(file_path):
    if (
        RESUME_EXISTING_RUN
        and file_path.exists()
    ):
        return pd.read_csv(file_path)

    return pd.DataFrame()


fold_performance_results = load_checkpoint(
    FOLD_PERFORMANCE_FILE
)

held_out_prediction_results = load_checkpoint(
    PREDICTIONS_FILE
)

selected_hyperparameter_results = load_checkpoint(
    HYPERPARAMETERS_FILE
)

permutation_importance_results = load_checkpoint(
    PERMUTATION_IMPORTANCE_FILE
)

logistic_coefficient_results = load_checkpoint(
    COEFFICIENTS_FILE
)


# ------------------------------------------------------------
# Identify already completed evaluations
# ------------------------------------------------------------

completed_evaluations = set()


if not fold_performance_results.empty:

    completed_evaluations = set(
        zip(
            fold_performance_results[
                "Model"
            ],
            fold_performance_results[
                "Repetition"
            ].astype(int),
            fold_performance_results[
                "Outer_Fold"
            ].astype(int)
        )
    )


print(
    "Previously completed model-fold evaluations:",
    len(completed_evaluations)
)


# ------------------------------------------------------------
# Recreate the deterministic outer splits
# ------------------------------------------------------------

full_outer_cross_validation = (
    RepeatedStratifiedKFold(
        n_splits=OUTER_FOLDS,
        n_repeats=OUTER_REPEATS,
        random_state=RANDOM_SEED
    )
)


total_model_evaluations = (
    OUTER_FOLDS
    * OUTER_REPEATS
    * len(model_specifications)
)


overall_start_time = time.perf_counter()


# ------------------------------------------------------------
# Outer validation loop
# ------------------------------------------------------------

for split_number, (
    outer_training_indices,
    outer_validation_indices
) in enumerate(
    full_outer_cross_validation.split(
        X,
        y
    ),
    start=1
):

    repetition_number = (
        (split_number - 1) // OUTER_FOLDS
    ) + 1

    outer_fold_number = (
        (split_number - 1) % OUTER_FOLDS
    ) + 1


    for model_number, (
        model_name,
        specification
    ) in enumerate(
        model_specifications.items(),
        start=1
    ):

        evaluation_key = (
            model_name,
            repetition_number,
            outer_fold_number
        )


        if evaluation_key in completed_evaluations:

            print(
                "Skipping completed evaluation:",
                model_name,
                "repetition",
                repetition_number,
                "fold",
                outer_fold_number
            )

            continue


        completed_before_current = len(
            completed_evaluations
        )

        print(
            "\nEvaluation "
            f"{completed_before_current + 1}/"
            f"{total_model_evaluations}"
        )

        print(
            model_name,
            "| repetition",
            repetition_number,
            "| outer fold",
            outer_fold_number
        )


        selected_predictors = (
            specification[
                "predictors"
            ]
        )


        X_outer_training = (
            X.iloc[
                outer_training_indices
            ][
                selected_predictors
            ]
            .copy()
        )

        y_outer_training = (
            y.iloc[
                outer_training_indices
            ]
            .copy()
        )

        X_outer_validation = (
            X.iloc[
                outer_validation_indices
            ][
                selected_predictors
            ]
            .copy()
        )

        y_outer_validation = (
            y.iloc[
                outer_validation_indices
            ]
            .copy()
        )


        # A deterministic but distinct seed for every
        # repetition, fold, and model.
        evaluation_seed = (
            RANDOM_SEED
            + repetition_number * 1000
            + outer_fold_number * 100
            + model_number
        )


        inner_cross_validation = StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=evaluation_seed
        )


        candidate_pipeline = clone(
            specification["pipeline"]
        )


        # Update the classifier seed for this evaluation.
        if "random_state" in (
            candidate_pipeline
            .named_steps["model"]
            .get_params()
        ):
            candidate_pipeline.set_params(
                model__random_state=(
                    evaluation_seed
                )
            )


        if (
            specification[
                "search_method"
            ]
            == "grid"
        ):

            model_search = GridSearchCV(
                estimator=candidate_pipeline,
                param_grid=specification[
                    "parameter_space"
                ],
                scoring="average_precision",
                cv=inner_cross_validation,
                refit=True,
                n_jobs=1,
                return_train_score=False,
                error_score="raise"
            )

        elif (
            specification[
                "search_method"
            ]
            == "randomized"
        ):

            model_search = RandomizedSearchCV(
                estimator=candidate_pipeline,
                param_distributions=specification[
                    "parameter_space"
                ],
                n_iter=specification[
                    "search_iterations"
                ],
                scoring="average_precision",
                cv=inner_cross_validation,
                refit=True,
                random_state=evaluation_seed,
                n_jobs=1,
                return_train_score=False,
                error_score="raise"
            )

        else:
            raise ValueError(
                "Unknown search method for "
                f"{model_name}."
            )


        evaluation_start_time = (
            time.perf_counter()
        )


        # ----------------------------------------------------
        # Inner hyperparameter search
        # ----------------------------------------------------

        model_search.fit(
            X_outer_training,
            y_outer_training
        )


        best_estimator = (
            model_search.best_estimator_
        )


        # ----------------------------------------------------
        # Training-derived threshold
        # ----------------------------------------------------

        inner_held_out_probabilities = (
            cross_val_predict(
                clone(best_estimator),
                X_outer_training,
                y_outer_training,
                cv=inner_cross_validation,
                method="predict_proba",
                n_jobs=1
            )[:, 1]
        )


        selected_threshold = (
            select_youden_threshold(
                y_outer_training,
                inner_held_out_probabilities
            )
        )


        # ----------------------------------------------------
        # Held-out outer-fold predictions
        # ----------------------------------------------------

        outer_probabilities = (
            best_estimator.predict_proba(
                X_outer_validation
            )[:, 1]
        )


        outer_predicted_classes = (
            outer_probabilities
            >= selected_threshold
        ).astype(int)


        outer_default_classes = (
            outer_probabilities
            >= 0.50
        ).astype(int)


        # ----------------------------------------------------
        # Held-out performance
        # ----------------------------------------------------

        fold_performance = (
            calculate_binary_performance(
                y_outer_validation,
                outer_probabilities,
                selected_threshold
            )
        )


        elapsed_seconds = (
            time.perf_counter()
            - evaluation_start_time
        )


        fold_performance_record = {
            "Model": model_name,
            "Repetition": repetition_number,
            "Outer_Fold": outer_fold_number,
            "Training_Patients": len(
                outer_training_indices
            ),
            "Training_Events": int(
                y_outer_training.sum()
            ),
            "Validation_Patients": len(
                outer_validation_indices
            ),
            "Validation_Events": int(
                y_outer_validation.sum()
            ),
            "Inner_Best_PR_AUC": float(
                model_search.best_score_
            ),
            "Elapsed_Seconds": float(
                elapsed_seconds
            ),
            **fold_performance
        }


        fold_performance_results = pd.concat(
            [
                fold_performance_results,
                pd.DataFrame(
                    [fold_performance_record]
                )
            ],
            ignore_index=True
        )


        # ----------------------------------------------------
        # Save patient-level held-out predictions
        # ----------------------------------------------------

        prediction_records = pd.DataFrame({
            "Model": model_name,
            "Repetition": repetition_number,
            "Outer_Fold": outer_fold_number,
            "Modeling_Patient_ID": (
                patient_id.iloc[
                    outer_validation_indices
                ].to_numpy()
            ),
            "Observed_Outcome": (
                y_outer_validation.to_numpy()
            ),
            "Predicted_Probability": (
                outer_probabilities
            ),
            "Selected_Threshold": (
                selected_threshold
            ),
            "Predicted_Class": (
                outer_predicted_classes
            ),
            "Default_050_Class": (
                outer_default_classes
            )
        })


        held_out_prediction_results = pd.concat(
            [
                held_out_prediction_results,
                prediction_records
            ],
            ignore_index=True
        )


        # ----------------------------------------------------
        # Save selected hyperparameters
        # ----------------------------------------------------

        hyperparameter_record = {
            "Model": model_name,
            "Repetition": repetition_number,
            "Outer_Fold": outer_fold_number,
            "Inner_Best_PR_AUC": float(
                model_search.best_score_
            ),
            "Selected_Threshold": float(
                selected_threshold
            ),
            "Best_Parameters": json.dumps(
                model_search.best_params_,
                default=str,
                sort_keys=True
            )
        }


        selected_hyperparameter_results = (
            pd.concat(
                [
                    selected_hyperparameter_results,
                    pd.DataFrame(
                        [hyperparameter_record]
                    )
                ],
                ignore_index=True
            )
        )


        # ----------------------------------------------------
        # Held-out permutation importance
        # ----------------------------------------------------

        held_out_importance = (
            permutation_importance(
                best_estimator,
                X_outer_validation,
                y_outer_validation,
                scoring="average_precision",
                n_repeats=5,
                random_state=evaluation_seed,
                n_jobs=1
            )
        )


        importance_records = pd.DataFrame({
            "Model": model_name,
            "Repetition": repetition_number,
            "Outer_Fold": outer_fold_number,
            "Predictor": selected_predictors,
            "Importance_Mean": (
                held_out_importance[
                    "importances_mean"
                ]
            ),
            "Importance_SD": (
                held_out_importance[
                    "importances_std"
                ]
            )
        })


        permutation_importance_results = (
            pd.concat(
                [
                    permutation_importance_results,
                    importance_records
                ],
                ignore_index=True
            )
        )


        # ----------------------------------------------------
        # Save logistic-regression coefficients
        # ----------------------------------------------------

        if model_name in [
            "Core_Ridge_Logistic",
            "Elastic_Net_Logistic"
        ]:

            fitted_preprocessor = (
                best_estimator.named_steps[
                    "preprocessor"
                ]
            )

            fitted_classifier = (
                best_estimator.named_steps[
                    "model"
                ]
            )

            encoded_feature_names = (
                fitted_preprocessor
                .get_feature_names_out()
            )

            fitted_coefficients = (
                fitted_classifier.coef_[0]
            )


            assert len(
                encoded_feature_names
            ) == len(
                fitted_coefficients
            ), (
                "Encoded feature names and logistic "
                "coefficients do not reconcile."
            )


            coefficient_records = pd.DataFrame({
                "Model": model_name,
                "Repetition": repetition_number,
                "Outer_Fold": outer_fold_number,
                "Encoded_Feature": (
                    encoded_feature_names
                ),
                "Coefficient": (
                    fitted_coefficients
                ),
                "Nonzero": (
                    np.abs(
                        fitted_coefficients
                    ) > 1e-8
                )
            })


            logistic_coefficient_results = (
                pd.concat(
                    [
                        logistic_coefficient_results,
                        coefficient_records
                    ],
                    ignore_index=True
                )
            )


        # ----------------------------------------------------
        # Save recovery checkpoints
        # ----------------------------------------------------

        fold_performance_results.to_csv(
            FOLD_PERFORMANCE_FILE,
            index=False
        )

        held_out_prediction_results.to_csv(
            PREDICTIONS_FILE,
            index=False
        )

        selected_hyperparameter_results.to_csv(
            HYPERPARAMETERS_FILE,
            index=False
        )

        permutation_importance_results.to_csv(
            PERMUTATION_IMPORTANCE_FILE,
            index=False
        )

        logistic_coefficient_results.to_csv(
            COEFFICIENTS_FILE,
            index=False
        )


        completed_evaluations.add(
            evaluation_key
        )


        print(
            "Completed in",
            round(elapsed_seconds, 1),
            "seconds"
        )

        print(
            "Held-out PR-AUC:",
            round(
                fold_performance[
                    "PR_AUC"
                ],
                4
            )
        )

        print(
            "Held-out ROC-AUC:",
            round(
                fold_performance[
                    "ROC_AUC"
                ],
                4
            )
        )


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

expected_evaluations = (
    OUTER_FOLDS
    * OUTER_REPEATS
    * len(model_specifications)
)


assert len(
    fold_performance_results
) == expected_evaluations, (
    "The full nested-validation run is incomplete."
)

assert len(
    held_out_prediction_results
) == (
    len(analysis_data)
    * OUTER_REPEATS
    * len(model_specifications)
), (
    "The number of held-out predictions is incorrect."
)


total_elapsed_minutes = (
    time.perf_counter()
    - overall_start_time
) / 60


print("\nFULL NESTED VALIDATION COMPLETED")
print("--------------------------------")
print(
    "Model-fold evaluations:",
    len(fold_performance_results)
)
print(
    "Held-out predictions:",
    len(held_out_prediction_results)
)
print(
    "Elapsed time during this run:",
    round(total_elapsed_minutes, 2),
    "minutes"
)

print("\nCheckpoint directory:")
print(MODEL_RESULTS_DIRECTORY)


In [ ]:
# ============================================================
# Verify completed unweighted nested-validation checkpoints
# ============================================================

import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Load all five checkpoint files
# ------------------------------------------------------------

fold_performance = pd.read_csv(
    FOLD_PERFORMANCE_FILE
)

held_out_predictions = pd.read_csv(
    PREDICTIONS_FILE
)

selected_hyperparameters = pd.read_csv(
    HYPERPARAMETERS_FILE
)

permutation_importance_results = pd.read_csv(
    PERMUTATION_IMPORTANCE_FILE
)

logistic_coefficient_results = pd.read_csv(
    COEFFICIENTS_FILE
)


# ------------------------------------------------------------
# Expected dimensions
# ------------------------------------------------------------

EXPECTED_MODELS = 4

EXPECTED_EVALUATIONS = (
    OUTER_FOLDS
    * OUTER_REPEATS
    * EXPECTED_MODELS
)

EXPECTED_PREDICTIONS = (
    len(analysis_data)
    * OUTER_REPEATS
    * EXPECTED_MODELS
)

EXPECTED_MODEL_NAMES = {
    "Core_Ridge_Logistic",
    "Elastic_Net_Logistic",
    "Random_Forest",
    "Gradient_Boosting"
}

evaluation_key_columns = [
    "Model",
    "Repetition",
    "Outer_Fold"
]


# ------------------------------------------------------------
# Verify fold-performance results
# ------------------------------------------------------------

assert len(fold_performance) == EXPECTED_EVALUATIONS, (
    "The fold-performance file does not contain "
    f"{EXPECTED_EVALUATIONS} evaluations."
)

assert not fold_performance[
    evaluation_key_columns
].duplicated().any(), (
    "Duplicate model-fold evaluations were found."
)

assert set(
    fold_performance["Model"].unique()
) == EXPECTED_MODEL_NAMES, (
    "The fold-performance file contains unexpected "
    "or missing model names."
)

assert fold_performance[
    evaluation_key_columns
].notna().all().all(), (
    "Missing model, repetition or fold identifiers "
    "were found."
)


# ------------------------------------------------------------
# Verify held-out predictions
# ------------------------------------------------------------

assert len(
    held_out_predictions
) == EXPECTED_PREDICTIONS, (
    "The held-out prediction count is incorrect."
)

assert held_out_predictions[
    "Predicted_Probability"
].between(0, 1).all(), (
    "Predicted probabilities outside [0, 1] were found."
)

assert held_out_predictions[
    "Observed_Outcome"
].isin([0, 1]).all(), (
    "Unexpected observed outcome values were found."
)

assert held_out_predictions[
    "Predicted_Class"
].isin([0, 1]).all(), (
    "Unexpected predicted-class values were found."
)

prediction_key_columns = [
    "Model",
    "Repetition",
    "Outer_Fold",
    "Modeling_Patient_ID"
]

assert not held_out_predictions[
    prediction_key_columns
].duplicated().any(), (
    "Duplicate patient-level held-out predictions "
    "were found."
)


# Every patient must have one held-out prediction in
# each repetition for each model.

predictions_per_patient_model = (
    held_out_predictions
    .groupby(
        [
            "Model",
            "Modeling_Patient_ID"
        ]
    )
    .size()
)

assert (
    predictions_per_patient_model
    == OUTER_REPEATS
).all(), (
    "At least one patient does not have exactly "
    f"{OUTER_REPEATS} predictions per model."
)


# Every patient must have a consistent observed outcome.

outcomes_per_patient = (
    held_out_predictions
    .groupby(
        "Modeling_Patient_ID"
    )["Observed_Outcome"]
    .nunique()
)

assert (
    outcomes_per_patient == 1
).all(), (
    "At least one patient has inconsistent observed "
    "outcomes across predictions."
)


# ------------------------------------------------------------
# Verify selected hyperparameters
# ------------------------------------------------------------

assert len(
    selected_hyperparameters
) == EXPECTED_EVALUATIONS, (
    "The selected-hyperparameter count is incorrect."
)

assert not selected_hyperparameters[
    evaluation_key_columns
].duplicated().any(), (
    "Duplicate selected-hyperparameter records "
    "were found."
)


def parse_parameters(value):

    if isinstance(value, dict):
        return value

    return json.loads(value)


parsed_parameter_records = (
    selected_hyperparameters[
        "Best_Parameters"
    ]
    .apply(parse_parameters)
)


class_weight_entries = [
    parameters.get(
        "model__class_weight",
        "NOT_PRESENT"
    )
    for parameters in parsed_parameter_records
]


assert all(
    value == "NOT_PRESENT"
    for value in class_weight_entries
), (
    "At least one unweighted evaluation still contains "
    "a tuned class-weight parameter."
)


# ------------------------------------------------------------
# Verify evaluation-key agreement
# ------------------------------------------------------------

fold_keys = set(
    map(
        tuple,
        fold_performance[
            evaluation_key_columns
        ].to_numpy()
    )
)

prediction_keys = set(
    map(
        tuple,
        held_out_predictions[
            evaluation_key_columns
        ]
        .drop_duplicates()
        .to_numpy()
    )
)

hyperparameter_keys = set(
    map(
        tuple,
        selected_hyperparameters[
            evaluation_key_columns
        ].to_numpy()
    )
)

importance_keys = set(
    map(
        tuple,
        permutation_importance_results[
            evaluation_key_columns
        ]
        .drop_duplicates()
        .to_numpy()
    )
)

coefficient_keys = set(
    map(
        tuple,
        logistic_coefficient_results[
            evaluation_key_columns
        ]
        .drop_duplicates()
        .to_numpy()
    )
)


assert fold_keys == prediction_keys, (
    "Prediction evaluation keys do not match "
    "fold-performance keys."
)

assert fold_keys == hyperparameter_keys, (
    "Hyperparameter evaluation keys do not match "
    "fold-performance keys."
)

assert fold_keys == importance_keys, (
    "Permutation-importance evaluation keys do not "
    "match fold-performance keys."
)


expected_logistic_keys = {
    key
    for key in fold_keys
    if key[0] in {
        "Core_Ridge_Logistic",
        "Elastic_Net_Logistic"
    }
}


assert coefficient_keys == expected_logistic_keys, (
    "Logistic-coefficient evaluation keys are "
    "incomplete or unexpected."
)


# ------------------------------------------------------------
# Verify missingness in essential variables
# ------------------------------------------------------------

essential_performance_columns = [
    "PR_AUC",
    "ROC_AUC",
    "Brier_Score",
    "Calibration_Intercept",
    "Calibration_Slope",
    "Threshold"
]

assert fold_performance[
    essential_performance_columns
].notna().all().all(), (
    "Missing values were found in essential "
    "performance metrics."
)

assert held_out_predictions[
    [
        "Modeling_Patient_ID",
        "Observed_Outcome",
        "Predicted_Probability",
        "Selected_Threshold",
        "Predicted_Class"
    ]
].notna().all().all(), (
    "Missing values were found in essential "
    "prediction variables."
)


# ------------------------------------------------------------
# Produce verification summary
# ------------------------------------------------------------

verification_summary = pd.DataFrame({
    "Checkpoint": [
        "Fold performance",
        "Held-out predictions",
        "Selected hyperparameters",
        "Permutation importance",
        "Logistic coefficients"
    ],
    "Records": [
        len(fold_performance),
        len(held_out_predictions),
        len(selected_hyperparameters),
        len(permutation_importance_results),
        len(logistic_coefficient_results)
    ],
    "Unique_Evaluations": [
        len(fold_keys),
        len(prediction_keys),
        len(hyperparameter_keys),
        len(importance_keys),
        len(coefficient_keys)
    ]
})


model_completion_summary = (
    fold_performance
    .groupby("Model")
    .agg(
        Evaluations=(
            "PR_AUC",
            "size"
        ),
        Repetitions=(
            "Repetition",
            "nunique"
        ),
        Outer_Folds=(
            "Outer_Fold",
            "nunique"
        )
    )
    .reset_index()
)


print("UNWEIGHTED CHECKPOINT VERIFICATION PASSED")
print("-----------------------------------------")

display(verification_summary)

print("\nMODEL COMPLETION SUMMARY")
print("------------------------")

display(model_completion_summary)

print("\nExpected evaluations:", EXPECTED_EVALUATIONS)
print("Expected predictions:", EXPECTED_PREDICTIONS)

print(
    "\nAll five checkpoint files are complete, "
    "consistent and confirmed as unweighted."
)

In [ ]:
# ============================================================
# Patient-level performance with bootstrap confidence intervals
# ============================================================

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score
)


# ------------------------------------------------------------
# Bootstrap settings
# ------------------------------------------------------------

NUMBER_OF_BOOTSTRAP_SAMPLES = 2000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
BOOTSTRAP_RANDOM_SEED = RANDOM_SEED + 10000


model_order = [
    "Core_Ridge_Logistic",
    "Elastic_Net_Logistic",
    "Random_Forest",
    "Gradient_Boosting"
]


# ------------------------------------------------------------
# Average each patient's ten held-out predictions
# ------------------------------------------------------------

patient_level_predictions = (
    held_out_predictions
    .groupby(
        [
            "Model",
            "Modeling_Patient_ID"
        ],
        as_index=False
    )
    .agg(
        Observed_Outcome=(
            "Observed_Outcome",
            "first"
        ),
        Mean_Predicted_Probability=(
            "Predicted_Probability",
            "mean"
        ),
        Mean_Selected_Threshold=(
            "Selected_Threshold",
            "mean"
        ),
        Prediction_SD=(
            "Predicted_Probability",
            "std"
        ),
        Prediction_Repetitions=(
            "Predicted_Probability",
            "size"
        ),
        Outcome_Values=(
            "Observed_Outcome",
            "nunique"
        )
    )
)


assert (
    patient_level_predictions[
        "Prediction_Repetitions"
    ]
    == OUTER_REPEATS
).all(), (
    "At least one patient does not have ten "
    "held-out predictions."
)


assert (
    patient_level_predictions[
        "Outcome_Values"
    ]
    == 1
).all(), (
    "At least one patient has inconsistent outcomes."
)


assert len(patient_level_predictions) == (
    len(analysis_data)
    * len(model_order)
), (
    "The patient-level prediction count is incorrect."
)


patient_level_predictions[
    "Predicted_Class"
] = (
    patient_level_predictions[
        "Mean_Predicted_Probability"
    ]
    >= patient_level_predictions[
        "Mean_Selected_Threshold"
    ]
).astype(int)


# ------------------------------------------------------------
# Calibration intercept and slope
# ------------------------------------------------------------

def calculate_calibration_parameters(
    observed_outcomes,
    predicted_probabilities
):

    observed_outcomes = np.asarray(
        observed_outcomes,
        dtype=float
    )

    predicted_probabilities = np.asarray(
        predicted_probabilities,
        dtype=float
    )

    clipped_probabilities = np.clip(
        predicted_probabilities,
        1e-6,
        1 - 1e-6
    )

    prediction_logit = np.log(
        clipped_probabilities
        / (1 - clipped_probabilities)
    )


    def negative_log_likelihood(parameters):

        intercept, slope = parameters

        linear_predictor = (
            intercept
            + slope * prediction_logit
        )

        fitted_probabilities = (
            1
            / (
                1
                + np.exp(
                    -np.clip(
                        linear_predictor,
                        -35,
                        35
                    )
                )
            )
        )

        fitted_probabilities = np.clip(
            fitted_probabilities,
            1e-12,
            1 - 1e-12
        )

        return -np.sum(
            observed_outcomes
            * np.log(fitted_probabilities)
            + (
                1 - observed_outcomes
            )
            * np.log(
                1 - fitted_probabilities
            )
        )


    optimization_result = minimize(
        negative_log_likelihood,
        x0=np.array([0.0, 1.0]),
        method="BFGS"
    )


    if not optimization_result.success:

        optimization_result = minimize(
            negative_log_likelihood,
            x0=np.array([0.0, 1.0]),
            method="Nelder-Mead"
        )


    calibration_intercept = float(
        optimization_result.x[0]
    )

    calibration_slope = float(
        optimization_result.x[1]
    )


    return (
        calibration_intercept,
        calibration_slope
    )


# ------------------------------------------------------------
# Complete performance function
# ------------------------------------------------------------

def calculate_patient_level_performance(
    observed_outcomes,
    predicted_probabilities,
    predicted_classes
):

    observed_outcomes = np.asarray(
        observed_outcomes,
        dtype=int
    )

    predicted_probabilities = np.asarray(
        predicted_probabilities,
        dtype=float
    )

    predicted_classes = np.asarray(
        predicted_classes,
        dtype=int
    )


    true_negative, false_positive, (
        false_negative
    ), true_positive = confusion_matrix(
        observed_outcomes,
        predicted_classes,
        labels=[0, 1]
    ).ravel()


    sensitivity = (
        true_positive
        / (
            true_positive
            + false_negative
        )
        if (
            true_positive
            + false_negative
        ) > 0
        else np.nan
    )


    specificity = (
        true_negative
        / (
            true_negative
            + false_positive
        )
        if (
            true_negative
            + false_positive
        ) > 0
        else np.nan
    )


    negative_predictive_value = (
        true_negative
        / (
            true_negative
            + false_negative
        )
        if (
            true_negative
            + false_negative
        ) > 0
        else np.nan
    )


    calibration_intercept, (
        calibration_slope
    ) = calculate_calibration_parameters(
        observed_outcomes,
        predicted_probabilities
    )


    return {
        "PR_AUC": average_precision_score(
            observed_outcomes,
            predicted_probabilities
        ),

        "ROC_AUC": roc_auc_score(
            observed_outcomes,
            predicted_probabilities
        ),

        "Brier_Score": brier_score_loss(
            observed_outcomes,
            predicted_probabilities
        ),

        "Calibration_Intercept": (
            calibration_intercept
        ),

        "Calibration_Slope": (
            calibration_slope
        ),

        "Sensitivity": sensitivity,

        "Specificity": specificity,

        "Positive_Predictive_Value": (
            precision_score(
                observed_outcomes,
                predicted_classes,
                zero_division=0
            )
        ),

        "Negative_Predictive_Value": (
            negative_predictive_value
        ),

        "F1_Score": f1_score(
            observed_outcomes,
            predicted_classes,
            zero_division=0
        ),

        "Balanced_Accuracy": (
            balanced_accuracy_score(
                observed_outcomes,
                predicted_classes
            )
        )
    }


# ------------------------------------------------------------
# Point estimates and patient-level bootstrap
# ------------------------------------------------------------

bootstrap_random_generator = (
    np.random.default_rng(
        BOOTSTRAP_RANDOM_SEED
    )
)


point_estimate_records = []
bootstrap_records = []


for model_name in model_order:

    model_predictions = (
        patient_level_predictions.loc[
            patient_level_predictions[
                "Model"
            ] == model_name
        ]
        .sort_values(
            "Modeling_Patient_ID"
        )
        .reset_index(drop=True)
    )


    observed_outcomes = (
        model_predictions[
            "Observed_Outcome"
        ].to_numpy(dtype=int)
    )

    predicted_probabilities = (
        model_predictions[
            "Mean_Predicted_Probability"
        ].to_numpy(dtype=float)
    )

    predicted_classes = (
        model_predictions[
            "Predicted_Class"
        ].to_numpy(dtype=int)
    )


    point_estimates = (
        calculate_patient_level_performance(
            observed_outcomes,
            predicted_probabilities,
            predicted_classes
        )
    )


    for metric_name, estimate in (
        point_estimates.items()
    ):

        point_estimate_records.append({
            "Model": model_name,
            "Metric": metric_name,
            "Estimate": float(estimate)
        })


    number_of_patients = len(
        model_predictions
    )


    for bootstrap_number in range(
        1,
        NUMBER_OF_BOOTSTRAP_SAMPLES + 1
    ):

        sampled_indices = (
            bootstrap_random_generator.choice(
                number_of_patients,
                size=number_of_patients,
                replace=True
            )
        )


        bootstrap_outcomes = (
            observed_outcomes[
                sampled_indices
            ]
        )

        bootstrap_probabilities = (
            predicted_probabilities[
                sampled_indices
            ]
        )

        bootstrap_classes = (
            predicted_classes[
                sampled_indices
            ]
        )


        # This should be extremely rare, but protects
        # ROC-AUC if a resample contains only one class.
        if np.unique(
            bootstrap_outcomes
        ).size < 2:
            continue


        bootstrap_performance = (
            calculate_patient_level_performance(
                bootstrap_outcomes,
                bootstrap_probabilities,
                bootstrap_classes
            )
        )


        for metric_name, estimate in (
            bootstrap_performance.items()
        ):

            bootstrap_records.append({
                "Model": model_name,
                "Bootstrap": bootstrap_number,
                "Metric": metric_name,
                "Estimate": float(estimate)
            })


    print(
        model_name,
        "| completed bootstrap samples:",
        NUMBER_OF_BOOTSTRAP_SAMPLES
    )


point_estimates_table = pd.DataFrame(
    point_estimate_records
)

bootstrap_estimates = pd.DataFrame(
    bootstrap_records
)


# ------------------------------------------------------------
# Percentile confidence intervals
# ------------------------------------------------------------

alpha = (
    1 - BOOTSTRAP_CONFIDENCE_LEVEL
)

lower_percentile = (
    100 * alpha / 2
)

upper_percentile = (
    100 * (
        1 - alpha / 2
    )
)


confidence_interval_table = (
    bootstrap_estimates
    .groupby(
        [
            "Model",
            "Metric"
        ]
    )["Estimate"]
    .agg(
        CI_Lower=lambda values: np.percentile(
            values,
            lower_percentile
        ),
        CI_Upper=lambda values: np.percentile(
            values,
            upper_percentile
        ),
        Successful_Bootstrap_Samples="count"
    )
    .reset_index()
)


final_performance_table = (
    point_estimates_table
    .merge(
        confidence_interval_table,
        on=[
            "Model",
            "Metric"
        ],
        how="left",
        validate="one_to_one"
    )
)


final_performance_table[
    "Model"
] = pd.Categorical(
    final_performance_table["Model"],
    categories=model_order,
    ordered=True
)


metric_order = [
    "PR_AUC",
    "ROC_AUC",
    "Brier_Score",
    "Calibration_Intercept",
    "Calibration_Slope",
    "Sensitivity",
    "Specificity",
    "Positive_Predictive_Value",
    "Negative_Predictive_Value",
    "F1_Score",
    "Balanced_Accuracy"
]


final_performance_table[
    "Metric"
] = pd.Categorical(
    final_performance_table["Metric"],
    categories=metric_order,
    ordered=True
)


final_performance_table = (
    final_performance_table
    .sort_values(
        [
            "Model",
            "Metric"
        ]
    )
    .reset_index(drop=True)
)


final_performance_table[
    "Estimate_CI"
] = (
    final_performance_table[
        "Estimate"
    ].map(lambda value: f"{value:.3f}")
    + " ("
    + final_performance_table[
        "CI_Lower"
    ].map(lambda value: f"{value:.3f}")
    + "–"
    + final_performance_table[
        "CI_Upper"
    ].map(lambda value: f"{value:.3f}")
    + ")"
)


# ------------------------------------------------------------
# Create wide manuscript table
# ------------------------------------------------------------

manuscript_performance_table = (
    final_performance_table
    .pivot(
        index="Model",
        columns="Metric",
        values="Estimate_CI"
    )
    .reindex(
        columns=metric_order
    )
    .reset_index()
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

PATIENT_LEVEL_PREDICTIONS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_patient_level_predictions.csv"
)

BOOTSTRAP_ESTIMATES_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_patient_bootstrap_estimates.csv"
)

FINAL_PERFORMANCE_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_final_performance_with_ci.xlsx"
)


patient_level_predictions.to_csv(
    PATIENT_LEVEL_PREDICTIONS_FILE,
    index=False
)

bootstrap_estimates.to_csv(
    BOOTSTRAP_ESTIMATES_FILE,
    index=False
)


with pd.ExcelWriter(
    FINAL_PERFORMANCE_FILE,
    engine="openpyxl"
) as writer:

    final_performance_table.to_excel(
        writer,
        sheet_name="Long Format",
        index=False
    )

    manuscript_performance_table.to_excel(
        writer,
        sheet_name="Manuscript Table",
        index=False
    )

    patient_level_predictions.to_excel(
        writer,
        sheet_name="Patient Predictions",
        index=False
    )


# ------------------------------------------------------------
# Display final results
# ------------------------------------------------------------

print(
    "\nFINAL PATIENT-LEVEL PERFORMANCE "
    "WITH 95% BOOTSTRAP CONFIDENCE INTERVALS"
)

print(
    "--------------------------------------------------------"
)

display(manuscript_performance_table)


print("\nFiles saved:")

print(PATIENT_LEVEL_PREDICTIONS_FILE)
print(BOOTSTRAP_ESTIMATES_FILE)
print(FINAL_PERFORMANCE_FILE)


In [ ]:
# ============================================================
# Create held-out ROC, precision-recall and calibration plots
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    auc,
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve
)


# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

model_order = [
    "Core_Ridge_Logistic",
    "Elastic_Net_Logistic",
    "Random_Forest",
    "Gradient_Boosting"
]


model_labels = {
    "Core_Ridge_Logistic": "Core ridge logistic",
    "Elastic_Net_Logistic": "Elastic-net logistic",
    "Random_Forest": "Random forest",
    "Gradient_Boosting": "Gradient boosting"
}


model_colors = {
    "Core_Ridge_Logistic": "#0072B2",
    "Elastic_Net_Logistic": "#009E73",
    "Random_Forest": "#D55E00",
    "Gradient_Boosting": "#CC79A7"
}


model_line_styles = {
    "Core_Ridge_Logistic": "-",
    "Elastic_Net_Logistic": "--",
    "Random_Forest": "-.",
    "Gradient_Boosting": ":"
}


NUMBER_OF_CALIBRATION_BINS = 10


# ------------------------------------------------------------
# Confirm that patient-level predictions are available
# ------------------------------------------------------------

required_patient_columns = [
    "Model",
    "Modeling_Patient_ID",
    "Observed_Outcome",
    "Mean_Predicted_Probability"
]


missing_patient_columns = [
    column
    for column in required_patient_columns
    if column not in patient_level_predictions.columns
]


assert not missing_patient_columns, (
    "The patient-level prediction file is missing: "
    f"{missing_patient_columns}"
)


assert (
    patient_level_predictions[
        "Mean_Predicted_Probability"
    ]
    .between(0, 1)
    .all()
), (
    "Predicted probabilities outside [0, 1] were found."
)


# ------------------------------------------------------------
# Extract calibration statistics for plot labels
# ------------------------------------------------------------

calibration_summary = (
    final_performance_table.loc[
        final_performance_table[
            "Metric"
        ].isin(
            [
                "Brier_Score",
                "Calibration_Intercept",
                "Calibration_Slope"
            ]
        ),
        [
            "Model",
            "Metric",
            "Estimate"
        ]
    ]
    .copy()
)


calibration_summary["Model"] = (
    calibration_summary["Model"].astype(str)
)


calibration_summary = (
    calibration_summary
    .pivot(
        index="Model",
        columns="Metric",
        values="Estimate"
    )
)


# ------------------------------------------------------------
# Create curve-data containers
# ------------------------------------------------------------

roc_curve_records = []
precision_recall_records = []
calibration_curve_records = []


# ------------------------------------------------------------
# Create the three-panel figure
# ------------------------------------------------------------

figure, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(19, 6.2)
)


roc_axis = axes[0]
precision_recall_axis = axes[1]
calibration_axis = axes[2]


for model_name in model_order:

    model_data = (
        patient_level_predictions.loc[
            patient_level_predictions[
                "Model"
            ] == model_name
        ]
        .sort_values("Modeling_Patient_ID")
        .reset_index(drop=True)
    )


    observed_outcomes = (
        model_data[
            "Observed_Outcome"
        ].to_numpy(dtype=int)
    )

    predicted_probabilities = (
        model_data[
            "Mean_Predicted_Probability"
        ].to_numpy(dtype=float)
    )


    # --------------------------------------------------------
    # ROC curve
    # --------------------------------------------------------

    false_positive_rate, (
        true_positive_rate
    ), roc_thresholds = roc_curve(
        observed_outcomes,
        predicted_probabilities
    )


    roc_auc = roc_auc_score(
        observed_outcomes,
        predicted_probabilities
    )


    roc_axis.plot(
        false_positive_rate,
        true_positive_rate,
        color=model_colors[model_name],
        linestyle=model_line_styles[model_name],
        linewidth=2.2,
        label=(
            f"{model_labels[model_name]} "
            f"(AUC = {roc_auc:.3f})"
        )
    )


    roc_curve_records.extend(
        {
            "Model": model_name,
            "False_Positive_Rate": float(fpr),
            "True_Positive_Rate": float(tpr),
            "Threshold": (
                float(threshold)
                if np.isfinite(threshold)
                else np.nan
            )
        }
        for fpr, tpr, threshold in zip(
            false_positive_rate,
            true_positive_rate,
            roc_thresholds
        )
    )


    # --------------------------------------------------------
    # Precision-recall curve
    # --------------------------------------------------------

    precision_values, (
        recall_values
    ), pr_thresholds = precision_recall_curve(
        observed_outcomes,
        predicted_probabilities
    )


    average_precision = average_precision_score(
        observed_outcomes,
        predicted_probabilities
    )


    precision_recall_axis.plot(
        recall_values,
        precision_values,
        color=model_colors[model_name],
        linestyle=model_line_styles[model_name],
        linewidth=2.2,
        label=(
            f"{model_labels[model_name]} "
            f"(PR-AUC = {average_precision:.3f})"
        )
    )


    for curve_position in range(
        len(precision_values)
    ):

        threshold = (
            pr_thresholds[curve_position]
            if curve_position < len(pr_thresholds)
            else np.nan
        )

        precision_recall_records.append({
            "Model": model_name,
            "Recall": float(
                recall_values[curve_position]
            ),
            "Precision": float(
                precision_values[curve_position]
            ),
            "Threshold": (
                float(threshold)
                if np.isfinite(threshold)
                else np.nan
            )
        })


    # --------------------------------------------------------
    # Calibration curve
    # --------------------------------------------------------

    observed_fraction, (
        mean_predicted_probability
    ) = calibration_curve(
        observed_outcomes,
        predicted_probabilities,
        n_bins=NUMBER_OF_CALIBRATION_BINS,
        strategy="quantile"
    )


    calibration_intercept = float(
        calibration_summary.loc[
            model_name,
            "Calibration_Intercept"
        ]
    )

    calibration_slope = float(
        calibration_summary.loc[
            model_name,
            "Calibration_Slope"
        ]
    )

    brier_score = float(
        calibration_summary.loc[
            model_name,
            "Brier_Score"
        ]
    )


    calibration_axis.plot(
        mean_predicted_probability,
        observed_fraction,
        marker="o",
        markersize=5.5,
        color=model_colors[model_name],
        linestyle=model_line_styles[model_name],
        linewidth=2.0,
        label=(
            f"{model_labels[model_name]} "
            f"(Brier = {brier_score:.3f})"
        )
    )


    for bin_number, (
        mean_probability,
        event_fraction
    ) in enumerate(
        zip(
            mean_predicted_probability,
            observed_fraction
        ),
        start=1
    ):

        calibration_curve_records.append({
            "Model": model_name,
            "Calibration_Bin": bin_number,
            "Mean_Predicted_Probability": float(
                mean_probability
            ),
            "Observed_Event_Proportion": float(
                event_fraction
            ),
            "Calibration_Intercept": (
                calibration_intercept
            ),
            "Calibration_Slope": (
                calibration_slope
            ),
            "Brier_Score": brier_score
        })


# ------------------------------------------------------------
# ROC formatting
# ------------------------------------------------------------

roc_axis.plot(
    [0, 1],
    [0, 1],
    color="#666666",
    linestyle="--",
    linewidth=1.2,
    label="No discrimination"
)

roc_axis.set_xlim(0, 1)
roc_axis.set_ylim(0, 1.02)

roc_axis.set_xlabel(
    "False-positive rate",
    fontsize=11
)

roc_axis.set_ylabel(
    "True-positive rate",
    fontsize=11
)

roc_axis.set_title(
    "A. Held-out ROC curves",
    loc="left",
    fontsize=13,
    fontweight="bold"
)

roc_axis.legend(
    loc="lower right",
    frameon=True,
    fontsize=8.5
)

roc_axis.grid(
    alpha=0.20
)


# ------------------------------------------------------------
# Precision-recall formatting
# ------------------------------------------------------------

overall_event_prevalence = float(
    patient_level_predictions
    .drop_duplicates(
        "Modeling_Patient_ID"
    )[
        "Observed_Outcome"
    ]
    .mean()
)


precision_recall_axis.axhline(
    overall_event_prevalence,
    color="#666666",
    linestyle="--",
    linewidth=1.2,
    label=(
        "No-skill prevalence "
        f"({overall_event_prevalence:.3f})"
    )
)

precision_recall_axis.set_xlim(0, 1)
precision_recall_axis.set_ylim(0, 1.02)

precision_recall_axis.set_xlabel(
    "Recall (sensitivity)",
    fontsize=11
)

precision_recall_axis.set_ylabel(
    "Precision (positive predictive value)",
    fontsize=11
)

precision_recall_axis.set_title(
    "B. Held-out precision–recall curves",
    loc="left",
    fontsize=13,
    fontweight="bold"
)

precision_recall_axis.legend(
    loc="upper right",
    frameon=True,
    fontsize=8.5
)

precision_recall_axis.grid(
    alpha=0.20
)


# ------------------------------------------------------------
# Calibration formatting
# ------------------------------------------------------------

maximum_calibration_value = max(
    0.20,
    float(
        patient_level_predictions[
            "Mean_Predicted_Probability"
        ].quantile(0.995)
    ),
    float(
        pd.DataFrame(
            calibration_curve_records
        )[
            "Observed_Event_Proportion"
        ].max()
    )
)


maximum_calibration_value = min(
    1.0,
    np.ceil(
        maximum_calibration_value
        * 10
    ) / 10
)


calibration_axis.plot(
    [0, maximum_calibration_value],
    [0, maximum_calibration_value],
    color="#222222",
    linestyle="--",
    linewidth=1.3,
    label="Perfect calibration"
)

calibration_axis.set_xlim(
    0,
    maximum_calibration_value
)

calibration_axis.set_ylim(
    0,
    maximum_calibration_value
)

calibration_axis.set_xlabel(
    "Mean predicted recurrence probability",
    fontsize=11
)

calibration_axis.set_ylabel(
    "Observed recurrence proportion",
    fontsize=11
)

calibration_axis.set_title(
    "C. Calibration of held-out predictions",
    loc="left",
    fontsize=13,
    fontweight="bold"
)

calibration_axis.legend(
    loc="upper left",
    frameon=True,
    fontsize=8.2
)

calibration_axis.grid(
    alpha=0.20
)


# ------------------------------------------------------------
# Overall formatting
# ------------------------------------------------------------

for axis in axes:

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

    axis.tick_params(
        axis="both",
        labelsize=9.5
    )


figure.suptitle(
    "Internal validation of five-year breast-cancer "
    "recurrence models",
    fontsize=15,
    fontweight="bold",
    y=1.02
)


figure.text(
    0.5,
    -0.01,
    (
        "Curves are based on each patient's mean probability "
        "across ten repeated held-out outer-fold predictions. "
        "No training predictions are shown."
    ),
    ha="center",
    fontsize=9.5
)


figure.tight_layout(
    rect=[0, 0.04, 1, 0.97]
)


# ------------------------------------------------------------
# Save figure and underlying data
# ------------------------------------------------------------

VALIDATION_FIGURE_PNG = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_validation_curves.png"
)

VALIDATION_FIGURE_PDF = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_validation_curves.pdf"
)

VALIDATION_CURVE_DATA_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_validation_curve_data.xlsx"
)


figure.savefig(
    VALIDATION_FIGURE_PNG,
    dpi=600,
    bbox_inches="tight"
)

figure.savefig(
    VALIDATION_FIGURE_PDF,
    bbox_inches="tight"
)


roc_curve_data = pd.DataFrame(
    roc_curve_records
)

precision_recall_curve_data = pd.DataFrame(
    precision_recall_records
)

calibration_curve_data = pd.DataFrame(
    calibration_curve_records
)


with pd.ExcelWriter(
    VALIDATION_CURVE_DATA_FILE,
    engine="openpyxl"
) as writer:

    roc_curve_data.to_excel(
        writer,
        sheet_name="ROC Curves",
        index=False
    )

    precision_recall_curve_data.to_excel(
        writer,
        sheet_name="PR Curves",
        index=False
    )

    calibration_curve_data.to_excel(
        writer,
        sheet_name="Calibration Curves",
        index=False
    )


plt.show()


print("\nVALIDATION FIGURES SAVED")
print("------------------------")
print(VALIDATION_FIGURE_PNG)
print(VALIDATION_FIGURE_PDF)
print(VALIDATION_CURVE_DATA_FILE)

In [ ]:
# ============================================================
# Create enclosed ROC, precision-recall and calibration plots
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


plt.close("all")


# ------------------------------------------------------------
# Shared publication formatting
# ------------------------------------------------------------

def create_enclosed_figure():

    figure, axis = plt.subplots(
        figsize=(8.5, 7.5),
        facecolor="white"
    )

    axis.set_facecolor("white")

    return figure, axis


def format_enclosed_axis(
    axis,
    title,
    x_label,
    y_label
):

    axis.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        color="black",
        pad=16
    )

    axis.set_xlabel(
        x_label,
        fontsize=13,
        color="black",
        labelpad=10
    )

    axis.set_ylabel(
        y_label,
        fontsize=13,
        color="black",
        labelpad=10
    )

    axis.tick_params(
        axis="both",
        which="major",
        labelsize=11,
        colors="black",
        direction="out",
        length=5,
        width=1.0
    )

    # Display all four sides of the plotting box.

    for spine_name in [
        "top",
        "right",
        "bottom",
        "left"
    ]:

        axis.spines[
            spine_name
        ].set_visible(True)

        axis.spines[
            spine_name
        ].set_color("black")

        axis.spines[
            spine_name
        ].set_linewidth(1.2)

    # No background or grid shading.

    axis.grid(False)


def add_internal_legend(
    axis,
    location
):

    legend = axis.legend(
        loc=location,
        borderaxespad=0.8,
        frameon=True,
        fancybox=False,
        shadow=False,
        fontsize=9.5,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
        handlelength=3.0,
        labelspacing=0.7,
        borderpad=0.8
    )

    legend.get_frame().set_linewidth(
        1.0
    )

    return legend


def save_enclosed_figure(
    figure,
    png_path,
    pdf_path
):

    figure.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False
    )

    figure.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False
    )


# ============================================================
# 1. Enclosed held-out ROC curve
# ============================================================

roc_figure, roc_axis = (
    create_enclosed_figure()
)


for model_name in model_order:

    model_roc_data = (
        roc_curve_data.loc[
            roc_curve_data[
                "Model"
            ] == model_name
        ]
        .sort_values(
            "False_Positive_Rate"
        )
    )


    false_positive_rates = (
        model_roc_data[
            "False_Positive_Rate"
        ]
        .to_numpy(dtype=float)
    )

    true_positive_rates = (
        model_roc_data[
            "True_Positive_Rate"
        ]
        .to_numpy(dtype=float)
    )


    model_auc = float(
        final_performance_table.loc[
            (
                final_performance_table[
                    "Model"
                ].astype(str) == model_name
            )
            & (
                final_performance_table[
                    "Metric"
                ].astype(str) == "ROC_AUC"
            ),
            "Estimate"
        ].iloc[0]
    )


    roc_axis.plot(
        false_positive_rates,
        true_positive_rates,
        color=model_colors[model_name],
        linestyle=model_line_styles[
            model_name
        ],
        linewidth=2.6,
        label=(
            f"{model_labels[model_name]} "
            f"(AUC = {model_auc:.3f})"
        )
    )


roc_axis.plot(
    np.array([0.0, 1.0]),
    np.array([0.0, 1.0]),
    color="#555555",
    linestyle="--",
    linewidth=1.5,
    label="No discrimination"
)


roc_axis.set_xlim(0, 1)
roc_axis.set_ylim(0, 1.02)


format_enclosed_axis(
    axis=roc_axis,
    title="Held-out ROC curves",
    x_label="False-positive rate",
    y_label="True-positive rate"
)


add_internal_legend(
    axis=roc_axis,
    location="lower right"
)


roc_figure.text(
    0.5,
    0.015,
    (
        "Each curve uses the mean of ten repeated "
        "held-out predictions per patient."
    ),
    ha="center",
    va="bottom",
    fontsize=10,
    color="black"
)


roc_figure.tight_layout(
    rect=[0.02, 0.055, 0.98, 0.98]
)


SEPARATE_ROC_PNG = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_roc_curve.png"
)

SEPARATE_ROC_PDF = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_roc_curve.pdf"
)


save_enclosed_figure(
    roc_figure,
    SEPARATE_ROC_PNG,
    SEPARATE_ROC_PDF
)


plt.show()


# ============================================================
# 2. Enclosed held-out precision-recall curve
# ============================================================

pr_figure, pr_axis = (
    create_enclosed_figure()
)


for model_name in model_order:

    model_pr_data = (
        precision_recall_curve_data.loc[
            precision_recall_curve_data[
                "Model"
            ] == model_name
        ]
    )


    recall_values = (
        model_pr_data[
            "Recall"
        ]
        .to_numpy(dtype=float)
    )

    precision_values = (
        model_pr_data[
            "Precision"
        ]
        .to_numpy(dtype=float)
    )


    model_pr_auc = float(
        final_performance_table.loc[
            (
                final_performance_table[
                    "Model"
                ].astype(str) == model_name
            )
            & (
                final_performance_table[
                    "Metric"
                ].astype(str) == "PR_AUC"
            ),
            "Estimate"
        ].iloc[0]
    )


    pr_axis.plot(
        recall_values,
        precision_values,
        color=model_colors[model_name],
        linestyle=model_line_styles[
            model_name
        ],
        linewidth=2.6,
        label=(
            f"{model_labels[model_name]} "
            f"(PR-AUC = {model_pr_auc:.3f})"
        )
    )


pr_axis.axhline(
    y=overall_event_prevalence,
    color="#555555",
    linestyle="--",
    linewidth=1.5,
    label=(
        "No-skill prevalence "
        f"({overall_event_prevalence:.3f})"
    )
)


pr_axis.set_xlim(0, 1)
pr_axis.set_ylim(0, 1.02)


format_enclosed_axis(
    axis=pr_axis,
    title="Held-out precision–recall curves",
    x_label="Recall (sensitivity)",
    y_label=(
        "Precision "
        "(positive predictive value)"
    )
)


add_internal_legend(
    axis=pr_axis,
    location="upper right"
)


pr_figure.text(
    0.5,
    0.015,
    (
        "The dashed horizontal line represents the "
        "observed recurrence prevalence."
    ),
    ha="center",
    va="bottom",
    fontsize=10,
    color="black"
)


pr_figure.tight_layout(
    rect=[0.02, 0.055, 0.98, 0.98]
)


SEPARATE_PR_PNG = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_precision_recall_curve.png"
)

SEPARATE_PR_PDF = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_precision_recall_curve.pdf"
)


save_enclosed_figure(
    pr_figure,
    SEPARATE_PR_PNG,
    SEPARATE_PR_PDF
)


plt.show()


# ============================================================
# 3. Enclosed held-out calibration plot
# ============================================================

calibration_figure, (
    calibration_axis
) = create_enclosed_figure()


for model_name in model_order:

    model_calibration_data = (
        calibration_curve_data.loc[
            calibration_curve_data[
                "Model"
            ] == model_name
        ]
        .sort_values(
            "Mean_Predicted_Probability"
        )
    )


    mean_probabilities = (
        model_calibration_data[
            "Mean_Predicted_Probability"
        ]
        .to_numpy(dtype=float)
    )

    observed_proportions = (
        model_calibration_data[
            "Observed_Event_Proportion"
        ]
        .to_numpy(dtype=float)
    )


    brier_score = float(
        model_calibration_data[
            "Brier_Score"
        ].iloc[0]
    )

    calibration_slope = float(
        model_calibration_data[
            "Calibration_Slope"
        ].iloc[0]
    )


    calibration_axis.plot(
        mean_probabilities,
        observed_proportions,
        marker="o",
        markersize=6,
        markerfacecolor="white",
        markeredgewidth=1.4,
        color=model_colors[model_name],
        linestyle=model_line_styles[
            model_name
        ],
        linewidth=2.5,
        label=(
            f"{model_labels[model_name]} "
            f"(Brier = {brier_score:.3f}; "
            f"slope = {calibration_slope:.2f})"
        )
    )


calibration_axis.plot(
    np.array(
        [
            0.0,
            maximum_calibration_value
        ]
    ),
    np.array(
        [
            0.0,
            maximum_calibration_value
        ]
    ),
    color="#222222",
    linestyle="--",
    linewidth=1.5,
    label="Perfect calibration"
)


calibration_axis.set_xlim(
    0,
    maximum_calibration_value
)

calibration_axis.set_ylim(
    0,
    maximum_calibration_value
)

calibration_axis.set_aspect(
    "equal",
    adjustable="box"
)


format_enclosed_axis(
    axis=calibration_axis,
    title=(
        "Calibration of held-out predictions"
    ),
    x_label=(
        "Mean predicted recurrence probability"
    ),
    y_label=(
        "Observed recurrence proportion"
    )
)


add_internal_legend(
    axis=calibration_axis,
    location="upper left"
)


calibration_figure.text(
    0.5,
    0.015,
    (
        "Quantile-based calibration groups; "
        "the dashed diagonal indicates "
        "perfect calibration."
    ),
    ha="center",
    va="bottom",
    fontsize=10,
    color="black"
)


calibration_figure.tight_layout(
    rect=[0.02, 0.055, 0.98, 0.98]
)


SEPARATE_CALIBRATION_PNG = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_calibration_plot.png"
)

SEPARATE_CALIBRATION_PDF = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_calibration_plot.pdf"
)


save_enclosed_figure(
    calibration_figure,
    SEPARATE_CALIBRATION_PNG,
    SEPARATE_CALIBRATION_PDF
)


plt.show()


# ------------------------------------------------------------
# Confirm saved figures
# ------------------------------------------------------------

print("\nENCLOSED VALIDATION FIGURES SAVED")
print("---------------------------------")

print("\nROC curve:")
print(SEPARATE_ROC_PNG)
print(SEPARATE_ROC_PDF)

print("\nPrecision-recall curve:")
print(SEPARATE_PR_PNG)
print(SEPARATE_PR_PDF)

print("\nCalibration plot:")
print(SEPARATE_CALIBRATION_PNG)
print(SEPARATE_CALIBRATION_PDF)

In [ ]:
# ============================================================
# Decision-curve analysis using held-out patient predictions
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Decision-curve settings
# ------------------------------------------------------------

DECISION_THRESHOLD_MINIMUM = 0.01
DECISION_THRESHOLD_MAXIMUM = 0.30
DECISION_THRESHOLD_STEP = 0.005


decision_thresholds = np.arange(
    DECISION_THRESHOLD_MINIMUM,
    DECISION_THRESHOLD_MAXIMUM
    + DECISION_THRESHOLD_STEP,
    DECISION_THRESHOLD_STEP
)


# ------------------------------------------------------------
# Decision-curve net-benefit function
# ------------------------------------------------------------

def calculate_net_benefit(
    observed_outcomes,
    predicted_probabilities,
    threshold_probability
):

    observed_outcomes = np.asarray(
        observed_outcomes,
        dtype=int
    )

    predicted_probabilities = np.asarray(
        predicted_probabilities,
        dtype=float
    )


    predicted_positive = (
        predicted_probabilities
        >= threshold_probability
    )


    true_positives = np.sum(
        predicted_positive
        & (
            observed_outcomes == 1
        )
    )

    false_positives = np.sum(
        predicted_positive
        & (
            observed_outcomes == 0
        )
    )


    number_of_patients = len(
        observed_outcomes
    )


    threshold_odds = (
        threshold_probability
        / (
            1 - threshold_probability
        )
    )


    net_benefit = (
        true_positives
        / number_of_patients
    ) - (
        false_positives
        / number_of_patients
    ) * threshold_odds


    return float(net_benefit)


# ------------------------------------------------------------
# Extract one observed outcome per patient
# ------------------------------------------------------------

patient_outcomes = (
    patient_level_predictions[
        [
            "Modeling_Patient_ID",
            "Observed_Outcome"
        ]
    ]
    .drop_duplicates(
        "Modeling_Patient_ID"
    )
    .sort_values(
        "Modeling_Patient_ID"
    )
    .reset_index(drop=True)
)


assert len(patient_outcomes) == len(
    analysis_data
), (
    "The number of unique patients is incorrect."
)


observed_outcomes = (
    patient_outcomes[
        "Observed_Outcome"
    ]
    .to_numpy(dtype=int)
)


recurrence_prevalence = float(
    observed_outcomes.mean()
)


# ------------------------------------------------------------
# Calculate model, treat-all and treat-none net benefit
# ------------------------------------------------------------

decision_curve_records = []


for threshold_probability in decision_thresholds:

    treat_none_net_benefit = 0.0


    treat_all_net_benefit = (
        recurrence_prevalence
        - (
            1 - recurrence_prevalence
        )
        * (
            threshold_probability
            / (
                1 - threshold_probability
            )
        )
    )


    decision_curve_records.append({
        "Strategy": "Treat none",
        "Threshold_Probability": float(
            threshold_probability
        ),
        "Net_Benefit": float(
            treat_none_net_benefit
        )
    })


    decision_curve_records.append({
        "Strategy": "Treat all",
        "Threshold_Probability": float(
            threshold_probability
        ),
        "Net_Benefit": float(
            treat_all_net_benefit
        )
    })


for model_name in model_order:

    model_predictions = (
        patient_level_predictions.loc[
            patient_level_predictions[
                "Model"
            ] == model_name,
            [
                "Modeling_Patient_ID",
                "Observed_Outcome",
                "Mean_Predicted_Probability"
            ]
        ]
        .sort_values(
            "Modeling_Patient_ID"
        )
        .reset_index(drop=True)
    )


    assert np.array_equal(
        model_predictions[
            "Observed_Outcome"
        ].to_numpy(dtype=int),
        observed_outcomes
    ), (
        f"Outcome ordering failed for {model_name}."
    )


    predicted_probabilities = (
        model_predictions[
            "Mean_Predicted_Probability"
        ]
        .to_numpy(dtype=float)
    )


    for threshold_probability in (
        decision_thresholds
    ):

        net_benefit = (
            calculate_net_benefit(
                observed_outcomes,
                predicted_probabilities,
                threshold_probability
            )
        )


        decision_curve_records.append({
            "Strategy": model_name,
            "Threshold_Probability": float(
                threshold_probability
            ),
            "Net_Benefit": net_benefit
        })


decision_curve_data = pd.DataFrame(
    decision_curve_records
)


# ------------------------------------------------------------
# Calculate benefit relative to treat-all and treat-none
# ------------------------------------------------------------

reference_net_benefit = (
    decision_curve_data.loc[
        decision_curve_data[
            "Strategy"
        ].isin(
            [
                "Treat all",
                "Treat none"
            ]
        )
    ]
    .pivot(
        index="Threshold_Probability",
        columns="Strategy",
        values="Net_Benefit"
    )
    .reset_index()
)


model_decision_curve_data = (
    decision_curve_data.loc[
        decision_curve_data[
            "Strategy"
        ].isin(model_order)
    ]
    .merge(
        reference_net_benefit,
        on="Threshold_Probability",
        how="left",
        validate="many_to_one"
    )
)


model_decision_curve_data[
    "Better_Than_Treat_None"
] = (
    model_decision_curve_data[
        "Net_Benefit"
    ]
    > model_decision_curve_data[
        "Treat none"
    ]
)


model_decision_curve_data[
    "Better_Than_Treat_All"
] = (
    model_decision_curve_data[
        "Net_Benefit"
    ]
    > model_decision_curve_data[
        "Treat all"
    ]
)


model_decision_curve_data[
    "Best_Reference_Net_Benefit"
] = (
    model_decision_curve_data[
        [
            "Treat none",
            "Treat all"
        ]
    ]
    .max(axis=1)
)


model_decision_curve_data[
    "Net_Benefit_Above_Best_Reference"
] = (
    model_decision_curve_data[
        "Net_Benefit"
    ]
    - model_decision_curve_data[
        "Best_Reference_Net_Benefit"
    ]
)


# ------------------------------------------------------------
# Create clean decision-curve plot
# ------------------------------------------------------------

decision_figure, (
    decision_axis
) = plt.subplots(
    figsize=(9.5, 7),
    facecolor="white"
)


decision_axis.set_facecolor(
    "white"
)


for model_name in model_order:

    model_curve = (
        decision_curve_data.loc[
            decision_curve_data[
                "Strategy"
            ] == model_name
        ]
        .sort_values(
            "Threshold_Probability"
        )
    )


    decision_axis.plot(
        model_curve[
            "Threshold_Probability"
        ].to_numpy(dtype=float),
        model_curve[
            "Net_Benefit"
        ].to_numpy(dtype=float),
        color=model_colors[model_name],
        linestyle=model_line_styles[model_name],
        linewidth=2.5,
        label=model_labels[model_name]
    )


treat_all_curve = (
    decision_curve_data.loc[
        decision_curve_data[
            "Strategy"
        ] == "Treat all"
    ]
    .sort_values(
        "Threshold_Probability"
    )
)


decision_axis.plot(
    treat_all_curve[
        "Threshold_Probability"
    ].to_numpy(dtype=float),
    treat_all_curve[
        "Net_Benefit"
    ].to_numpy(dtype=float),
    color="#555555",
    linestyle="--",
    linewidth=1.8,
    label="Treat all"
)


decision_axis.axhline(
    y=0,
    color="black",
    linestyle="-",
    linewidth=1.3,
    label="Treat none"
)


# ------------------------------------------------------------
# Set axis range without allowing extreme negative values
# to compress the clinically relevant curves
# ------------------------------------------------------------

model_and_reference_values = (
    decision_curve_data.loc[
        decision_curve_data[
            "Threshold_Probability"
        ] <= 0.20,
        "Net_Benefit"
    ]
    .to_numpy(dtype=float)
)


upper_net_benefit_limit = max(
    0.06,
    np.nanmax(
        model_and_reference_values
    ) * 1.15
)


decision_axis.set_xlim(
    DECISION_THRESHOLD_MINIMUM,
    DECISION_THRESHOLD_MAXIMUM
)

decision_axis.set_ylim(
    -0.02,
    upper_net_benefit_limit
)


decision_axis.set_xlabel(
    "Threshold probability",
    fontsize=12,
    color="black",
    labelpad=9
)

decision_axis.set_ylabel(
    "Net benefit",
    fontsize=12,
    color="black",
    labelpad=9
)

decision_axis.set_title(
    "Decision-curve analysis using held-out predictions",
    fontsize=15,
    fontweight="bold",
    color="black",
    backgroundcolor="white",
    pad=16
)


decision_axis.tick_params(
    axis="both",
    which="both",
    labelsize=11,
    colors="black",
    direction="out"
)


decision_axis.spines[
    "top"
].set_visible(False)

decision_axis.spines[
    "right"
].set_visible(False)

decision_axis.spines[
    "left"
].set_color("black")

decision_axis.spines[
    "bottom"
].set_color("black")


decision_axis.grid(False)


decision_legend = decision_axis.legend(
    loc="upper right",
    frameon=True,
    fontsize=9.5,
    facecolor="white",
    edgecolor="black",
    framealpha=1.0
)

decision_legend.get_frame().set_linewidth(
    0.8
)


decision_figure.text(
    0.5,
    0.01,
    (
        "Positive net benefit above both reference "
        "strategies indicates potential clinical utility."
    ),
    ha="center",
    va="bottom",
    fontsize=9.5,
    color="black"
)


decision_figure.tight_layout(
    rect=[0, 0.05, 1, 1]
)


# ------------------------------------------------------------
# Save figure and data
# ------------------------------------------------------------

DECISION_CURVE_PNG = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_decision_curve.png"
)

DECISION_CURVE_PDF = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_decision_curve.pdf"
)

DECISION_CURVE_DATA_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_decision_curve_data.xlsx"
)


decision_figure.savefig(
    DECISION_CURVE_PNG,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)

decision_figure.savefig(
    DECISION_CURVE_PDF,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)


with pd.ExcelWriter(
    DECISION_CURVE_DATA_FILE,
    engine="openpyxl"
) as writer:

    decision_curve_data.to_excel(
        writer,
        sheet_name="All Strategies",
        index=False
    )

    model_decision_curve_data.to_excel(
        writer,
        sheet_name="Reference Comparisons",
        index=False
    )


plt.show()


# ------------------------------------------------------------
# Summarize useful threshold ranges
# ------------------------------------------------------------

clinical_utility_summary = (
    model_decision_curve_data
    .groupby("Strategy")
    .agg(
        Evaluated_Thresholds=(
            "Threshold_Probability",
            "size"
        ),
        Better_Than_Both_Count=(
            "Net_Benefit_Above_Best_Reference",
            lambda values: int(
                (
                    values > 0
                ).sum()
            )
        ),
        Minimum_Threshold_With_Benefit=(
            "Threshold_Probability",
            lambda values: (
                float(
                    values.loc[
                        model_decision_curve_data.loc[
                            values.index,
                            "Net_Benefit_Above_Best_Reference"
                        ] > 0
                    ].min()
                )
                if (
                    model_decision_curve_data.loc[
                        values.index,
                        "Net_Benefit_Above_Best_Reference"
                    ] > 0
                ).any()
                else np.nan
            )
        ),
        Maximum_Threshold_With_Benefit=(
            "Threshold_Probability",
            lambda values: (
                float(
                    values.loc[
                        model_decision_curve_data.loc[
                            values.index,
                            "Net_Benefit_Above_Best_Reference"
                        ] > 0
                    ].max()
                )
                if (
                    model_decision_curve_data.loc[
                        values.index,
                        "Net_Benefit_Above_Best_Reference"
                    ] > 0
                ).any()
                else np.nan
            )
        ),
        Maximum_Incremental_Net_Benefit=(
            "Net_Benefit_Above_Best_Reference",
            "max"
        )
    )
    .reset_index()
    .rename(
        columns={
            "Strategy": "Model"
        }
    )
)


clinical_utility_summary[
    "Percentage_Thresholds_Better_Than_Both"
] = (
    clinical_utility_summary[
        "Better_Than_Both_Count"
    ]
    / clinical_utility_summary[
        "Evaluated_Thresholds"
    ]
    * 100
)


clinical_utility_summary[
    "Model"
] = pd.Categorical(
    clinical_utility_summary["Model"],
    categories=model_order,
    ordered=True
)


clinical_utility_summary = (
    clinical_utility_summary
    .sort_values("Model")
    .reset_index(drop=True)
)


numeric_summary_columns = (
    clinical_utility_summary
    .select_dtypes(include=np.number)
    .columns
)


clinical_utility_summary[
    numeric_summary_columns
] = clinical_utility_summary[
    numeric_summary_columns
].round(4)


print("DECISION-CURVE CLINICAL-UTILITY SUMMARY")
print("---------------------------------------")

display(clinical_utility_summary)


print("\nFiles saved:")
print(DECISION_CURVE_PNG)
print(DECISION_CURVE_PDF)
print(DECISION_CURVE_DATA_FILE)

In [ ]:
# ============================================================
# Recreate enclosed decision-curve plot with internal legend
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


plt.close("all")


# ------------------------------------------------------------
# Create enclosed plotting area
# ------------------------------------------------------------

decision_figure, (
    decision_axis
) = plt.subplots(
    figsize=(9.5, 7.5),
    facecolor="white"
)


decision_axis.set_facecolor(
    "white"
)


# ------------------------------------------------------------
# Plot model decision curves
# ------------------------------------------------------------

for model_name in model_order:

    model_curve = (
        decision_curve_data.loc[
            decision_curve_data[
                "Strategy"
            ] == model_name
        ]
        .sort_values(
            "Threshold_Probability"
        )
    )


    decision_axis.plot(
        model_curve[
            "Threshold_Probability"
        ].to_numpy(dtype=float),
        model_curve[
            "Net_Benefit"
        ].to_numpy(dtype=float),
        color=model_colors[model_name],
        linestyle=model_line_styles[model_name],
        linewidth=2.6,
        label=model_labels[model_name]
    )


# ------------------------------------------------------------
# Plot treat-all reference strategy
# ------------------------------------------------------------

treat_all_curve = (
    decision_curve_data.loc[
        decision_curve_data[
            "Strategy"
        ] == "Treat all"
    ]
    .sort_values(
        "Threshold_Probability"
    )
)


decision_axis.plot(
    treat_all_curve[
        "Threshold_Probability"
    ].to_numpy(dtype=float),
    treat_all_curve[
        "Net_Benefit"
    ].to_numpy(dtype=float),
    color="#666666",
    linestyle="--",
    linewidth=2.0,
    label="Treat all"
)


# ------------------------------------------------------------
# Plot treat-none reference strategy
# ------------------------------------------------------------

decision_axis.axhline(
    y=0,
    color="black",
    linestyle="-",
    linewidth=1.5,
    label="Treat none"
)


# ------------------------------------------------------------
# Determine suitable y-axis limits
# ------------------------------------------------------------

displayed_decision_values = (
    decision_curve_data.loc[
        decision_curve_data[
            "Threshold_Probability"
        ].between(
            DECISION_THRESHOLD_MINIMUM,
            DECISION_THRESHOLD_MAXIMUM
        ),
        "Net_Benefit"
    ]
    .to_numpy(dtype=float)
)


upper_net_benefit_limit = max(
    0.06,
    np.nanmax(
        displayed_decision_values
    ) * 1.15
)


# Prevent strongly negative treat-all values from
# compressing the clinically informative part.

lower_net_benefit_limit = -0.02


decision_axis.set_xlim(
    DECISION_THRESHOLD_MINIMUM,
    DECISION_THRESHOLD_MAXIMUM
)

decision_axis.set_ylim(
    lower_net_benefit_limit,
    upper_net_benefit_limit
)


# ------------------------------------------------------------
# Axis labels and title
# ------------------------------------------------------------

decision_axis.set_xlabel(
    "Threshold probability",
    fontsize=13,
    fontweight="normal",
    color="black",
    labelpad=10
)

decision_axis.set_ylabel(
    "Net benefit",
    fontsize=13,
    fontweight="normal",
    color="black",
    labelpad=10
)

decision_axis.set_title(
    "Decision-curve analysis using held-out predictions",
    fontsize=15,
    fontweight="bold",
    color="black",
    pad=16
)


# ------------------------------------------------------------
# Enclose the plotting area
# ------------------------------------------------------------

for spine_name in [
    "top",
    "right",
    "bottom",
    "left"
]:

    decision_axis.spines[
        spine_name
    ].set_visible(True)

    decision_axis.spines[
        spine_name
    ].set_color("black")

    decision_axis.spines[
        spine_name
    ].set_linewidth(1.2)


decision_axis.tick_params(
    axis="both",
    which="major",
    labelsize=11,
    colors="black",
    direction="out",
    length=5,
    width=1.0
)


decision_axis.tick_params(
    axis="both",
    which="minor",
    direction="out",
    length=3,
    width=0.8
)


# Remove background shading and grid lines.

decision_axis.grid(False)


# ------------------------------------------------------------
# Place legend inside the enclosed plotting area
# ------------------------------------------------------------

decision_legend = decision_axis.legend(
    loc="upper right",
    bbox_to_anchor=(0.975, 0.975),
    borderaxespad=0,
    frameon=True,
    fancybox=False,
    shadow=False,
    fontsize=10,
    facecolor="white",
    edgecolor="black",
    framealpha=1.0,
    handlelength=3.0,
    labelspacing=0.7,
    borderpad=0.8
)


decision_legend.get_frame().set_linewidth(
    1.0
)


# ------------------------------------------------------------
# Add concise methodological note
# ------------------------------------------------------------

decision_figure.text(
    0.5,
    0.015,
    (
        "A model has potential clinical utility where its "
        "net benefit exceeds both reference strategies."
    ),
    ha="center",
    va="bottom",
    fontsize=10,
    color="black"
)


decision_figure.tight_layout(
    rect=[0.02, 0.055, 0.98, 0.98]
)


# ------------------------------------------------------------
# Save revised figure
# ------------------------------------------------------------

DECISION_CURVE_PNG = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_decision_curve.png"
)

DECISION_CURVE_PDF = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_decision_curve.pdf"
)


decision_figure.savefig(
    DECISION_CURVE_PNG,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)

decision_figure.savefig(
    DECISION_CURVE_PDF,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)


plt.show()


print("REVISED DECISION-CURVE FIGURE SAVED")
print("-----------------------------------")
print(DECISION_CURVE_PNG)
print(DECISION_CURVE_PDF)

In [ ]:
# ============================================================
# Paired bootstrap comparisons between prediction models
# ============================================================

import itertools
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)


# ------------------------------------------------------------
# Paired-bootstrap settings
# ------------------------------------------------------------

NUMBER_OF_PAIRED_BOOTSTRAP_SAMPLES = 2000
PAIRED_BOOTSTRAP_RANDOM_SEED = (
    RANDOM_SEED + 20000
)


comparison_metrics = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score"
]


# ------------------------------------------------------------
# Align predictions by patient
# ------------------------------------------------------------

outcome_by_patient = (
    patient_level_predictions[
        [
            "Modeling_Patient_ID",
            "Observed_Outcome"
        ]
    ]
    .drop_duplicates(
        "Modeling_Patient_ID"
    )
    .sort_values(
        "Modeling_Patient_ID"
    )
    .reset_index(drop=True)
)


probability_matrix = (
    patient_level_predictions
    .pivot(
        index="Modeling_Patient_ID",
        columns="Model",
        values="Mean_Predicted_Probability"
    )
    .reindex(
        columns=model_order
    )
    .reset_index()
    .sort_values(
        "Modeling_Patient_ID"
    )
    .reset_index(drop=True)
)


assert np.array_equal(
    outcome_by_patient[
        "Modeling_Patient_ID"
    ].to_numpy(),
    probability_matrix[
        "Modeling_Patient_ID"
    ].to_numpy()
), (
    "Patient ordering does not match between "
    "outcomes and predictions."
)


assert probability_matrix[
    model_order
].notna().all().all(), (
    "Missing patient-level model predictions "
    "were found."
)


observed_outcomes = (
    outcome_by_patient[
        "Observed_Outcome"
    ]
    .to_numpy(dtype=int)
)


model_probability_arrays = {
    model_name: (
        probability_matrix[
            model_name
        ]
        .to_numpy(dtype=float)
    )
    for model_name in model_order
}


number_of_patients = len(
    observed_outcomes
)


# ------------------------------------------------------------
# Metric function
# ------------------------------------------------------------

def calculate_comparison_metrics(
    outcomes,
    probabilities
):

    return {
        "ROC_AUC": roc_auc_score(
            outcomes,
            probabilities
        ),

        "PR_AUC": average_precision_score(
            outcomes,
            probabilities
        ),

        "Brier_Score": brier_score_loss(
            outcomes,
            probabilities
        )
    }


# ------------------------------------------------------------
# Calculate point estimates
# ------------------------------------------------------------

model_point_metrics = {
    model_name: calculate_comparison_metrics(
        observed_outcomes,
        model_probability_arrays[
            model_name
        ]
    )
    for model_name in model_order
}


model_pairs = list(
    itertools.combinations(
        model_order,
        2
    )
)


point_difference_records = []


for model_1, model_2 in model_pairs:

    for metric_name in comparison_metrics:

        point_difference_records.append({
            "Model_1": model_1,
            "Model_2": model_2,
            "Metric": metric_name,
            "Model_1_Estimate": (
                model_point_metrics[
                    model_1
                ][metric_name]
            ),
            "Model_2_Estimate": (
                model_point_metrics[
                    model_2
                ][metric_name]
            ),
            "Difference_Model_1_Minus_Model_2": (
                model_point_metrics[
                    model_1
                ][metric_name]
                - model_point_metrics[
                    model_2
                ][metric_name]
            )
        })


point_difference_table = pd.DataFrame(
    point_difference_records
)


# ------------------------------------------------------------
# Perform paired patient-level bootstrap
# ------------------------------------------------------------

paired_random_generator = (
    np.random.default_rng(
        PAIRED_BOOTSTRAP_RANDOM_SEED
    )
)


paired_bootstrap_records = []


for bootstrap_number in range(
    1,
    NUMBER_OF_PAIRED_BOOTSTRAP_SAMPLES + 1
):

    sampled_indices = (
        paired_random_generator.choice(
            number_of_patients,
            size=number_of_patients,
            replace=True
        )
    )


    bootstrap_outcomes = (
        observed_outcomes[
            sampled_indices
        ]
    )


    # Protect ROC-AUC from an extremely rare
    # one-class bootstrap sample.

    if np.unique(
        bootstrap_outcomes
    ).size < 2:
        continue


    bootstrap_model_metrics = {}


    for model_name in model_order:

        bootstrap_probabilities = (
            model_probability_arrays[
                model_name
            ][sampled_indices]
        )


        bootstrap_model_metrics[
            model_name
        ] = calculate_comparison_metrics(
            bootstrap_outcomes,
            bootstrap_probabilities
        )


    for model_1, model_2 in model_pairs:

        for metric_name in comparison_metrics:

            bootstrap_difference = (
                bootstrap_model_metrics[
                    model_1
                ][metric_name]
                - bootstrap_model_metrics[
                    model_2
                ][metric_name]
            )


            paired_bootstrap_records.append({
                "Bootstrap": bootstrap_number,
                "Model_1": model_1,
                "Model_2": model_2,
                "Metric": metric_name,
                "Difference_Model_1_Minus_Model_2": (
                    float(
                        bootstrap_difference
                    )
                )
            })


    if bootstrap_number % 250 == 0:

        print(
            "Completed paired bootstrap sample",
            bootstrap_number,
            "of",
            NUMBER_OF_PAIRED_BOOTSTRAP_SAMPLES
        )


paired_bootstrap_differences = (
    pd.DataFrame(
        paired_bootstrap_records
    )
)


# ------------------------------------------------------------
# Summarize paired differences
# ------------------------------------------------------------

def calculate_two_sided_bootstrap_p_value(
    differences
):

    differences = np.asarray(
        differences,
        dtype=float
    )


    number_of_differences = len(
        differences
    )


    probability_nonpositive = (
        (
            np.sum(
                differences <= 0
            )
            + 1
        )
        / (
            number_of_differences
            + 1
        )
    )


    probability_nonnegative = (
        (
            np.sum(
                differences >= 0
            )
            + 1
        )
        / (
            number_of_differences
            + 1
        )
    )


    return min(
        1.0,
        2 * min(
            probability_nonpositive,
            probability_nonnegative
        )
    )


paired_interval_table = (
    paired_bootstrap_differences
    .groupby(
        [
            "Model_1",
            "Model_2",
            "Metric"
        ]
    )[
        "Difference_Model_1_Minus_Model_2"
    ]
    .agg(
        CI_Lower=lambda values: np.percentile(
            values,
            2.5
        ),
        CI_Upper=lambda values: np.percentile(
            values,
            97.5
        ),
        Bootstrap_P_Value=(
            calculate_two_sided_bootstrap_p_value
        ),
        Successful_Bootstrap_Samples="count"
    )
    .reset_index()
)


paired_comparison_table = (
    point_difference_table
    .merge(
        paired_interval_table,
        on=[
            "Model_1",
            "Model_2",
            "Metric"
        ],
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Holm adjustment for 18 pairwise tests
# ------------------------------------------------------------

def holm_adjust_p_values(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    number_of_tests = len(
        p_values
    )


    sorted_indices = np.argsort(
        p_values
    )

    sorted_p_values = (
        p_values[
            sorted_indices
        ]
    )


    adjusted_sorted = np.empty(
        number_of_tests,
        dtype=float
    )


    running_maximum = 0.0


    for rank, p_value in enumerate(
        sorted_p_values
    ):

        adjusted_value = (
            number_of_tests - rank
        ) * p_value


        running_maximum = max(
            running_maximum,
            adjusted_value
        )


        adjusted_sorted[
            rank
        ] = min(
            1.0,
            running_maximum
        )


    adjusted_p_values = np.empty(
        number_of_tests,
        dtype=float
    )


    adjusted_p_values[
        sorted_indices
    ] = adjusted_sorted


    return adjusted_p_values


paired_comparison_table[
    "Holm_Adjusted_P_Value"
] = holm_adjust_p_values(
    paired_comparison_table[
        "Bootstrap_P_Value"
    ].to_numpy(dtype=float)
)


paired_comparison_table[
    "Statistically_Significant"
] = (
    paired_comparison_table[
        "Holm_Adjusted_P_Value"
    ] < 0.05
)


# ------------------------------------------------------------
# Identify the direction favouring Model 1
# ------------------------------------------------------------

paired_comparison_table[
    "Direction_Interpretation"
] = np.where(
    paired_comparison_table[
        "Metric"
    ].isin(
        [
            "ROC_AUC",
            "PR_AUC"
        ]
    ),
    np.where(
        paired_comparison_table[
            "Difference_Model_1_Minus_Model_2"
        ] > 0,
        "Favours Model 1",
        "Favours Model 2"
    ),
    np.where(
        paired_comparison_table[
            "Difference_Model_1_Minus_Model_2"
        ] < 0,
        "Favours Model 1",
        "Favours Model 2"
    )
)


# ------------------------------------------------------------
# Add manuscript-ready confidence-interval text
# ------------------------------------------------------------

paired_comparison_table[
    "Difference_95_CI"
] = (
    paired_comparison_table[
        "Difference_Model_1_Minus_Model_2"
    ].map(
        lambda value: f"{value:.4f}"
    )
    + " ("
    + paired_comparison_table[
        "CI_Lower"
    ].map(
        lambda value: f"{value:.4f}"
    )
    + " to "
    + paired_comparison_table[
        "CI_Upper"
    ].map(
        lambda value: f"{value:.4f}"
    )
    + ")"
)


# ------------------------------------------------------------
# Order and round output
# ------------------------------------------------------------

metric_order_for_comparison = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score"
]


paired_comparison_table[
    "Metric"
] = pd.Categorical(
    paired_comparison_table["Metric"],
    categories=metric_order_for_comparison,
    ordered=True
)


paired_comparison_table = (
    paired_comparison_table
    .sort_values(
        [
            "Metric",
            "Model_1",
            "Model_2"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Primary-model comparisons
# ------------------------------------------------------------

core_ridge_comparisons = (
    paired_comparison_table.loc[
        (
            paired_comparison_table[
                "Model_1"
            ] == "Core_Ridge_Logistic"
        )
        | (
            paired_comparison_table[
                "Model_2"
            ] == "Core_Ridge_Logistic"
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# Save comparison results
# ------------------------------------------------------------

PAIRED_COMPARISON_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_paired_model_comparisons.xlsx"
)

PAIRED_BOOTSTRAP_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_paired_bootstrap_differences.csv"
)


paired_bootstrap_differences.to_csv(
    PAIRED_BOOTSTRAP_FILE,
    index=False
)


with pd.ExcelWriter(
    PAIRED_COMPARISON_FILE,
    engine="openpyxl"
) as writer:

    paired_comparison_table.to_excel(
        writer,
        sheet_name="All Comparisons",
        index=False
    )

    core_ridge_comparisons.to_excel(
        writer,
        sheet_name="Core Ridge Comparisons",
        index=False
    )

    point_estimate_table = (
        pd.DataFrame(
            model_point_metrics
        )
        .T
        .reset_index()
        .rename(
            columns={
                "index": "Model"
            }
        )
    )

    point_estimate_table.to_excel(
        writer,
        sheet_name="Model Point Estimates",
        index=False
    )


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

display_columns = [
    "Metric",
    "Model_1",
    "Model_2",
    "Model_1_Estimate",
    "Model_2_Estimate",
    "Difference_95_CI",
    "Bootstrap_P_Value",
    "Holm_Adjusted_P_Value",
    "Statistically_Significant",
    "Direction_Interpretation"
]


print(
    "PAIRED MODEL COMPARISONS WITH "
    "HOLM-ADJUSTED P-VALUES"
)

print(
    "-----------------------------------------------"
)

display(
    paired_comparison_table[
        display_columns
    ].round(
        {
            "Model_1_Estimate": 4,
            "Model_2_Estimate": 4,
            "Bootstrap_P_Value": 4,
            "Holm_Adjusted_P_Value": 4
        }
    )
)


print("\nCORE RIDGE LOGISTIC COMPARISONS")
print("-------------------------------")

display(
    core_ridge_comparisons[
        display_columns
    ].round(
        {
            "Model_1_Estimate": 4,
            "Model_2_Estimate": 4,
            "Bootstrap_P_Value": 4,
            "Holm_Adjusted_P_Value": 4
        }
    )
)


print("\nFiles saved:")
print(PAIRED_COMPARISON_FILE)
print(PAIRED_BOOTSTRAP_FILE)

In [ ]:
# ============================================================
# Summarize held-out permutation importance and stability
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Predictor display labels
# ------------------------------------------------------------

predictor_labels = {
    "Age_at_Diagnosis": "Age at diagnosis",
    "Marital_Status": "Marital status",
    "Nationality": "Nationality",
    "Menopause_Status": "Menopausal status",
    "Breast_Cancer_Family_History": "Family history",
    "Primary_Site_Group": "Primary site",
    "Laterality": "Laterality",
    "Histology_Group": "Histology",
    "Tumor_Grade": "Tumour grade",
    "Tumor_Size_mm": "Tumour size",
    "Summary_Stage": "Summary stage",
    "ER_Status": "ER status",
    "PR_Status": "PR status",
    "HER2_Status": "HER2 status",
    "Surgery": "Surgery",
    "Chemotherapy": "Chemotherapy",
    "Hormone_Therapy": "Hormone therapy"
}


# ------------------------------------------------------------
# Verify permutation-importance structure
# ------------------------------------------------------------

required_importance_columns = [
    "Model",
    "Repetition",
    "Outer_Fold",
    "Predictor",
    "Importance_Mean",
    "Importance_SD"
]


missing_importance_columns = [
    column
    for column in required_importance_columns
    if column not in permutation_importance_results.columns
]


assert not missing_importance_columns, (
    "Missing permutation-importance columns: "
    f"{missing_importance_columns}"
)


assert permutation_importance_results[
    required_importance_columns
].notna().all().all(), (
    "Missing values were found in the permutation-"
    "importance results."
)


# ------------------------------------------------------------
# Rank predictors within every model-fold evaluation
# ------------------------------------------------------------

importance_with_ranks = (
    permutation_importance_results
    .copy()
)


importance_with_ranks[
    "Within_Evaluation_Rank"
] = (
    importance_with_ranks
    .groupby(
        [
            "Model",
            "Repetition",
            "Outer_Fold"
        ]
    )[
        "Importance_Mean"
    ]
    .rank(
        method="average",
        ascending=False
    )
)


importance_with_ranks[
    "Positive_Importance"
] = (
    importance_with_ranks[
        "Importance_Mean"
    ] > 0
)


importance_with_ranks[
    "Top_Five"
] = (
    importance_with_ranks[
        "Within_Evaluation_Rank"
    ] <= 5
)


# ------------------------------------------------------------
# Aggregate across the 50 held-out evaluations per model
# ------------------------------------------------------------

importance_summary = (
    importance_with_ranks
    .groupby(
        [
            "Model",
            "Predictor"
        ]
    )
    .agg(
        Evaluations=(
            "Importance_Mean",
            "size"
        ),
        Mean_Importance=(
            "Importance_Mean",
            "mean"
        ),
        SD_Across_Evaluations=(
            "Importance_Mean",
            "std"
        ),
        Median_Importance=(
            "Importance_Mean",
            "median"
        ),
        Q1_Importance=(
            "Importance_Mean",
            lambda values: np.percentile(
                values,
                25
            )
        ),
        Q3_Importance=(
            "Importance_Mean",
            lambda values: np.percentile(
                values,
                75
            )
        ),
        Empirical_2_5_Percentile=(
            "Importance_Mean",
            lambda values: np.percentile(
                values,
                2.5
            )
        ),
        Empirical_97_5_Percentile=(
            "Importance_Mean",
            lambda values: np.percentile(
                values,
                97.5
            )
        ),
        Median_Rank=(
            "Within_Evaluation_Rank",
            "median"
        ),
        Mean_Rank=(
            "Within_Evaluation_Rank",
            "mean"
        ),
        Positive_Evaluations=(
            "Positive_Importance",
            "sum"
        ),
        Top_Five_Evaluations=(
            "Top_Five",
            "sum"
        )
    )
    .reset_index()
)


importance_summary[
    "Positive_Percentage"
] = (
    importance_summary[
        "Positive_Evaluations"
    ]
    / importance_summary[
        "Evaluations"
    ]
    * 100
)


importance_summary[
    "Top_Five_Percentage"
] = (
    importance_summary[
        "Top_Five_Evaluations"
    ]
    / importance_summary[
        "Evaluations"
    ]
    * 100
)


importance_summary[
    "Predictor_Label"
] = (
    importance_summary[
        "Predictor"
    ]
    .map(predictor_labels)
    .fillna(
        importance_summary[
            "Predictor"
        ]
    )
)


# ------------------------------------------------------------
# Assign overall rank using median held-out importance
# ------------------------------------------------------------

importance_summary[
    "Overall_Rank"
] = (
    importance_summary
    .groupby("Model")[
        "Median_Importance"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


importance_summary[
    "Model"
] = pd.Categorical(
    importance_summary["Model"],
    categories=model_order,
    ordered=True
)


importance_summary = (
    importance_summary
    .sort_values(
        [
            "Model",
            "Overall_Rank"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Check expected evaluation counts
# ------------------------------------------------------------

assert (
    importance_summary[
        "Evaluations"
    ] == 50
).all(), (
    "Each model-predictor combination should contain "
    "50 outer-fold evaluations."
)


# ------------------------------------------------------------
# Define stable predictors
# ------------------------------------------------------------

importance_summary[
    "Stable_Positive_Importance"
] = (
    importance_summary[
        "Positive_Percentage"
    ] >= 75
)


importance_summary[
    "Stable_Top_Five"
] = (
    importance_summary[
        "Top_Five_Percentage"
    ] >= 75
)


# ------------------------------------------------------------
# Display ranked importance tables
# ------------------------------------------------------------

importance_display_columns = [
    "Model",
    "Overall_Rank",
    "Predictor_Label",
    "Median_Importance",
    "Q1_Importance",
    "Q3_Importance",
    "Median_Rank",
    "Positive_Percentage",
    "Top_Five_Percentage",
    "Stable_Positive_Importance",
    "Stable_Top_Five"
]


print("HELD-OUT PERMUTATION-IMPORTANCE SUMMARY")
print("---------------------------------------")

display(
    importance_summary[
        importance_display_columns
    ].round(4)
)


print("\nCORE RIDGE LOGISTIC IMPORTANCE")
print("------------------------------")

core_ridge_importance = (
    importance_summary.loc[
        importance_summary[
            "Model"
        ].astype(str)
        == "Core_Ridge_Logistic"
    ]
    .sort_values(
        "Overall_Rank"
    )
    .reset_index(drop=True)
)


display(
    core_ridge_importance[
        importance_display_columns
    ].round(4)
)


# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

TOP_PREDICTORS_PER_MODEL = 10


def format_importance_axis(
    axis,
    title
):

    axis.set_facecolor("white")

    axis.set_title(
        title,
        fontsize=13,
        fontweight="bold",
        color="black",
        pad=12
    )

    axis.set_xlabel(
        "Median reduction in held-out PR-AUC",
        fontsize=11,
        color="black",
        labelpad=8
    )

    axis.set_ylabel(
        "",
        fontsize=11
    )

    axis.tick_params(
        axis="both",
        which="major",
        labelsize=9.5,
        colors="black",
        direction="out"
    )

    for spine_name in [
        "top",
        "right",
        "bottom",
        "left"
    ]:

        axis.spines[
            spine_name
        ].set_visible(True)

        axis.spines[
            spine_name
        ].set_color("black")

        axis.spines[
            spine_name
        ].set_linewidth(1.0)

    axis.grid(False)

    axis.axvline(
        x=0,
        color="black",
        linewidth=0.9,
        linestyle="-"
    )


# ------------------------------------------------------------
# Create combined four-panel importance figure
# ------------------------------------------------------------

importance_figure, importance_axes = (
    plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(15, 12),
        facecolor="white"
    )
)


importance_axes = (
    importance_axes.flatten()
)


panel_letters = [
    "A",
    "B",
    "C",
    "D"
]


for axis, panel_letter, model_name in zip(
    importance_axes,
    panel_letters,
    model_order
):

    model_importance = (
        importance_summary.loc[
            importance_summary[
                "Model"
            ].astype(str)
            == model_name
        ]
        .nsmallest(
            TOP_PREDICTORS_PER_MODEL,
            "Overall_Rank"
        )
        .sort_values(
            "Median_Importance",
            ascending=True
        )
        .copy()
    )


    median_values = (
        model_importance[
            "Median_Importance"
        ]
        .to_numpy(dtype=float)
    )

    lower_errors = (
        median_values
        - model_importance[
            "Q1_Importance"
        ].to_numpy(dtype=float)
    )

    upper_errors = (
        model_importance[
            "Q3_Importance"
        ].to_numpy(dtype=float)
        - median_values
    )


    lower_errors = np.maximum(
        lower_errors,
        0
    )

    upper_errors = np.maximum(
        upper_errors,
        0
    )


    axis.barh(
        y=np.arange(
            len(model_importance)
        ),
        width=median_values,
        color=model_colors[model_name],
        edgecolor="black",
        linewidth=0.7,
        alpha=0.90,
        xerr=np.vstack(
            [
                lower_errors,
                upper_errors
            ]
        ),
        error_kw={
            "ecolor": "black",
            "elinewidth": 1.0,
            "capsize": 3,
            "capthick": 1.0
        }
    )


    axis.set_yticks(
        np.arange(
            len(model_importance)
        )
    )

    axis.set_yticklabels(
        model_importance[
            "Predictor_Label"
        ].tolist()
    )


    format_importance_axis(
        axis,
        (
            f"{panel_letter}. "
            f"{model_labels[model_name]}"
        )
    )


importance_figure.suptitle(
    "Held-out permutation importance across "
    "repeated nested validation",
    fontsize=16,
    fontweight="bold",
    color="black",
    y=0.995
)


importance_figure.text(
    0.5,
    0.015,
    (
        "Bars show median reduction in held-out PR-AUC; "
        "error bars show the interquartile range across "
        "50 outer-fold evaluations."
    ),
    ha="center",
    fontsize=10,
    color="black"
)


importance_figure.tight_layout(
    rect=[0.02, 0.045, 0.98, 0.97],
    h_pad=3.0,
    w_pad=3.0
)


# ------------------------------------------------------------
# Save combined importance figure
# ------------------------------------------------------------

IMPORTANCE_FIGURE_PNG = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_permutation_importance.png"
)

IMPORTANCE_FIGURE_PDF = (
    FIGURES_DIRECTORY
    / "unweighted_held_out_permutation_importance.pdf"
)


importance_figure.savefig(
    IMPORTANCE_FIGURE_PNG,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)

importance_figure.savefig(
    IMPORTANCE_FIGURE_PDF,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)


plt.show()


# ------------------------------------------------------------
# Create separate importance plot for each model
# ------------------------------------------------------------

separate_importance_files = []


for model_name in model_order:

    model_importance = (
        importance_summary.loc[
            importance_summary[
                "Model"
            ].astype(str)
            == model_name
        ]
        .nsmallest(
            TOP_PREDICTORS_PER_MODEL,
            "Overall_Rank"
        )
        .sort_values(
            "Median_Importance",
            ascending=True
        )
        .copy()
    )


    median_values = (
        model_importance[
            "Median_Importance"
        ]
        .to_numpy(dtype=float)
    )

    lower_errors = (
        median_values
        - model_importance[
            "Q1_Importance"
        ].to_numpy(dtype=float)
    )

    upper_errors = (
        model_importance[
            "Q3_Importance"
        ].to_numpy(dtype=float)
        - median_values
    )


    lower_errors = np.maximum(
        lower_errors,
        0
    )

    upper_errors = np.maximum(
        upper_errors,
        0
    )


    model_figure, model_axis = (
        plt.subplots(
            figsize=(9, 7),
            facecolor="white"
        )
    )


    model_axis.barh(
        y=np.arange(
            len(model_importance)
        ),
        width=median_values,
        color=model_colors[model_name],
        edgecolor="black",
        linewidth=0.8,
        alpha=0.90,
        xerr=np.vstack(
            [
                lower_errors,
                upper_errors
            ]
        ),
        error_kw={
            "ecolor": "black",
            "elinewidth": 1.1,
            "capsize": 3,
            "capthick": 1.0
        }
    )


    model_axis.set_yticks(
        np.arange(
            len(model_importance)
        )
    )

    model_axis.set_yticklabels(
        model_importance[
            "Predictor_Label"
        ].tolist()
    )


    format_importance_axis(
        model_axis,
        (
            "Held-out permutation importance: "
            f"{model_labels[model_name]}"
        )
    )


    model_figure.text(
        0.5,
        0.015,
        (
            "Bars show median reduction in held-out "
            "PR-AUC; error bars show the interquartile "
            "range across 50 evaluations."
        ),
        ha="center",
        fontsize=9.5,
        color="black"
    )


    model_figure.tight_layout(
        rect=[0.02, 0.055, 0.98, 0.98]
    )


    model_png_file = (
        FIGURES_DIRECTORY
        / (
            "unweighted_permutation_importance_"
            f"{model_name.lower()}.png"
        )
    )

    model_pdf_file = (
        FIGURES_DIRECTORY
        / (
            "unweighted_permutation_importance_"
            f"{model_name.lower()}.pdf"
        )
    )


    model_figure.savefig(
        model_png_file,
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False
    )

    model_figure.savefig(
        model_pdf_file,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False
    )


    separate_importance_files.extend(
        [
            model_png_file,
            model_pdf_file
        ]
    )


    plt.show()


# ------------------------------------------------------------
# Save detailed importance tables
# ------------------------------------------------------------

IMPORTANCE_SUMMARY_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_permutation_importance_summary.xlsx"
)


with pd.ExcelWriter(
    IMPORTANCE_SUMMARY_FILE,
    engine="openpyxl"
) as writer:

    importance_summary.to_excel(
        writer,
        sheet_name="All Models",
        index=False
    )

    core_ridge_importance.to_excel(
        writer,
        sheet_name="Core Ridge",
        index=False
    )

    importance_with_ranks.to_excel(
        writer,
        sheet_name="All Fold Results",
        index=False
    )


# ------------------------------------------------------------
# Report saved outputs
# ------------------------------------------------------------

print("\nPERMUTATION-IMPORTANCE RESULTS SAVED")
print("------------------------------------")

print(IMPORTANCE_SUMMARY_FILE)
print(IMPORTANCE_FIGURE_PNG)
print(IMPORTANCE_FIGURE_PDF)

print("\nSeparate model figures:")

for saved_file in separate_importance_files:
    print(saved_file)

In [ ]:
# ============================================================
# Summarize logistic-coefficient direction and stability
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Verify coefficient data
# ------------------------------------------------------------

required_coefficient_columns = [
    "Model",
    "Repetition",
    "Outer_Fold",
    "Encoded_Feature",
    "Coefficient",
    "Nonzero"
]


missing_coefficient_columns = [
    column
    for column in required_coefficient_columns
    if column not in logistic_coefficient_results.columns
]


assert not missing_coefficient_columns, (
    "Missing logistic-coefficient columns: "
    f"{missing_coefficient_columns}"
)


# ------------------------------------------------------------
# Create readable encoded-feature labels
# ------------------------------------------------------------

def clean_encoded_feature_name(
    feature_name
):

    cleaned_name = str(
        feature_name
    )


    prefixes = [
        "continuous__",
        "categorical__",
        "numeric__",
        "cat__",
        "num__"
    ]


    for prefix in prefixes:

        if cleaned_name.startswith(
            prefix
        ):

            cleaned_name = cleaned_name[
                len(prefix):
            ]


    variable_replacements = {
        "Age_at_Diagnosis": "Age at diagnosis",
        "Tumor_Size_mm": "Tumour size",
        "Tumor_Grade": "Tumour grade",
        "Summary_Stage": "Summary stage",
        "ER_Status": "ER status",
        "PR_Status": "PR status",
        "HER2_Status": "HER2 status",
        "Marital_Status": "Marital status",
        "Nationality": "Nationality",
        "Menopause_Status": "Menopausal status",
        "Breast_Cancer_Family_History": "Family history",
        "Primary_Site_Group": "Primary site",
        "Laterality": "Laterality",
        "Histology_Group": "Histology",
        "Surgery": "Surgery",
        "Chemotherapy": "Chemotherapy",
        "Hormone_Therapy": "Hormone therapy"
    }


    for original_name, readable_name in (
        variable_replacements.items()
    ):

        if cleaned_name == original_name:

            cleaned_name = readable_name

            break

        if cleaned_name.startswith(
            original_name + "_"
        ):

            category_name = cleaned_name[
                len(original_name) + 1:
            ]

            category_name = (
                category_name
                .replace("_", " ")
            )

            cleaned_name = (
                f"{readable_name}: "
                f"{category_name}"
            )

            break


    return cleaned_name


coefficient_data = (
    logistic_coefficient_results
    .copy()
)


coefficient_data[
    "Feature_Label"
] = (
    coefficient_data[
        "Encoded_Feature"
    ]
    .apply(
        clean_encoded_feature_name
    )
)


coefficient_data[
    "Positive_Coefficient"
] = (
    coefficient_data[
        "Coefficient"
    ] > 0
)


coefficient_data[
    "Negative_Coefficient"
] = (
    coefficient_data[
        "Coefficient"
    ] < 0
)


# ------------------------------------------------------------
# Summarize coefficient magnitude, direction and stability
# ------------------------------------------------------------

coefficient_summary = (
    coefficient_data
    .groupby(
        [
            "Model",
            "Encoded_Feature",
            "Feature_Label"
        ]
    )
    .agg(
        Evaluations_Available=(
            "Coefficient",
            "size"
        ),
        Mean_Coefficient=(
            "Coefficient",
            "mean"
        ),
        SD_Coefficient=(
            "Coefficient",
            "std"
        ),
        Median_Coefficient=(
            "Coefficient",
            "median"
        ),
        Q1_Coefficient=(
            "Coefficient",
            lambda values: np.percentile(
                values,
                25
            )
        ),
        Q3_Coefficient=(
            "Coefficient",
            lambda values: np.percentile(
                values,
                75
            )
        ),
        Empirical_2_5_Percentile=(
            "Coefficient",
            lambda values: np.percentile(
                values,
                2.5
            )
        ),
        Empirical_97_5_Percentile=(
            "Coefficient",
            lambda values: np.percentile(
                values,
                97.5
            )
        ),
        Nonzero_Evaluations=(
            "Nonzero",
            "sum"
        ),
        Positive_Evaluations=(
            "Positive_Coefficient",
            "sum"
        ),
        Negative_Evaluations=(
            "Negative_Coefficient",
            "sum"
        )
    )
    .reset_index()
)


coefficient_summary[
    "Availability_Percentage"
] = (
    coefficient_summary[
        "Evaluations_Available"
    ]
    / 50
    * 100
)


coefficient_summary[
    "Nonzero_Percentage"
] = (
    coefficient_summary[
        "Nonzero_Evaluations"
    ]
    / coefficient_summary[
        "Evaluations_Available"
    ]
    * 100
)


coefficient_summary[
    "Positive_Percentage"
] = (
    coefficient_summary[
        "Positive_Evaluations"
    ]
    / coefficient_summary[
        "Evaluations_Available"
    ]
    * 100
)


coefficient_summary[
    "Negative_Percentage"
] = (
    coefficient_summary[
        "Negative_Evaluations"
    ]
    / coefficient_summary[
        "Evaluations_Available"
    ]
    * 100
)


coefficient_summary[
    "Sign_Stability_Percentage"
] = (
    coefficient_summary[
        [
            "Positive_Percentage",
            "Negative_Percentage"
        ]
    ]
    .max(axis=1)
)


coefficient_summary[
    "Dominant_Direction"
] = np.where(
    coefficient_summary[
        "Positive_Percentage"
    ]
    >= coefficient_summary[
        "Negative_Percentage"
    ],
    "Higher predicted recurrence probability",
    "Lower predicted recurrence probability"
)


coefficient_summary[
    "Absolute_Median_Coefficient"
] = np.abs(
    coefficient_summary[
        "Median_Coefficient"
    ]
)


coefficient_summary[
    "Stable_Direction"
] = (
    coefficient_summary[
        "Sign_Stability_Percentage"
    ] >= 75
)


# ------------------------------------------------------------
# Order summaries
# ------------------------------------------------------------

logistic_model_order = [
    "Core_Ridge_Logistic",
    "Elastic_Net_Logistic"
]


coefficient_summary[
    "Model"
] = pd.Categorical(
    coefficient_summary["Model"],
    categories=logistic_model_order,
    ordered=True
)


coefficient_summary = (
    coefficient_summary
    .sort_values(
        [
            "Model",
            "Absolute_Median_Coefficient"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Core ridge coefficient summary
# ------------------------------------------------------------

core_ridge_coefficient_summary = (
    coefficient_summary.loc[
        coefficient_summary[
            "Model"
        ].astype(str)
        == "Core_Ridge_Logistic"
    ]
    .sort_values(
        "Median_Coefficient"
    )
    .reset_index(drop=True)
)


coefficient_display_columns = [
    "Feature_Label",
    "Evaluations_Available",
    "Median_Coefficient",
    "Q1_Coefficient",
    "Q3_Coefficient",
    "Positive_Percentage",
    "Negative_Percentage",
    "Sign_Stability_Percentage",
    "Dominant_Direction",
    "Stable_Direction"
]


print("CORE RIDGE COEFFICIENT DIRECTION AND STABILITY")
print("----------------------------------------------")

display(
    core_ridge_coefficient_summary[
        coefficient_display_columns
    ].round(4)
)


# ------------------------------------------------------------
# Create enclosed core-ridge coefficient plot
# ------------------------------------------------------------

coefficient_plot_data = (
    core_ridge_coefficient_summary
    .sort_values(
        "Median_Coefficient",
        ascending=True
    )
    .copy()
)


median_coefficients = (
    coefficient_plot_data[
        "Median_Coefficient"
    ]
    .to_numpy(dtype=float)
)


lower_errors = (
    median_coefficients
    - coefficient_plot_data[
        "Q1_Coefficient"
    ].to_numpy(dtype=float)
)


upper_errors = (
    coefficient_plot_data[
        "Q3_Coefficient"
    ].to_numpy(dtype=float)
    - median_coefficients
)


lower_errors = np.maximum(
    lower_errors,
    0
)

upper_errors = np.maximum(
    upper_errors,
    0
)


coefficient_colors = np.where(
    median_coefficients >= 0,
    "#D55E00",
    "#0072B2"
)


coefficient_figure, (
    coefficient_axis
) = plt.subplots(
    figsize=(10, 10),
    facecolor="white"
)


coefficient_axis.set_facecolor(
    "white"
)


coefficient_axis.barh(
    y=np.arange(
        len(coefficient_plot_data)
    ),
    width=median_coefficients,
    color=coefficient_colors,
    edgecolor="black",
    linewidth=0.8,
    xerr=np.vstack(
        [
            lower_errors,
            upper_errors
        ]
    ),
    error_kw={
        "ecolor": "black",
        "elinewidth": 1.0,
        "capsize": 3,
        "capthick": 1.0
    }
)


coefficient_axis.set_yticks(
    np.arange(
        len(coefficient_plot_data)
    )
)


coefficient_axis.set_yticklabels(
    coefficient_plot_data[
        "Feature_Label"
    ].tolist(),
    fontsize=9.5
)


coefficient_axis.axvline(
    x=0,
    color="black",
    linestyle="-",
    linewidth=1.1
)


coefficient_axis.set_xlabel(
    "Median regularized logistic coefficient",
    fontsize=12,
    color="black",
    labelpad=10
)


coefficient_axis.set_ylabel(
    "",
    fontsize=12
)


coefficient_axis.set_title(
    "Core ridge coefficient direction and stability",
    fontsize=15,
    fontweight="bold",
    color="black",
    pad=16
)


for spine_name in [
    "top",
    "right",
    "bottom",
    "left"
]:

    coefficient_axis.spines[
        spine_name
    ].set_visible(True)

    coefficient_axis.spines[
        spine_name
    ].set_color("black")

    coefficient_axis.spines[
        spine_name
    ].set_linewidth(1.2)


coefficient_axis.tick_params(
    axis="x",
    labelsize=10.5,
    colors="black",
    direction="out"
)


coefficient_axis.tick_params(
    axis="y",
    colors="black",
    direction="out"
)


coefficient_axis.grid(False)


coefficient_figure.text(
    0.5,
    0.015,
    (
        "Orange indicates higher and blue indicates lower "
        "predicted recurrence probability. Error bars show "
        "the interquartile range across 50 outer-fold models."
    ),
    ha="center",
    fontsize=9.5,
    color="black"
)


coefficient_figure.tight_layout(
    rect=[0.02, 0.055, 0.98, 0.98]
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

COEFFICIENT_FIGURE_PNG = (
    FIGURES_DIRECTORY
    / "unweighted_core_ridge_coefficient_stability.png"
)

COEFFICIENT_FIGURE_PDF = (
    FIGURES_DIRECTORY
    / "unweighted_core_ridge_coefficient_stability.pdf"
)

COEFFICIENT_SUMMARY_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_logistic_coefficient_summary.xlsx"
)


coefficient_figure.savefig(
    COEFFICIENT_FIGURE_PNG,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)

coefficient_figure.savefig(
    COEFFICIENT_FIGURE_PDF,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)


with pd.ExcelWriter(
    COEFFICIENT_SUMMARY_FILE,
    engine="openpyxl"
) as writer:

    coefficient_summary.to_excel(
        writer,
        sheet_name="All Logistic Models",
        index=False
    )

    core_ridge_coefficient_summary.to_excel(
        writer,
        sheet_name="Core Ridge",
        index=False
    )

    coefficient_data.to_excel(
        writer,
        sheet_name="All Fold Coefficients",
        index=False
    )


plt.show()


print("\nCOEFFICIENT RESULTS SAVED")
print("-------------------------")
print(COEFFICIENT_FIGURE_PNG)
print(COEFFICIENT_FIGURE_PDF)
print(COEFFICIENT_SUMMARY_FILE)

In [ ]:
# ============================================================
# Leave-one-predictor-out ablation study: core ridge model
# ============================================================

import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

ABLATION_EXPECTED_EVALUATIONS = (
    len(core_predictors)
    * OUTER_FOLDS
    * OUTER_REPEATS
)

ABLATION_EXPECTED_PREDICTIONS = (
    len(core_predictors)
    * len(analysis_data)
    * OUTER_REPEATS
)


ABLATION_PERFORMANCE_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_core_ridge_ablation_performance.csv"
)

ABLATION_PREDICTIONS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_core_ridge_ablation_predictions.csv"
)

ABLATION_HYPERPARAMETERS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_core_ridge_ablation_hyperparameters.csv"
)


warnings.filterwarnings(
    "ignore",
    message=(
        "Found unknown categories in columns .* "
        "during transform.*"
    ),
    category=UserWarning,
    module="sklearn.preprocessing._encoders"
)


# ------------------------------------------------------------
# Hyperparameter space identical to the primary core ridge model
# ------------------------------------------------------------

ablation_ridge_parameter_space = {
    "model__C": np.logspace(
        -3,
        3,
        9
    ).tolist()
}


# ------------------------------------------------------------
# Define the nine ablation configurations
# ------------------------------------------------------------

ablation_configurations = {}


for omitted_predictor in core_predictors:

    retained_predictors = [
        predictor
        for predictor in core_predictors
        if predictor != omitted_predictor
    ]


    ablation_name = (
        "Without_"
        + omitted_predictor
    )


    ablation_configurations[
        ablation_name
    ] = {
        "Omitted_Predictor": (
            omitted_predictor
        ),
        "Retained_Predictors": (
            retained_predictors
        )
    }


assert len(
    ablation_configurations
) == len(
    core_predictors
), (
    "The number of ablation configurations "
    "is incorrect."
)


assert all(
    len(
        configuration[
            "Retained_Predictors"
        ]
    ) == (
        len(core_predictors) - 1
    )
    for configuration
    in ablation_configurations.values()
), (
    "Each ablation model must retain exactly "
    "eight of the nine core predictors."
)


# ------------------------------------------------------------
# Load existing ablation checkpoints
# ------------------------------------------------------------

def load_ablation_checkpoint(
    file_path
):

    if file_path.exists():

        return pd.read_csv(
            file_path
        )

    return pd.DataFrame()


ablation_performance = (
    load_ablation_checkpoint(
        ABLATION_PERFORMANCE_FILE
    )
)

ablation_predictions = (
    load_ablation_checkpoint(
        ABLATION_PREDICTIONS_FILE
    )
)

ablation_hyperparameters = (
    load_ablation_checkpoint(
        ABLATION_HYPERPARAMETERS_FILE
    )
)


# ------------------------------------------------------------
# Identify committed evaluations
#
# The performance file is written last and therefore acts
# as the completion record for each evaluation.
# ------------------------------------------------------------

ablation_key_columns = [
    "Ablation",
    "Repetition",
    "Outer_Fold"
]


completed_ablation_evaluations = set()


if not ablation_performance.empty:

    completed_ablation_evaluations = set(
        map(
            tuple,
            ablation_performance[
                ablation_key_columns
            ].to_numpy()
        )
    )


print("CORE RIDGE ABLATION STUDY")
print("-------------------------")

print(
    "Previously completed evaluations:",
    len(
        completed_ablation_evaluations
    ),
    "of",
    ABLATION_EXPECTED_EVALUATIONS
)


print("\nAblation configurations:")


for ablation_name, configuration in (
    ablation_configurations.items()
):

    print(
        "-",
        ablation_name,
        "| omitted:",
        configuration[
            "Omitted_Predictor"
        ]
    )


# ------------------------------------------------------------
# Recreate deterministic outer splits
# ------------------------------------------------------------

ablation_outer_cross_validation = (
    RepeatedStratifiedKFold(
        n_splits=OUTER_FOLDS,
        n_repeats=OUTER_REPEATS,
        random_state=RANDOM_SEED
    )
)


ablation_start_time = (
    time.perf_counter()
)


# ------------------------------------------------------------
# Outer validation loop
# ------------------------------------------------------------

for split_number, (
    outer_training_indices,
    outer_validation_indices
) in enumerate(
    ablation_outer_cross_validation.split(
        X,
        y
    ),
    start=1
):

    repetition_number = (
        (split_number - 1) // OUTER_FOLDS
    ) + 1

    outer_fold_number = (
        (split_number - 1) % OUTER_FOLDS
    ) + 1


    # Use the same inner-CV seed as the primary
    # core ridge model for this repetition and fold.

    evaluation_seed = (
        RANDOM_SEED
        + repetition_number * 1000
        + outer_fold_number * 100
        + 1
    )


    inner_cross_validation = (
        StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=evaluation_seed
        )
    )


    for ablation_name, configuration in (
        ablation_configurations.items()
    ):

        evaluation_key = (
            ablation_name,
            repetition_number,
            outer_fold_number
        )


        if (
            evaluation_key
            in completed_ablation_evaluations
        ):

            continue


        completed_before_current = len(
            completed_ablation_evaluations
        )


        print(
            "\nAblation evaluation "
            f"{completed_before_current + 1}/"
            f"{ABLATION_EXPECTED_EVALUATIONS}"
        )

        print(
            ablation_name,
            "| repetition",
            repetition_number,
            "| outer fold",
            outer_fold_number
        )


        omitted_predictor = (
            configuration[
                "Omitted_Predictor"
            ]
        )

        retained_predictors = (
            configuration[
                "Retained_Predictors"
            ]
        )


        # ----------------------------------------------------
        # Remove any incomplete orphan records for this key
        # ----------------------------------------------------

        if not ablation_predictions.empty:

            incomplete_mask = (
                (
                    ablation_predictions[
                        "Ablation"
                    ] == ablation_name
                )
                & (
                    ablation_predictions[
                        "Repetition"
                    ] == repetition_number
                )
                & (
                    ablation_predictions[
                        "Outer_Fold"
                    ] == outer_fold_number
                )
            )

            ablation_predictions = (
                ablation_predictions.loc[
                    ~incomplete_mask
                ]
                .reset_index(drop=True)
            )


        if not ablation_hyperparameters.empty:

            incomplete_mask = (
                (
                    ablation_hyperparameters[
                        "Ablation"
                    ] == ablation_name
                )
                & (
                    ablation_hyperparameters[
                        "Repetition"
                    ] == repetition_number
                )
                & (
                    ablation_hyperparameters[
                        "Outer_Fold"
                    ] == outer_fold_number
                )
            )

            ablation_hyperparameters = (
                ablation_hyperparameters.loc[
                    ~incomplete_mask
                ]
                .reset_index(drop=True)
            )


        # ----------------------------------------------------
        # Construct leakage-safe preprocessing
        # ----------------------------------------------------

        ablation_preprocessor = (
            create_preprocessor(
                selected_predictors=(
                    retained_predictors
                ),
                drop_reference_category=True
            )
        )


        ablation_pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    ablation_preprocessor
                ),
                (
                    "model",
                    LogisticRegression(
                        penalty="l2",
                        solver="liblinear",
                        class_weight=None,
                        max_iter=5000,
                        random_state=(
                            evaluation_seed
                        )
                    )
                )
            ]
        )


        # ----------------------------------------------------
        # Extract the current outer split
        # ----------------------------------------------------

        X_outer_training = (
            X.iloc[
                outer_training_indices
            ][
                retained_predictors
            ]
            .copy()
        )

        y_outer_training = (
            y.iloc[
                outer_training_indices
            ]
            .copy()
        )

        X_outer_validation = (
            X.iloc[
                outer_validation_indices
            ][
                retained_predictors
            ]
            .copy()
        )

        y_outer_validation = (
            y.iloc[
                outer_validation_indices
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Inner hyperparameter tuning
        # ----------------------------------------------------

        ablation_search = GridSearchCV(
            estimator=ablation_pipeline,
            param_grid=(
                ablation_ridge_parameter_space
            ),
            scoring="average_precision",
            cv=inner_cross_validation,
            refit=True,
            n_jobs=1,
            return_train_score=False,
            error_score="raise"
        )


        evaluation_start_time = (
            time.perf_counter()
        )


        ablation_search.fit(
            X_outer_training,
            y_outer_training
        )


        best_ablation_estimator = (
            ablation_search.best_estimator_
        )


        # ----------------------------------------------------
        # Generate held-out outer-fold probabilities
        # ----------------------------------------------------

        outer_probabilities = (
            best_ablation_estimator
            .predict_proba(
                X_outer_validation
            )[:, 1]
        )


        elapsed_seconds = (
            time.perf_counter()
            - evaluation_start_time
        )


        held_out_pr_auc = (
            average_precision_score(
                y_outer_validation,
                outer_probabilities
            )
        )

        held_out_roc_auc = (
            roc_auc_score(
                y_outer_validation,
                outer_probabilities
            )
        )

        held_out_brier_score = (
            brier_score_loss(
                y_outer_validation,
                outer_probabilities
            )
        )


        # ----------------------------------------------------
        # Create checkpoint records
        # ----------------------------------------------------

        performance_record = {
            "Ablation": ablation_name,
            "Omitted_Predictor": (
                omitted_predictor
            ),
            "Repetition": (
                repetition_number
            ),
            "Outer_Fold": (
                outer_fold_number
            ),
            "Training_Patients": int(
                len(
                    outer_training_indices
                )
            ),
            "Training_Events": int(
                y_outer_training.sum()
            ),
            "Validation_Patients": int(
                len(
                    outer_validation_indices
                )
            ),
            "Validation_Events": int(
                y_outer_validation.sum()
            ),
            "Inner_Best_PR_AUC": float(
                ablation_search.best_score_
            ),
            "Held_Out_PR_AUC": float(
                held_out_pr_auc
            ),
            "Held_Out_ROC_AUC": float(
                held_out_roc_auc
            ),
            "Held_Out_Brier_Score": float(
                held_out_brier_score
            ),
            "Elapsed_Seconds": float(
                elapsed_seconds
            )
        }


        prediction_records = pd.DataFrame({
            "Ablation": ablation_name,
            "Omitted_Predictor": (
                omitted_predictor
            ),
            "Repetition": (
                repetition_number
            ),
            "Outer_Fold": (
                outer_fold_number
            ),
            "Modeling_Patient_ID": (
                patient_id.iloc[
                    outer_validation_indices
                ].to_numpy()
            ),
            "Observed_Outcome": (
                y_outer_validation.to_numpy()
            ),
            "Predicted_Probability": (
                outer_probabilities
            )
        })


        hyperparameter_record = {
            "Ablation": ablation_name,
            "Omitted_Predictor": (
                omitted_predictor
            ),
            "Repetition": (
                repetition_number
            ),
            "Outer_Fold": (
                outer_fold_number
            ),
            "Inner_Best_PR_AUC": float(
                ablation_search.best_score_
            ),
            "Best_Parameters": json.dumps(
                ablation_search.best_params_,
                default=str,
                sort_keys=True
            )
        }


        # ----------------------------------------------------
        # Append and write checkpoints
        #
        # Predictions and hyperparameters are written before
        # performance. The performance file is the commit log.
        # ----------------------------------------------------

        ablation_predictions = pd.concat(
            [
                ablation_predictions,
                prediction_records
            ],
            ignore_index=True
        )


        ablation_hyperparameters = pd.concat(
            [
                ablation_hyperparameters,
                pd.DataFrame(
                    [
                        hyperparameter_record
                    ]
                )
            ],
            ignore_index=True
        )


        ablation_predictions.to_csv(
            ABLATION_PREDICTIONS_FILE,
            index=False
        )


        ablation_hyperparameters.to_csv(
            ABLATION_HYPERPARAMETERS_FILE,
            index=False
        )


        ablation_performance = pd.concat(
            [
                ablation_performance,
                pd.DataFrame(
                    [
                        performance_record
                    ]
                )
            ],
            ignore_index=True
        )


        ablation_performance.to_csv(
            ABLATION_PERFORMANCE_FILE,
            index=False
        )


        completed_ablation_evaluations.add(
            evaluation_key
        )


        print(
            "Completed in",
            round(
                elapsed_seconds,
                1
            ),
            "seconds"
        )

        print(
            "Held-out PR-AUC:",
            round(
                held_out_pr_auc,
                4
            )
        )

        print(
            "Held-out ROC-AUC:",
            round(
                held_out_roc_auc,
                4
            )
        )


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert len(
    ablation_performance
) == ABLATION_EXPECTED_EVALUATIONS, (
    "The ablation-performance checkpoint is incomplete."
)


assert not ablation_performance[
    ablation_key_columns
].duplicated().any(), (
    "Duplicate ablation-performance evaluations "
    "were found."
)


assert len(
    ablation_predictions
) == ABLATION_EXPECTED_PREDICTIONS, (
    "The number of ablation predictions is incorrect."
)


prediction_key_columns = [
    "Ablation",
    "Repetition",
    "Outer_Fold",
    "Modeling_Patient_ID"
]


assert not ablation_predictions[
    prediction_key_columns
].duplicated().any(), (
    "Duplicate ablation predictions were found."
)


assert len(
    ablation_hyperparameters
) == ABLATION_EXPECTED_EVALUATIONS, (
    "The ablation-hyperparameter checkpoint "
    "is incomplete."
)


assert not ablation_hyperparameters[
    ablation_key_columns
].duplicated().any(), (
    "Duplicate ablation-hyperparameter records "
    "were found."
)


ablation_elapsed_minutes = (
    time.perf_counter()
    - ablation_start_time
) / 60


print("\nCORE RIDGE ABLATION STUDY COMPLETED")
print("-----------------------------------")

print(
    "Completed evaluations:",
    len(
        ablation_performance
    )
)

print(
    "Held-out predictions:",
    len(
        ablation_predictions
    )
)

print(
    "Elapsed time during this run:",
    round(
        ablation_elapsed_minutes,
        2
    ),
    "minutes"
)

print("\nCheckpoint files:")

print(ABLATION_PERFORMANCE_FILE)
print(ABLATION_PREDICTIONS_FILE)
print(ABLATION_HYPERPARAMETERS_FILE)

In [ ]:
# ============================================================
# Analyze leave-one-predictor-out ablation results
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)


# ------------------------------------------------------------
# Reload and verify completed ablation checkpoints
# ------------------------------------------------------------

ablation_performance = pd.read_csv(
    ABLATION_PERFORMANCE_FILE
)

ablation_predictions = pd.read_csv(
    ABLATION_PREDICTIONS_FILE
)

ablation_hyperparameters = pd.read_csv(
    ABLATION_HYPERPARAMETERS_FILE
)


assert len(
    ablation_performance
) == 450, (
    "Expected 450 completed ablation evaluations."
)

assert len(
    ablation_predictions
) == 123300, (
    "Expected 123,300 ablation predictions."
)

assert len(
    ablation_hyperparameters
) == 450, (
    "Expected 450 hyperparameter records."
)


# ------------------------------------------------------------
# Average ten held-out predictions per patient
# ------------------------------------------------------------

patient_level_ablation_predictions = (
    ablation_predictions
    .groupby(
        [
            "Ablation",
            "Omitted_Predictor",
            "Modeling_Patient_ID"
        ],
        as_index=False
    )
    .agg(
        Observed_Outcome=(
            "Observed_Outcome",
            "first"
        ),
        Mean_Predicted_Probability=(
            "Predicted_Probability",
            "mean"
        ),
        Prediction_Repetitions=(
            "Predicted_Probability",
            "size"
        ),
        Outcome_Values=(
            "Observed_Outcome",
            "nunique"
        )
    )
)


assert (
    patient_level_ablation_predictions[
        "Prediction_Repetitions"
    ] == OUTER_REPEATS
).all(), (
    "At least one ablation patient does not have "
    "ten predictions."
)


assert (
    patient_level_ablation_predictions[
        "Outcome_Values"
    ] == 1
).all(), (
    "Inconsistent outcomes were found."
)


# ------------------------------------------------------------
# Align complete and ablated model predictions
# ------------------------------------------------------------

complete_core_predictions = (
    patient_level_predictions.loc[
        patient_level_predictions[
            "Model"
        ] == "Core_Ridge_Logistic",
        [
            "Modeling_Patient_ID",
            "Observed_Outcome",
            "Mean_Predicted_Probability"
        ]
    ]
    .rename(
        columns={
            "Mean_Predicted_Probability": (
                "Complete_Probability"
            )
        }
    )
    .sort_values(
        "Modeling_Patient_ID"
    )
    .reset_index(drop=True)
)


ablation_probability_table = (
    patient_level_ablation_predictions
    .pivot(
        index="Modeling_Patient_ID",
        columns="Omitted_Predictor",
        values="Mean_Predicted_Probability"
    )
    .reindex(
        columns=core_predictors
    )
    .reset_index()
    .sort_values(
        "Modeling_Patient_ID"
    )
    .reset_index(drop=True)
)


aligned_ablation_data = (
    complete_core_predictions
    .merge(
        ablation_probability_table,
        on="Modeling_Patient_ID",
        how="inner",
        validate="one_to_one"
    )
)


assert len(
    aligned_ablation_data
) == len(
    analysis_data
), (
    "The aligned ablation dataset is incomplete."
)


assert aligned_ablation_data[
    core_predictors
    + [
        "Complete_Probability"
    ]
].notna().all().all(), (
    "Missing aligned probabilities were found."
)


observed_outcomes = (
    aligned_ablation_data[
        "Observed_Outcome"
    ]
    .to_numpy(dtype=int)
)

complete_probabilities = (
    aligned_ablation_data[
        "Complete_Probability"
    ]
    .to_numpy(dtype=float)
)


# ------------------------------------------------------------
# Metric function
# ------------------------------------------------------------

def calculate_ablation_metrics(
    outcomes,
    probabilities
):

    return {
        "ROC_AUC": roc_auc_score(
            outcomes,
            probabilities
        ),

        "PR_AUC": average_precision_score(
            outcomes,
            probabilities
        ),

        "Brier_Score": brier_score_loss(
            outcomes,
            probabilities
        )
    }


complete_metrics = (
    calculate_ablation_metrics(
        observed_outcomes,
        complete_probabilities
    )
)


# ------------------------------------------------------------
# Point-estimate changes after removing predictors
#
# Positive Performance_Loss means the complete model
# performed better than the reduced model.
# ------------------------------------------------------------

ablation_point_records = []


for omitted_predictor in core_predictors:

    reduced_probabilities = (
        aligned_ablation_data[
            omitted_predictor
        ]
        .to_numpy(dtype=float)
    )


    reduced_metrics = (
        calculate_ablation_metrics(
            observed_outcomes,
            reduced_probabilities
        )
    )


    for metric_name in [
        "ROC_AUC",
        "PR_AUC",
        "Brier_Score"
    ]:

        if metric_name in [
            "ROC_AUC",
            "PR_AUC"
        ]:

            performance_loss = (
                complete_metrics[
                    metric_name
                ]
                - reduced_metrics[
                    metric_name
                ]
            )

        else:

            # A larger Brier score is worse.
            performance_loss = (
                reduced_metrics[
                    metric_name
                ]
                - complete_metrics[
                    metric_name
                ]
            )


        ablation_point_records.append({
            "Omitted_Predictor": (
                omitted_predictor
            ),
            "Predictor_Label": (
                predictor_labels.get(
                    omitted_predictor,
                    omitted_predictor
                )
            ),
            "Metric": metric_name,
            "Complete_Model_Estimate": (
                complete_metrics[
                    metric_name
                ]
            ),
            "Reduced_Model_Estimate": (
                reduced_metrics[
                    metric_name
                ]
            ),
            "Performance_Loss_After_Removal": (
                float(
                    performance_loss
                )
            )
        })


ablation_point_table = pd.DataFrame(
    ablation_point_records
)


# ------------------------------------------------------------
# Paired patient-level bootstrap
# ------------------------------------------------------------

NUMBER_OF_ABLATION_BOOTSTRAPS = 2000
ABLATION_BOOTSTRAP_SEED = (
    RANDOM_SEED + 30000
)


ablation_random_generator = (
    np.random.default_rng(
        ABLATION_BOOTSTRAP_SEED
    )
)


number_of_patients = len(
    aligned_ablation_data
)


ablation_bootstrap_records = []


for bootstrap_number in range(
    1,
    NUMBER_OF_ABLATION_BOOTSTRAPS + 1
):

    sampled_indices = (
        ablation_random_generator.choice(
            number_of_patients,
            size=number_of_patients,
            replace=True
        )
    )


    bootstrap_outcomes = (
        observed_outcomes[
            sampled_indices
        ]
    )


    if np.unique(
        bootstrap_outcomes
    ).size < 2:

        continue


    bootstrap_complete_metrics = (
        calculate_ablation_metrics(
            bootstrap_outcomes,
            complete_probabilities[
                sampled_indices
            ]
        )
    )


    for omitted_predictor in core_predictors:

        reduced_probabilities = (
            aligned_ablation_data[
                omitted_predictor
            ]
            .to_numpy(dtype=float)[
                sampled_indices
            ]
        )


        bootstrap_reduced_metrics = (
            calculate_ablation_metrics(
                bootstrap_outcomes,
                reduced_probabilities
            )
        )


        for metric_name in [
            "ROC_AUC",
            "PR_AUC",
            "Brier_Score"
        ]:

            if metric_name in [
                "ROC_AUC",
                "PR_AUC"
            ]:

                performance_loss = (
                    bootstrap_complete_metrics[
                        metric_name
                    ]
                    - bootstrap_reduced_metrics[
                        metric_name
                    ]
                )

            else:

                performance_loss = (
                    bootstrap_reduced_metrics[
                        metric_name
                    ]
                    - bootstrap_complete_metrics[
                        metric_name
                    ]
                )


            ablation_bootstrap_records.append({
                "Bootstrap": bootstrap_number,
                "Omitted_Predictor": (
                    omitted_predictor
                ),
                "Metric": metric_name,
                "Performance_Loss_After_Removal": (
                    float(
                        performance_loss
                    )
                )
            })


    if bootstrap_number % 250 == 0:

        print(
            "Completed ablation bootstrap",
            bootstrap_number,
            "of",
            NUMBER_OF_ABLATION_BOOTSTRAPS
        )


ablation_bootstrap_results = (
    pd.DataFrame(
        ablation_bootstrap_records
    )
)


# ------------------------------------------------------------
# Bootstrap confidence intervals and p-values
# ------------------------------------------------------------

def two_sided_ablation_p_value(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )

    number_of_values = len(
        values
    )


    probability_nonpositive = (
        (
            np.sum(
                values <= 0
            )
            + 1
        )
        / (
            number_of_values
            + 1
        )
    )


    probability_nonnegative = (
        (
            np.sum(
                values >= 0
            )
            + 1
        )
        / (
            number_of_values
            + 1
        )
    )


    return min(
        1.0,
        2
        * min(
            probability_nonpositive,
            probability_nonnegative
        )
    )


ablation_interval_table = (
    ablation_bootstrap_results
    .groupby(
        [
            "Omitted_Predictor",
            "Metric"
        ]
    )[
        "Performance_Loss_After_Removal"
    ]
    .agg(
        CI_Lower=lambda values: np.percentile(
            values,
            2.5
        ),
        CI_Upper=lambda values: np.percentile(
            values,
            97.5
        ),
        Bootstrap_P_Value=(
            two_sided_ablation_p_value
        ),
        Successful_Bootstraps="count"
    )
    .reset_index()
)


ablation_results = (
    ablation_point_table
    .merge(
        ablation_interval_table,
        on=[
            "Omitted_Predictor",
            "Metric"
        ],
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Holm adjustment across all 27 ablation comparisons
# ------------------------------------------------------------

def holm_adjust_ablation_p_values(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    number_of_tests = len(
        p_values
    )

    sorted_indices = np.argsort(
        p_values
    )

    sorted_p_values = (
        p_values[
            sorted_indices
        ]
    )

    adjusted_sorted = np.empty(
        number_of_tests,
        dtype=float
    )

    running_maximum = 0.0


    for rank, p_value in enumerate(
        sorted_p_values
    ):

        adjusted_value = (
            number_of_tests - rank
        ) * p_value

        running_maximum = max(
            running_maximum,
            adjusted_value
        )

        adjusted_sorted[
            rank
        ] = min(
            1.0,
            running_maximum
        )


    adjusted_values = np.empty(
        number_of_tests,
        dtype=float
    )

    adjusted_values[
        sorted_indices
    ] = adjusted_sorted


    return adjusted_values


ablation_results[
    "Holm_Adjusted_P_Value"
] = (
    holm_adjust_ablation_p_values(
        ablation_results[
            "Bootstrap_P_Value"
        ].to_numpy(dtype=float)
    )
)


ablation_results[
    "Statistically_Significant"
] = (
    ablation_results[
        "Holm_Adjusted_P_Value"
    ] < 0.05
)


ablation_results[
    "Loss_95_CI"
] = (
    ablation_results[
        "Performance_Loss_After_Removal"
    ].map(
        lambda value: f"{value:.4f}"
    )
    + " ("
    + ablation_results[
        "CI_Lower"
    ].map(
        lambda value: f"{value:.4f}"
    )
    + " to "
    + ablation_results[
        "CI_Upper"
    ].map(
        lambda value: f"{value:.4f}"
    )
    + ")"
)


# ------------------------------------------------------------
# Order results
# ------------------------------------------------------------

ablation_metric_order = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score"
]


ablation_results[
    "Metric"
] = pd.Categorical(
    ablation_results["Metric"],
    categories=ablation_metric_order,
    ordered=True
)


ablation_results[
    "Omitted_Predictor"
] = pd.Categorical(
    ablation_results[
        "Omitted_Predictor"
    ],
    categories=core_predictors,
    ordered=True
)


ablation_results = (
    ablation_results
    .sort_values(
        [
            "Metric",
            "Omitted_Predictor"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Display ablation results
# ------------------------------------------------------------

ablation_display_columns = [
    "Metric",
    "Predictor_Label",
    "Complete_Model_Estimate",
    "Reduced_Model_Estimate",
    "Loss_95_CI",
    "Bootstrap_P_Value",
    "Holm_Adjusted_P_Value",
    "Statistically_Significant"
]


print("LEAVE-ONE-PREDICTOR-OUT ABLATION RESULTS")
print("----------------------------------------")

display(
    ablation_results[
        ablation_display_columns
    ].round(
        {
            "Complete_Model_Estimate": 4,
            "Reduced_Model_Estimate": 4,
            "Bootstrap_P_Value": 4,
            "Holm_Adjusted_P_Value": 4
        }
    )
)


# ------------------------------------------------------------
# Create enclosed three-panel ablation figure
# ------------------------------------------------------------

ablation_figure, ablation_axes = (
    plt.subplots(
        nrows=1,
        ncols=3,
        figsize=(19, 7.5),
        facecolor="white"
    )
)


metric_titles = {
    "ROC_AUC": "A. ROC-AUC loss",
    "PR_AUC": "B. PR-AUC loss",
    "Brier_Score": "C. Brier-score deterioration"
}


for axis, metric_name in zip(
    ablation_axes,
    ablation_metric_order
):

    metric_data = (
        ablation_results.loc[
            ablation_results[
                "Metric"
            ].astype(str) == metric_name
        ]
        .sort_values(
            "Performance_Loss_After_Removal",
            ascending=True
        )
        .copy()
    )


    point_values = (
        metric_data[
            "Performance_Loss_After_Removal"
        ]
        .to_numpy(dtype=float)
    )

    lower_errors = (
        point_values
        - metric_data[
            "CI_Lower"
        ].to_numpy(dtype=float)
    )

    upper_errors = (
        metric_data[
            "CI_Upper"
        ].to_numpy(dtype=float)
        - point_values
    )


    bar_colors = np.where(
        metric_data[
            "Statistically_Significant"
        ].to_numpy(dtype=bool),
        "#D55E00",
        "#0072B2"
    )


    axis.barh(
        y=np.arange(
            len(metric_data)
        ),
        width=point_values,
        color=bar_colors,
        edgecolor="black",
        linewidth=0.8,
        xerr=np.vstack(
            [
                np.maximum(
                    lower_errors,
                    0
                ),
                np.maximum(
                    upper_errors,
                    0
                )
            ]
        ),
        error_kw={
            "ecolor": "black",
            "elinewidth": 1.0,
            "capsize": 3,
            "capthick": 1.0
        }
    )


    axis.set_yticks(
        np.arange(
            len(metric_data)
        )
    )

    axis.set_yticklabels(
        metric_data[
            "Predictor_Label"
        ].tolist(),
        fontsize=9.5
    )


    axis.axvline(
        x=0,
        color="black",
        linewidth=1.0
    )


    axis.set_title(
        metric_titles[
            metric_name
        ],
        fontsize=13,
        fontweight="bold",
        color="black",
        pad=12
    )

    axis.set_xlabel(
        "Performance loss after predictor removal",
        fontsize=10.5,
        color="black",
        labelpad=8
    )

    axis.set_facecolor(
        "white"
    )

    axis.grid(False)


    for spine_name in [
        "top",
        "right",
        "bottom",
        "left"
    ]:

        axis.spines[
            spine_name
        ].set_visible(True)

        axis.spines[
            spine_name
        ].set_color("black")

        axis.spines[
            spine_name
        ].set_linewidth(1.0)


    axis.tick_params(
        axis="both",
        colors="black",
        direction="out"
    )


ablation_figure.suptitle(
    "Leave-one-predictor-out ablation of the "
    "core ridge model",
    fontsize=16,
    fontweight="bold",
    color="black",
    y=0.99
)


ablation_figure.text(
    0.5,
    0.015,
    (
        "Positive values indicate deterioration after "
        "predictor removal. Orange denotes Holm-adjusted "
        "p < 0.05."
    ),
    ha="center",
    fontsize=10,
    color="black"
)


ablation_figure.tight_layout(
    rect=[0.02, 0.055, 0.98, 0.95],
    w_pad=3.0
)


# ------------------------------------------------------------
# Save ablation outputs
# ------------------------------------------------------------

ABLATION_RESULTS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_core_ridge_ablation_results.xlsx"
)

ABLATION_BOOTSTRAP_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_core_ridge_ablation_bootstrap.csv"
)

ABLATION_FIGURE_PNG = (
    FIGURES_DIRECTORY
    / "unweighted_core_ridge_ablation.png"
)

ABLATION_FIGURE_PDF = (
    FIGURES_DIRECTORY
    / "unweighted_core_ridge_ablation.pdf"
)


ablation_bootstrap_results.to_csv(
    ABLATION_BOOTSTRAP_FILE,
    index=False
)


with pd.ExcelWriter(
    ABLATION_RESULTS_FILE,
    engine="openpyxl"
) as writer:

    ablation_results.to_excel(
        writer,
        sheet_name="Ablation Comparisons",
        index=False
    )

    patient_level_ablation_predictions.to_excel(
        writer,
        sheet_name="Patient Predictions",
        index=False
    )

    ablation_performance.to_excel(
        writer,
        sheet_name="Fold Performance",
        index=False
    )


ablation_figure.savefig(
    ABLATION_FIGURE_PNG,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)

ablation_figure.savefig(
    ABLATION_FIGURE_PDF,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)


plt.show()


print("\nABLATION RESULTS SAVED")
print("----------------------")
print(ABLATION_RESULTS_FILE)
print(ABLATION_BOOTSTRAP_FILE)
print(ABLATION_FIGURE_PNG)
print(ABLATION_FIGURE_PDF)

In [ ]:
# ============================================================
# Sensitivity analysis: unweighted versus mixed class weighting
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)


# ------------------------------------------------------------
# Locate original mixed-weight checkpoint files
# ------------------------------------------------------------

MIXED_PERFORMANCE_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "nested_fold_performance.csv"
)

MIXED_PREDICTIONS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "nested_held_out_predictions.csv"
)

MIXED_HYPERPARAMETERS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "nested_selected_hyperparameters.csv"
)


for required_file in [
    MIXED_PERFORMANCE_FILE,
    MIXED_PREDICTIONS_FILE,
    MIXED_HYPERPARAMETERS_FILE
]:

    if not required_file.exists():

        raise FileNotFoundError(
            "Required mixed-weight checkpoint "
            f"not found:\n{required_file}"
        )


# ------------------------------------------------------------
# Load and verify the mixed-weight analysis
# ------------------------------------------------------------

mixed_fold_performance = pd.read_csv(
    MIXED_PERFORMANCE_FILE
)

mixed_predictions = pd.read_csv(
    MIXED_PREDICTIONS_FILE
)

mixed_hyperparameters = pd.read_csv(
    MIXED_HYPERPARAMETERS_FILE
)


assert len(
    mixed_fold_performance
) == 200, (
    "Expected 200 mixed-weight evaluations."
)

assert len(
    mixed_predictions
) == 54800, (
    "Expected 54,800 mixed-weight predictions."
)

assert len(
    mixed_hyperparameters
) == 200, (
    "Expected 200 mixed-weight hyperparameter records."
)


# ------------------------------------------------------------
# Average ten held-out predictions per patient and model
# ------------------------------------------------------------

mixed_patient_predictions = (
    mixed_predictions
    .groupby(
        [
            "Model",
            "Modeling_Patient_ID"
        ],
        as_index=False
    )
    .agg(
        Observed_Outcome=(
            "Observed_Outcome",
            "first"
        ),
        Mean_Predicted_Probability=(
            "Predicted_Probability",
            "mean"
        ),
        Prediction_Repetitions=(
            "Predicted_Probability",
            "size"
        ),
        Outcome_Values=(
            "Observed_Outcome",
            "nunique"
        )
    )
)


assert (
    mixed_patient_predictions[
        "Prediction_Repetitions"
    ] == OUTER_REPEATS
).all(), (
    "Incomplete mixed-weight patient predictions."
)


assert (
    mixed_patient_predictions[
        "Outcome_Values"
    ] == 1
).all(), (
    "Inconsistent outcomes were found."
)


# ------------------------------------------------------------
# Metric function
# ------------------------------------------------------------

def calculate_weight_sensitivity_metrics(
    outcomes,
    probabilities
):

    calibration_intercept, (
        calibration_slope
    ) = calculate_calibration_parameters(
        outcomes,
        probabilities
    )


    return {
        "ROC_AUC": roc_auc_score(
            outcomes,
            probabilities
        ),

        "PR_AUC": average_precision_score(
            outcomes,
            probabilities
        ),

        "Brier_Score": brier_score_loss(
            outcomes,
            probabilities
        ),

        "Calibration_Intercept": (
            calibration_intercept
        ),

        "Calibration_Slope": (
            calibration_slope
        ),

        "Mean_Predicted_Risk": float(
            np.mean(
                probabilities
            )
        ),

        "Observed_Prevalence": float(
            np.mean(
                outcomes
            )
        )
    }


# ------------------------------------------------------------
# Align unweighted and mixed predictions
# ------------------------------------------------------------

aligned_strategy_records = []
strategy_point_records = []


for model_name in model_order:

    unweighted_model_data = (
        patient_level_predictions.loc[
            patient_level_predictions[
                "Model"
            ] == model_name,
            [
                "Modeling_Patient_ID",
                "Observed_Outcome",
                "Mean_Predicted_Probability"
            ]
        ]
        .rename(
            columns={
                "Mean_Predicted_Probability": (
                    "Unweighted_Probability"
                )
            }
        )
    )


    mixed_model_data = (
        mixed_patient_predictions.loc[
            mixed_patient_predictions[
                "Model"
            ] == model_name,
            [
                "Modeling_Patient_ID",
                "Observed_Outcome",
                "Mean_Predicted_Probability"
            ]
        ]
        .rename(
            columns={
                "Observed_Outcome": (
                    "Mixed_Observed_Outcome"
                ),
                "Mean_Predicted_Probability": (
                    "Mixed_Probability"
                )
            }
        )
    )


    aligned_model_data = (
        unweighted_model_data
        .merge(
            mixed_model_data,
            on="Modeling_Patient_ID",
            how="inner",
            validate="one_to_one"
        )
        .sort_values(
            "Modeling_Patient_ID"
        )
        .reset_index(drop=True)
    )


    assert len(
        aligned_model_data
    ) == len(
        analysis_data
    ), (
        f"Incomplete alignment for {model_name}."
    )


    assert np.array_equal(
        aligned_model_data[
            "Observed_Outcome"
        ].to_numpy(dtype=int),
        aligned_model_data[
            "Mixed_Observed_Outcome"
        ].to_numpy(dtype=int)
    ), (
        f"Outcome mismatch for {model_name}."
    )


    aligned_model_data[
        "Model"
    ] = model_name


    aligned_strategy_records.append(
        aligned_model_data
    )


    outcomes = (
        aligned_model_data[
            "Observed_Outcome"
        ]
        .to_numpy(dtype=int)
    )


    for strategy_name, probability_column in [
        (
            "Primary unweighted",
            "Unweighted_Probability"
        ),
        (
            "Mixed class-weight tuning",
            "Mixed_Probability"
        )
    ]:

        probabilities = (
            aligned_model_data[
                probability_column
            ]
            .to_numpy(dtype=float)
        )


        strategy_metrics = (
            calculate_weight_sensitivity_metrics(
                outcomes,
                probabilities
            )
        )


        for metric_name, estimate in (
            strategy_metrics.items()
        ):

            strategy_point_records.append({
                "Model": model_name,
                "Strategy": strategy_name,
                "Metric": metric_name,
                "Estimate": float(
                    estimate
                )
            })


aligned_weight_sensitivity_data = (
    pd.concat(
        aligned_strategy_records,
        ignore_index=True
    )
)


weight_sensitivity_point_estimates = (
    pd.DataFrame(
        strategy_point_records
    )
)


# ------------------------------------------------------------
# Paired patient-level bootstrap
# ------------------------------------------------------------

NUMBER_OF_WEIGHT_BOOTSTRAPS = 2000
WEIGHT_BOOTSTRAP_SEED = (
    RANDOM_SEED + 40000
)


weight_random_generator = (
    np.random.default_rng(
        WEIGHT_BOOTSTRAP_SEED
    )
)


weight_bootstrap_records = []
weight_difference_records = []


for model_name in model_order:

    model_data = (
        aligned_weight_sensitivity_data.loc[
            aligned_weight_sensitivity_data[
                "Model"
            ] == model_name
        ]
        .sort_values(
            "Modeling_Patient_ID"
        )
        .reset_index(drop=True)
    )


    outcomes = (
        model_data[
            "Observed_Outcome"
        ]
        .to_numpy(dtype=int)
    )

    unweighted_probabilities = (
        model_data[
            "Unweighted_Probability"
        ]
        .to_numpy(dtype=float)
    )

    mixed_probabilities = (
        model_data[
            "Mixed_Probability"
        ]
        .to_numpy(dtype=float)
    )


    number_of_patients = len(
        model_data
    )


    for bootstrap_number in range(
        1,
        NUMBER_OF_WEIGHT_BOOTSTRAPS + 1
    ):

        sampled_indices = (
            weight_random_generator.choice(
                number_of_patients,
                size=number_of_patients,
                replace=True
            )
        )


        bootstrap_outcomes = (
            outcomes[
                sampled_indices
            ]
        )


        if np.unique(
            bootstrap_outcomes
        ).size < 2:

            continue


        unweighted_metrics = (
            calculate_weight_sensitivity_metrics(
                bootstrap_outcomes,
                unweighted_probabilities[
                    sampled_indices
                ]
            )
        )

        mixed_metrics = (
            calculate_weight_sensitivity_metrics(
                bootstrap_outcomes,
                mixed_probabilities[
                    sampled_indices
                ]
            )
        )


        for strategy_name, metrics in [
            (
                "Primary unweighted",
                unweighted_metrics
            ),
            (
                "Mixed class-weight tuning",
                mixed_metrics
            )
        ]:

            for metric_name, estimate in (
                metrics.items()
            ):

                weight_bootstrap_records.append({
                    "Model": model_name,
                    "Strategy": strategy_name,
                    "Bootstrap": bootstrap_number,
                    "Metric": metric_name,
                    "Estimate": float(
                        estimate
                    )
                })


        # Positive values consistently favour unweighted.

        for metric_name in [
            "ROC_AUC",
            "PR_AUC"
        ]:

            improvement = (
                unweighted_metrics[
                    metric_name
                ]
                - mixed_metrics[
                    metric_name
                ]
            )


            weight_difference_records.append({
                "Model": model_name,
                "Bootstrap": bootstrap_number,
                "Metric": metric_name,
                "Improvement_Favouring_Unweighted": (
                    float(
                        improvement
                    )
                )
            })


        brier_improvement = (
            mixed_metrics[
                "Brier_Score"
            ]
            - unweighted_metrics[
                "Brier_Score"
            ]
        )


        weight_difference_records.append({
            "Model": model_name,
            "Bootstrap": bootstrap_number,
            "Metric": "Brier_Score",
            "Improvement_Favouring_Unweighted": (
                float(
                    brier_improvement
                )
            )
        })


    print(
        model_name,
        "| completed weight-sensitivity bootstraps:",
        NUMBER_OF_WEIGHT_BOOTSTRAPS
    )


weight_bootstrap_estimates = (
    pd.DataFrame(
        weight_bootstrap_records
    )
)

weight_bootstrap_differences = (
    pd.DataFrame(
        weight_difference_records
    )
)


# ------------------------------------------------------------
# Confidence intervals for each strategy
# ------------------------------------------------------------

weight_strategy_intervals = (
    weight_bootstrap_estimates
    .groupby(
        [
            "Model",
            "Strategy",
            "Metric"
        ]
    )["Estimate"]
    .agg(
        CI_Lower=lambda values: np.percentile(
            values,
            2.5
        ),
        CI_Upper=lambda values: np.percentile(
            values,
            97.5
        ),
        Successful_Bootstraps="count"
    )
    .reset_index()
)


weight_sensitivity_summary = (
    weight_sensitivity_point_estimates
    .merge(
        weight_strategy_intervals,
        on=[
            "Model",
            "Strategy",
            "Metric"
        ],
        how="left",
        validate="one_to_one"
    )
)


weight_sensitivity_summary[
    "Estimate_95_CI"
] = (
    weight_sensitivity_summary[
        "Estimate"
    ].map(
        lambda value: f"{value:.3f}"
    )
    + " ("
    + weight_sensitivity_summary[
        "CI_Lower"
    ].map(
        lambda value: f"{value:.3f}"
    )
    + " to "
    + weight_sensitivity_summary[
        "CI_Upper"
    ].map(
        lambda value: f"{value:.3f}"
    )
    + ")"
)


# ------------------------------------------------------------
# Paired difference intervals
# ------------------------------------------------------------

weight_difference_summary = (
    weight_bootstrap_differences
    .groupby(
        [
            "Model",
            "Metric"
        ]
    )[
        "Improvement_Favouring_Unweighted"
    ]
    .agg(
        Median_Improvement="median",
        CI_Lower=lambda values: np.percentile(
            values,
            2.5
        ),
        CI_Upper=lambda values: np.percentile(
            values,
            97.5
        ),
        Probability_Unweighted_Better=(
            lambda values: np.mean(
                values > 0
            )
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Create manuscript comparison table
# ------------------------------------------------------------

weight_manuscript_table = (
    weight_sensitivity_summary
    .pivot(
        index=[
            "Model",
            "Metric"
        ],
        columns="Strategy",
        values="Estimate_95_CI"
    )
    .reset_index()
)


# ------------------------------------------------------------
# Plot Brier score and predicted-risk inflation
# ------------------------------------------------------------

weight_figure, weight_axes = (
    plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(15, 6.5),
        facecolor="white"
    )
)


strategy_colors = {
    "Primary unweighted": "#0072B2",
    "Mixed class-weight tuning": "#D55E00"
}


plot_metrics = [
    (
        "Brier_Score",
        "A. Brier score",
        "Brier score"
    ),
    (
        "Mean_Predicted_Risk",
        "B. Mean predicted recurrence risk",
        "Mean predicted probability"
    )
]


x_positions = np.arange(
    len(
        model_order
    )
)

bar_width = 0.36


for axis, (
    metric_name,
    panel_title,
    y_label
) in zip(
    weight_axes,
    plot_metrics
):

    for strategy_number, strategy_name in enumerate(
        [
            "Primary unweighted",
            "Mixed class-weight tuning"
        ]
    ):

        plot_data = (
            weight_sensitivity_summary.loc[
                (
                    weight_sensitivity_summary[
                        "Metric"
                    ] == metric_name
                )
                & (
                    weight_sensitivity_summary[
                        "Strategy"
                    ] == strategy_name
                )
            ]
            .set_index("Model")
            .reindex(model_order)
        )


        estimates = (
            plot_data[
                "Estimate"
            ]
            .to_numpy(dtype=float)
        )

        lower_errors = (
            estimates
            - plot_data[
                "CI_Lower"
            ].to_numpy(dtype=float)
        )

        upper_errors = (
            plot_data[
                "CI_Upper"
            ].to_numpy(dtype=float)
            - estimates
        )


        horizontal_shift = (
            strategy_number - 0.5
        ) * bar_width


        axis.bar(
            x_positions
            + horizontal_shift,
            estimates,
            width=bar_width,
            color=strategy_colors[
                strategy_name
            ],
            edgecolor="black",
            linewidth=0.8,
            label=strategy_name,
            yerr=np.vstack(
                [
                    np.maximum(
                        lower_errors,
                        0
                    ),
                    np.maximum(
                        upper_errors,
                        0
                    )
                ]
            ),
            error_kw={
                "ecolor": "black",
                "elinewidth": 1.0,
                "capsize": 3
            }
        )


    if metric_name == "Mean_Predicted_Risk":

        axis.axhline(
            y=overall_event_prevalence,
            color="black",
            linestyle="--",
            linewidth=1.3,
            label=(
                "Observed prevalence "
                f"({overall_event_prevalence:.3f})"
            )
        )


    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels(
        [
            "Core ridge",
            "Elastic net",
            "Random forest",
            "Gradient boosting"
        ],
        rotation=18,
        ha="right"
    )

    axis.set_title(
        panel_title,
        fontsize=13,
        fontweight="bold",
        color="black",
        pad=12
    )

    axis.set_ylabel(
        y_label,
        fontsize=11,
        color="black"
    )

    axis.set_facecolor(
        "white"
    )

    axis.grid(False)


    for spine_name in [
        "top",
        "right",
        "bottom",
        "left"
    ]:

        axis.spines[
            spine_name
        ].set_visible(True)

        axis.spines[
            spine_name
        ].set_color("black")

        axis.spines[
            spine_name
        ].set_linewidth(1.0)


    axis.tick_params(
        axis="both",
        colors="black",
        direction="out"
    )


    legend = axis.legend(
        loc="upper right",
        frameon=True,
        fancybox=False,
        shadow=False,
        fontsize=9,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0
    )

    legend.get_frame().set_linewidth(
        0.9
    )


weight_figure.suptitle(
    "Sensitivity to mixed class-weight tuning",
    fontsize=16,
    fontweight="bold",
    color="black",
    y=0.99
)


weight_figure.tight_layout(
    rect=[
        0.02,
        0.03,
        0.98,
        0.95
    ],
    w_pad=3.0
)


# ------------------------------------------------------------
# Save sensitivity outputs
# ------------------------------------------------------------

WEIGHT_SENSITIVITY_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "class_weight_sensitivity_analysis.xlsx"
)

WEIGHT_SENSITIVITY_BOOTSTRAP_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "class_weight_sensitivity_bootstrap.csv"
)

WEIGHT_SENSITIVITY_FIGURE_PNG = (
    FIGURES_DIRECTORY
    / "class_weight_sensitivity_analysis.png"
)

WEIGHT_SENSITIVITY_FIGURE_PDF = (
    FIGURES_DIRECTORY
    / "class_weight_sensitivity_analysis.pdf"
)


weight_bootstrap_estimates.to_csv(
    WEIGHT_SENSITIVITY_BOOTSTRAP_FILE,
    index=False
)


with pd.ExcelWriter(
    WEIGHT_SENSITIVITY_FILE,
    engine="openpyxl"
) as writer:

    weight_sensitivity_summary.to_excel(
        writer,
        sheet_name="Strategy Performance",
        index=False
    )

    weight_manuscript_table.to_excel(
        writer,
        sheet_name="Manuscript Table",
        index=False
    )

    weight_difference_summary.to_excel(
        writer,
        sheet_name="Paired Differences",
        index=False
    )


weight_figure.savefig(
    WEIGHT_SENSITIVITY_FIGURE_PNG,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)

weight_figure.savefig(
    WEIGHT_SENSITIVITY_FIGURE_PDF,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="white",
    transparent=False
)


plt.show()


# ------------------------------------------------------------
# Display key results
# ------------------------------------------------------------

key_weight_metrics = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Calibration_Intercept",
    "Calibration_Slope",
    "Mean_Predicted_Risk"
]


print("CLASS-WEIGHT SENSITIVITY ANALYSIS")
print("---------------------------------")

display(
    weight_manuscript_table.loc[
        weight_manuscript_table[
            "Metric"
        ].isin(
            key_weight_metrics
        )
    ]
)


print("\nPAIRED IMPROVEMENT FAVOURING UNWEIGHTED")
print("---------------------------------------")

display(
    weight_difference_summary.round(4)
)


print("\nFiles saved:")
print(WEIGHT_SENSITIVITY_FILE)
print(WEIGHT_SENSITIVITY_BOOTSTRAP_FILE)
print(WEIGHT_SENSITIVITY_FIGURE_PNG)
print(WEIGHT_SENSITIVITY_FIGURE_PDF)

In [ ]:
# ============================================================
# Final sensitivity analyses for the core ridge model
# ============================================================

import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# Checkpoint files
# ------------------------------------------------------------

FINAL_SENSITIVITY_PERFORMANCE_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "core_ridge_final_sensitivity_performance.csv"
)

FINAL_SENSITIVITY_PREDICTIONS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "core_ridge_final_sensitivity_predictions.csv"
)

FINAL_SENSITIVITY_HYPERPARAMETERS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "core_ridge_final_sensitivity_hyperparameters.csv"
)


# ------------------------------------------------------------
# Suppress harmless encoder warnings
# ------------------------------------------------------------

warnings.filterwarnings(
    "ignore",
    message=(
        "Found unknown categories in columns .* "
        "during transform.*"
    ),
    category=UserWarning,
    module="sklearn.preprocessing._encoders"
)


# ------------------------------------------------------------
# Define sensitivity scenarios
# ------------------------------------------------------------

temporally_unavailable_predictors = [
    "Tumor_Grade",
    "ER_Status",
    "HER2_Status"
]


reduced_availability_predictors = [
    predictor
    for predictor in core_predictors
    if predictor
    not in temporally_unavailable_predictors
]


non_distant_mask = (
    X["Summary_Stage"]
    .astype(str)
    .str.strip()
    .str.lower()
    != "distant"
)


sensitivity_scenarios = {
    "Exclude_Distant_Stage": {
        "X": (
            X.loc[
                non_distant_mask
            ]
            .reset_index(drop=True)
        ),
        "y": (
            y.loc[
                non_distant_mask
            ]
            .reset_index(drop=True)
        ),
        "patient_id": (
            patient_id.loc[
                non_distant_mask
            ]
            .reset_index(drop=True)
        ),
        "predictors": (
            core_predictors.copy()
        ),
        "Description": (
            "Core ridge model after excluding "
            "patients with distant disease"
        )
    },

    "Reduced_Availability_Predictors": {
        "X": X.reset_index(drop=True),
        "y": y.reset_index(drop=True),
        "patient_id": (
            patient_id.reset_index(
                drop=True
            )
        ),
        "predictors": (
            reduced_availability_predictors
        ),
        "Description": (
            "Core ridge model without tumour grade, "
            "ER status and HER2 status"
        )
    }
}


assert len(
    sensitivity_scenarios
) == 2, (
    "Expected exactly two sensitivity scenarios."
)


assert len(
    reduced_availability_predictors
) == 6, (
    "The reduced-availability model should contain "
    "six predictors."
)


assert int(
    non_distant_mask.sum()
) < len(
    analysis_data
), (
    "No distant-stage patients were excluded."
)


# ------------------------------------------------------------
# Display scenario definitions
# ------------------------------------------------------------

print("FINAL CORE RIDGE SENSITIVITY SCENARIOS")
print("--------------------------------------")


for scenario_name, scenario in (
    sensitivity_scenarios.items()
):

    print(f"\n{scenario_name}")

    print(
        "  Description:",
        scenario["Description"]
    )

    print(
        "  Patients:",
        len(
            scenario["y"]
        )
    )

    print(
        "  Recurrences:",
        int(
            scenario["y"].sum()
        )
    )

    print(
        "  Prevalence:",
        round(
            float(
                scenario["y"].mean()
            ),
            4
        )
    )

    print(
        "  Predictors:",
        len(
            scenario["predictors"]
        )
    )

    print(
        "  Predictor names:",
        scenario["predictors"]
    )


# ------------------------------------------------------------
# Define hyperparameter space
# ------------------------------------------------------------

sensitivity_parameter_space = {
    "model__C": np.logspace(
        -3,
        3,
        9
    ).tolist()
}


# ------------------------------------------------------------
# Load existing checkpoints
# ------------------------------------------------------------

def load_sensitivity_checkpoint(
    file_path
):

    if file_path.exists():

        return pd.read_csv(
            file_path
        )

    return pd.DataFrame()


sensitivity_performance = (
    load_sensitivity_checkpoint(
        FINAL_SENSITIVITY_PERFORMANCE_FILE
    )
)

sensitivity_predictions = (
    load_sensitivity_checkpoint(
        FINAL_SENSITIVITY_PREDICTIONS_FILE
    )
)

sensitivity_hyperparameters = (
    load_sensitivity_checkpoint(
        FINAL_SENSITIVITY_HYPERPARAMETERS_FILE
    )
)


sensitivity_key_columns = [
    "Scenario",
    "Repetition",
    "Outer_Fold"
]


completed_sensitivity_evaluations = set()


if not sensitivity_performance.empty:

    completed_sensitivity_evaluations = set(
        map(
            tuple,
            sensitivity_performance[
                sensitivity_key_columns
            ].to_numpy()
        )
    )


EXPECTED_SENSITIVITY_EVALUATIONS = (
    len(
        sensitivity_scenarios
    )
    * OUTER_FOLDS
    * OUTER_REPEATS
)


EXPECTED_SENSITIVITY_PREDICTIONS = (
    sum(
        len(
            scenario["y"]
        )
        for scenario
        in sensitivity_scenarios.values()
    )
    * OUTER_REPEATS
)


print(
    "\nPreviously completed evaluations:",
    len(
        completed_sensitivity_evaluations
    ),
    "of",
    EXPECTED_SENSITIVITY_EVALUATIONS
)


# ------------------------------------------------------------
# Run both sensitivity scenarios
# ------------------------------------------------------------

sensitivity_start_time = (
    time.perf_counter()
)


for scenario_number, (
    scenario_name,
    scenario
) in enumerate(
    sensitivity_scenarios.items(),
    start=1
):

    scenario_X = scenario["X"]
    scenario_y = scenario["y"]

    scenario_patient_id = (
        scenario["patient_id"]
    )

    selected_predictors = (
        scenario["predictors"]
    )


    scenario_preprocessor = (
        create_preprocessor(
            selected_predictors=(
                selected_predictors
            ),
            drop_reference_category=True
        )
    )


    scenario_outer_cross_validation = (
        RepeatedStratifiedKFold(
            n_splits=OUTER_FOLDS,
            n_repeats=OUTER_REPEATS,
            random_state=RANDOM_SEED
        )
    )


    for split_number, (
        outer_training_indices,
        outer_validation_indices
    ) in enumerate(
        scenario_outer_cross_validation.split(
            scenario_X[
                selected_predictors
            ],
            scenario_y
        ),
        start=1
    ):

        repetition_number = (
            (split_number - 1)
            // OUTER_FOLDS
        ) + 1

        outer_fold_number = (
            (split_number - 1)
            % OUTER_FOLDS
        ) + 1


        evaluation_key = (
            scenario_name,
            repetition_number,
            outer_fold_number
        )


        if (
            evaluation_key
            in completed_sensitivity_evaluations
        ):

            continue


        completed_before_current = len(
            completed_sensitivity_evaluations
        )


        print(
            "\nSensitivity evaluation "
            f"{completed_before_current + 1}/"
            f"{EXPECTED_SENSITIVITY_EVALUATIONS}"
        )

        print(
            scenario_name,
            "| repetition",
            repetition_number,
            "| outer fold",
            outer_fold_number
        )


        evaluation_seed = (
            RANDOM_SEED
            + repetition_number * 1000
            + outer_fold_number * 100
            + 1
        )


        inner_cross_validation = (
            StratifiedKFold(
                n_splits=INNER_FOLDS,
                shuffle=True,
                random_state=evaluation_seed
            )
        )


        scenario_pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    scenario_preprocessor
                ),
                (
                    "model",
                    LogisticRegression(
                        penalty="l2",
                        solver="liblinear",
                        class_weight=None,
                        max_iter=5000,
                        random_state=(
                            evaluation_seed
                        )
                    )
                )
            ]
        )


        X_outer_training = (
            scenario_X.iloc[
                outer_training_indices
            ][
                selected_predictors
            ]
            .copy()
        )

        y_outer_training = (
            scenario_y.iloc[
                outer_training_indices
            ]
            .copy()
        )

        X_outer_validation = (
            scenario_X.iloc[
                outer_validation_indices
            ][
                selected_predictors
            ]
            .copy()
        )

        y_outer_validation = (
            scenario_y.iloc[
                outer_validation_indices
            ]
            .copy()
        )


        # Remove any incomplete orphan records.

        if not sensitivity_predictions.empty:

            orphan_mask = (
                (
                    sensitivity_predictions[
                        "Scenario"
                    ] == scenario_name
                )
                & (
                    sensitivity_predictions[
                        "Repetition"
                    ] == repetition_number
                )
                & (
                    sensitivity_predictions[
                        "Outer_Fold"
                    ] == outer_fold_number
                )
            )

            sensitivity_predictions = (
                sensitivity_predictions.loc[
                    ~orphan_mask
                ]
                .reset_index(drop=True)
            )


        if not sensitivity_hyperparameters.empty:

            orphan_mask = (
                (
                    sensitivity_hyperparameters[
                        "Scenario"
                    ] == scenario_name
                )
                & (
                    sensitivity_hyperparameters[
                        "Repetition"
                    ] == repetition_number
                )
                & (
                    sensitivity_hyperparameters[
                        "Outer_Fold"
                    ] == outer_fold_number
                )
            )

            sensitivity_hyperparameters = (
                sensitivity_hyperparameters.loc[
                    ~orphan_mask
                ]
                .reset_index(drop=True)
            )


        scenario_search = GridSearchCV(
            estimator=scenario_pipeline,
            param_grid=(
                sensitivity_parameter_space
            ),
            scoring="average_precision",
            cv=inner_cross_validation,
            refit=True,
            n_jobs=1,
            return_train_score=False,
            error_score="raise"
        )


        evaluation_start_time = (
            time.perf_counter()
        )


        scenario_search.fit(
            X_outer_training,
            y_outer_training
        )


        outer_probabilities = (
            scenario_search
            .best_estimator_
            .predict_proba(
                X_outer_validation
            )[:, 1]
        )


        elapsed_seconds = (
            time.perf_counter()
            - evaluation_start_time
        )


        held_out_pr_auc = (
            average_precision_score(
                y_outer_validation,
                outer_probabilities
            )
        )

        held_out_roc_auc = (
            roc_auc_score(
                y_outer_validation,
                outer_probabilities
            )
        )

        held_out_brier_score = (
            brier_score_loss(
                y_outer_validation,
                outer_probabilities
            )
        )


        performance_record = {
            "Scenario": scenario_name,
            "Description": (
                scenario[
                    "Description"
                ]
            ),
            "Repetition": (
                repetition_number
            ),
            "Outer_Fold": (
                outer_fold_number
            ),
            "Patients": int(
                len(
                    scenario_y
                )
            ),
            "Events": int(
                scenario_y.sum()
            ),
            "Predictor_Count": int(
                len(
                    selected_predictors
                )
            ),
            "Inner_Best_PR_AUC": float(
                scenario_search.best_score_
            ),
            "Held_Out_PR_AUC": float(
                held_out_pr_auc
            ),
            "Held_Out_ROC_AUC": float(
                held_out_roc_auc
            ),
            "Held_Out_Brier_Score": float(
                held_out_brier_score
            ),
            "Elapsed_Seconds": float(
                elapsed_seconds
            )
        }


        prediction_records = pd.DataFrame({
            "Scenario": scenario_name,
            "Description": (
                scenario[
                    "Description"
                ]
            ),
            "Repetition": (
                repetition_number
            ),
            "Outer_Fold": (
                outer_fold_number
            ),
            "Modeling_Patient_ID": (
                scenario_patient_id.iloc[
                    outer_validation_indices
                ].to_numpy()
            ),
            "Observed_Outcome": (
                y_outer_validation.to_numpy()
            ),
            "Predicted_Probability": (
                outer_probabilities
            )
        })


        hyperparameter_record = {
            "Scenario": scenario_name,
            "Repetition": (
                repetition_number
            ),
            "Outer_Fold": (
                outer_fold_number
            ),
            "Inner_Best_PR_AUC": float(
                scenario_search.best_score_
            ),
            "Best_Parameters": json.dumps(
                scenario_search.best_params_,
                default=str,
                sort_keys=True
            )
        }


        sensitivity_predictions = pd.concat(
            [
                sensitivity_predictions,
                prediction_records
            ],
            ignore_index=True
        )


        sensitivity_hyperparameters = pd.concat(
            [
                sensitivity_hyperparameters,
                pd.DataFrame(
                    [
                        hyperparameter_record
                    ]
                )
            ],
            ignore_index=True
        )


        sensitivity_predictions.to_csv(
            FINAL_SENSITIVITY_PREDICTIONS_FILE,
            index=False
        )


        sensitivity_hyperparameters.to_csv(
            FINAL_SENSITIVITY_HYPERPARAMETERS_FILE,
            index=False
        )


        sensitivity_performance = pd.concat(
            [
                sensitivity_performance,
                pd.DataFrame(
                    [
                        performance_record
                    ]
                )
            ],
            ignore_index=True
        )


        sensitivity_performance.to_csv(
            FINAL_SENSITIVITY_PERFORMANCE_FILE,
            index=False
        )


        completed_sensitivity_evaluations.add(
            evaluation_key
        )


        print(
            "Completed in",
            round(
                elapsed_seconds,
                1
            ),
            "seconds"
        )

        print(
            "Held-out PR-AUC:",
            round(
                held_out_pr_auc,
                4
            )
        )

        print(
            "Held-out ROC-AUC:",
            round(
                held_out_roc_auc,
                4
            )
        )


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

assert len(
    sensitivity_performance
) == EXPECTED_SENSITIVITY_EVALUATIONS, (
    "The sensitivity-performance file is incomplete."
)


assert not sensitivity_performance[
    sensitivity_key_columns
].duplicated().any(), (
    "Duplicate sensitivity evaluations were found."
)


assert len(
    sensitivity_predictions
) == EXPECTED_SENSITIVITY_PREDICTIONS, (
    "The sensitivity-prediction count is incorrect."
)


assert len(
    sensitivity_hyperparameters
) == EXPECTED_SENSITIVITY_EVALUATIONS, (
    "The hyperparameter checkpoint is incomplete."
)


sensitivity_elapsed_minutes = (
    time.perf_counter()
    - sensitivity_start_time
) / 60


print("\nFINAL SENSITIVITY FITTING COMPLETED")
print("-----------------------------------")

print(
    "Completed evaluations:",
    len(
        sensitivity_performance
    )
)

print(
    "Held-out predictions:",
    len(
        sensitivity_predictions
    )
)

print(
    "Elapsed time:",
    round(
        sensitivity_elapsed_minutes,
        2
    ),
    "minutes"
)

print("\nCheckpoint files:")
print(FINAL_SENSITIVITY_PERFORMANCE_FILE)
print(FINAL_SENSITIVITY_PREDICTIONS_FILE)
print(FINAL_SENSITIVITY_HYPERPARAMETERS_FILE)

In [ ]:
# ============================================================
# Analyze final core ridge sensitivity scenarios
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)


# ------------------------------------------------------------
# Reload completed sensitivity predictions
# ------------------------------------------------------------

sensitivity_predictions = pd.read_csv(
    FINAL_SENSITIVITY_PREDICTIONS_FILE
)


# ------------------------------------------------------------
# Average ten predictions per patient
# ------------------------------------------------------------

patient_level_sensitivity_predictions = (
    sensitivity_predictions
    .groupby(
        [
            "Scenario",
            "Description",
            "Modeling_Patient_ID"
        ],
        as_index=False
    )
    .agg(
        Observed_Outcome=(
            "Observed_Outcome",
            "first"
        ),
        Mean_Predicted_Probability=(
            "Predicted_Probability",
            "mean"
        ),
        Prediction_Repetitions=(
            "Predicted_Probability",
            "size"
        ),
        Outcome_Values=(
            "Observed_Outcome",
            "nunique"
        )
    )
)


assert (
    patient_level_sensitivity_predictions[
        "Prediction_Repetitions"
    ] == OUTER_REPEATS
).all(), (
    "At least one patient does not have ten "
    "sensitivity predictions."
)


assert (
    patient_level_sensitivity_predictions[
        "Outcome_Values"
    ] == 1
).all(), (
    "Inconsistent outcomes were found."
)


# ------------------------------------------------------------
# Primary core-ridge predictions
# ------------------------------------------------------------

primary_core_predictions = (
    patient_level_predictions.loc[
        patient_level_predictions[
            "Model"
        ] == "Core_Ridge_Logistic",
        [
            "Modeling_Patient_ID",
            "Observed_Outcome",
            "Mean_Predicted_Probability"
        ]
    ]
    .rename(
        columns={
            "Mean_Predicted_Probability": (
                "Primary_Probability"
            )
        }
    )
    .copy()
)


# ------------------------------------------------------------
# Metric function
# ------------------------------------------------------------

def calculate_final_sensitivity_metrics(
    outcomes,
    probabilities
):

    calibration_intercept, (
        calibration_slope
    ) = calculate_calibration_parameters(
        outcomes,
        probabilities
    )


    return {
        "ROC_AUC": roc_auc_score(
            outcomes,
            probabilities
        ),

        "PR_AUC": average_precision_score(
            outcomes,
            probabilities
        ),

        "Brier_Score": brier_score_loss(
            outcomes,
            probabilities
        ),

        "Calibration_Intercept": (
            calibration_intercept
        ),

        "Calibration_Slope": (
            calibration_slope
        ),

        "Mean_Predicted_Risk": float(
            np.mean(
                probabilities
            )
        ),

        "Observed_Prevalence": float(
            np.mean(
                outcomes
            )
        )
    }


# ------------------------------------------------------------
# Align primary and sensitivity predictions
# ------------------------------------------------------------

aligned_scenario_data = {}


for scenario_name in [
    "Exclude_Distant_Stage",
    "Reduced_Availability_Predictors"
]:

    scenario_data = (
        patient_level_sensitivity_predictions.loc[
            patient_level_sensitivity_predictions[
                "Scenario"
            ] == scenario_name,
            [
                "Modeling_Patient_ID",
                "Observed_Outcome",
                "Mean_Predicted_Probability"
            ]
        ]
        .rename(
            columns={
                "Observed_Outcome": (
                    "Sensitivity_Observed_Outcome"
                ),
                "Mean_Predicted_Probability": (
                    "Sensitivity_Probability"
                )
            }
        )
    )


    aligned_data = (
        primary_core_predictions
        .merge(
            scenario_data,
            on="Modeling_Patient_ID",
            how="inner",
            validate="one_to_one"
        )
        .sort_values(
            "Modeling_Patient_ID"
        )
        .reset_index(drop=True)
    )


    assert np.array_equal(
        aligned_data[
            "Observed_Outcome"
        ].to_numpy(dtype=int),
        aligned_data[
            "Sensitivity_Observed_Outcome"
        ].to_numpy(dtype=int)
    ), (
        f"Outcome mismatch for {scenario_name}."
    )


    aligned_scenario_data[
        scenario_name
    ] = aligned_data


# ------------------------------------------------------------
# Calculate point estimates
# ------------------------------------------------------------

sensitivity_point_records = []


for scenario_name, aligned_data in (
    aligned_scenario_data.items()
):

    outcomes = (
        aligned_data[
            "Observed_Outcome"
        ]
        .to_numpy(dtype=int)
    )


    for strategy_name, probability_column in [
        (
            "Primary core ridge",
            "Primary_Probability"
        ),
        (
            "Sensitivity model",
            "Sensitivity_Probability"
        )
    ]:

        probabilities = (
            aligned_data[
                probability_column
            ]
            .to_numpy(dtype=float)
        )


        metric_results = (
            calculate_final_sensitivity_metrics(
                outcomes,
                probabilities
            )
        )


        for metric_name, estimate in (
            metric_results.items()
        ):

            sensitivity_point_records.append({
                "Scenario": scenario_name,
                "Patients": len(
                    aligned_data
                ),
                "Events": int(
                    outcomes.sum()
                ),
                "Strategy": strategy_name,
                "Metric": metric_name,
                "Estimate": float(
                    estimate
                )
            })


sensitivity_point_estimates = pd.DataFrame(
    sensitivity_point_records
)


# ------------------------------------------------------------
# Patient-level bootstrap confidence intervals
# ------------------------------------------------------------

NUMBER_OF_FINAL_SENSITIVITY_BOOTSTRAPS = 2000

FINAL_SENSITIVITY_BOOTSTRAP_SEED = (
    RANDOM_SEED + 50000
)


sensitivity_random_generator = (
    np.random.default_rng(
        FINAL_SENSITIVITY_BOOTSTRAP_SEED
    )
)


sensitivity_bootstrap_records = []
sensitivity_difference_records = []


for scenario_name, aligned_data in (
    aligned_scenario_data.items()
):

    outcomes = (
        aligned_data[
            "Observed_Outcome"
        ]
        .to_numpy(dtype=int)
    )

    primary_probabilities = (
        aligned_data[
            "Primary_Probability"
        ]
        .to_numpy(dtype=float)
    )

    sensitivity_probabilities = (
        aligned_data[
            "Sensitivity_Probability"
        ]
        .to_numpy(dtype=float)
    )


    number_of_patients = len(
        aligned_data
    )


    for bootstrap_number in range(
        1,
        NUMBER_OF_FINAL_SENSITIVITY_BOOTSTRAPS + 1
    ):

        sampled_indices = (
            sensitivity_random_generator.choice(
                number_of_patients,
                size=number_of_patients,
                replace=True
            )
        )


        bootstrap_outcomes = (
            outcomes[
                sampled_indices
            ]
        )


        if np.unique(
            bootstrap_outcomes
        ).size < 2:

            continue


        primary_metrics = (
            calculate_final_sensitivity_metrics(
                bootstrap_outcomes,
                primary_probabilities[
                    sampled_indices
                ]
            )
        )

        sensitivity_metrics = (
            calculate_final_sensitivity_metrics(
                bootstrap_outcomes,
                sensitivity_probabilities[
                    sampled_indices
                ]
            )
        )


        for strategy_name, metric_values in [
            (
                "Primary core ridge",
                primary_metrics
            ),
            (
                "Sensitivity model",
                sensitivity_metrics
            )
        ]:

            for metric_name, estimate in (
                metric_values.items()
            ):

                sensitivity_bootstrap_records.append({
                    "Scenario": scenario_name,
                    "Strategy": strategy_name,
                    "Bootstrap": bootstrap_number,
                    "Metric": metric_name,
                    "Estimate": float(
                        estimate
                    )
                })


        # Positive values mean sensitivity-model improvement.

        for metric_name in [
            "ROC_AUC",
            "PR_AUC"
        ]:

            difference = (
                sensitivity_metrics[
                    metric_name
                ]
                - primary_metrics[
                    metric_name
                ]
            )


            sensitivity_difference_records.append({
                "Scenario": scenario_name,
                "Bootstrap": bootstrap_number,
                "Metric": metric_name,
                "Improvement_Sensitivity_Minus_Primary": (
                    float(
                        difference
                    )
                )
            })


        brier_improvement = (
            primary_metrics[
                "Brier_Score"
            ]
            - sensitivity_metrics[
                "Brier_Score"
            ]
        )


        sensitivity_difference_records.append({
            "Scenario": scenario_name,
            "Bootstrap": bootstrap_number,
            "Metric": "Brier_Score",
            "Improvement_Sensitivity_Minus_Primary": (
                float(
                    brier_improvement
                )
            )
        })


    print(
        scenario_name,
        "| completed bootstrap samples:",
        NUMBER_OF_FINAL_SENSITIVITY_BOOTSTRAPS
    )


sensitivity_bootstrap_estimates = (
    pd.DataFrame(
        sensitivity_bootstrap_records
    )
)

sensitivity_bootstrap_differences = (
    pd.DataFrame(
        sensitivity_difference_records
    )
)


# ------------------------------------------------------------
# Confidence intervals for performance estimates
# ------------------------------------------------------------

sensitivity_interval_table = (
    sensitivity_bootstrap_estimates
    .groupby(
        [
            "Scenario",
            "Strategy",
            "Metric"
        ]
    )["Estimate"]
    .agg(
        CI_Lower=lambda values: np.percentile(
            values,
            2.5
        ),
        CI_Upper=lambda values: np.percentile(
            values,
            97.5
        ),
        Successful_Bootstraps="count"
    )
    .reset_index()
)


final_sensitivity_summary = (
    sensitivity_point_estimates
    .merge(
        sensitivity_interval_table,
        on=[
            "Scenario",
            "Strategy",
            "Metric"
        ],
        how="left",
        validate="one_to_one"
    )
)


final_sensitivity_summary[
    "Estimate_95_CI"
] = (
    final_sensitivity_summary[
        "Estimate"
    ].map(
        lambda value: f"{value:.3f}"
    )
    + " ("
    + final_sensitivity_summary[
        "CI_Lower"
    ].map(
        lambda value: f"{value:.3f}"
    )
    + " to "
    + final_sensitivity_summary[
        "CI_Upper"
    ].map(
        lambda value: f"{value:.3f}"
    )
    + ")"
)


# ------------------------------------------------------------
# Paired differences
# ------------------------------------------------------------

final_sensitivity_differences = (
    sensitivity_bootstrap_differences
    .groupby(
        [
            "Scenario",
            "Metric"
        ]
    )[
        "Improvement_Sensitivity_Minus_Primary"
    ]
    .agg(
        Median_Improvement="median",
        CI_Lower=lambda values: np.percentile(
            values,
            2.5
        ),
        CI_Upper=lambda values: np.percentile(
            values,
            97.5
        ),
        Probability_Sensitivity_Better=(
            lambda values: np.mean(
                values > 0
            )
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Manuscript-format comparison table
# ------------------------------------------------------------

final_sensitivity_manuscript_table = (
    final_sensitivity_summary
    .pivot(
        index=[
            "Scenario",
            "Patients",
            "Events",
            "Metric"
        ],
        columns="Strategy",
        values="Estimate_95_CI"
    )
    .reset_index()
)


key_sensitivity_metrics = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Calibration_Intercept",
    "Calibration_Slope",
    "Mean_Predicted_Risk",
    "Observed_Prevalence"
]


final_sensitivity_manuscript_table = (
    final_sensitivity_manuscript_table.loc[
        final_sensitivity_manuscript_table[
            "Metric"
        ].isin(
            key_sensitivity_metrics
        )
    ]
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Save final sensitivity results
# ------------------------------------------------------------

FINAL_SENSITIVITY_RESULTS_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "core_ridge_final_sensitivity_results.xlsx"
)

FINAL_SENSITIVITY_BOOTSTRAP_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "core_ridge_final_sensitivity_bootstrap.csv"
)


sensitivity_bootstrap_estimates.to_csv(
    FINAL_SENSITIVITY_BOOTSTRAP_FILE,
    index=False
)


with pd.ExcelWriter(
    FINAL_SENSITIVITY_RESULTS_FILE,
    engine="openpyxl"
) as writer:

    final_sensitivity_summary.to_excel(
        writer,
        sheet_name="Performance",
        index=False
    )

    final_sensitivity_manuscript_table.to_excel(
        writer,
        sheet_name="Manuscript Table",
        index=False
    )

    final_sensitivity_differences.to_excel(
        writer,
        sheet_name="Paired Differences",
        index=False
    )

    patient_level_sensitivity_predictions.to_excel(
        writer,
        sheet_name="Patient Predictions",
        index=False
    )


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("FINAL CORE RIDGE SENSITIVITY RESULTS")
print("------------------------------------")

display(
    final_sensitivity_manuscript_table
)


print("\nPAIRED DIFFERENCES")
print("------------------")

display(
    final_sensitivity_differences.round(4)
)


print("\nFiles saved:")
print(FINAL_SENSITIVITY_RESULTS_FILE)
print(FINAL_SENSITIVITY_BOOTSTRAP_FILE)

In [ ]:
# ============================================================
# Recreate calibration and decision-curve panels at equal size
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ------------------------------------------------------------
# File locations
# ------------------------------------------------------------

VALIDATION_CURVE_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_validation_curve_data.xlsx"
)

DECISION_CURVE_FILE = (
    MODEL_RESULTS_DIRECTORY
    / "unweighted_decision_curve_data.xlsx"
)

# Use the existing figure directory if it was defined previously.
if "FIGURES_DIRECTORY" not in globals():
    FIGURES_DIRECTORY = (
        MODEL_RESULTS_DIRECTORY.parent
        / "Figures"
    )

FIGURES_DIRECTORY = Path(FIGURES_DIRECTORY)
FIGURES_DIRECTORY.mkdir(parents=True, exist_ok=True)


assert VALIDATION_CURVE_FILE.exists(), (
    f"File not found: {VALIDATION_CURVE_FILE}"
)

assert DECISION_CURVE_FILE.exists(), (
    f"File not found: {DECISION_CURVE_FILE}"
)


# ------------------------------------------------------------
# Read the saved numerical results
# ------------------------------------------------------------

calibration_data = pd.read_excel(
    VALIDATION_CURVE_FILE,
    sheet_name="Calibration Curves",
)

decision_data = pd.read_excel(
    DECISION_CURVE_FILE,
    sheet_name="All Strategies",
)


# ------------------------------------------------------------
# Consistent model names, colours and line styles
# ------------------------------------------------------------

model_order = [
    "Core_Ridge_Logistic",
    "Elastic_Net_Logistic",
    "Random_Forest",
    "Gradient_Boosting",
]

model_labels = {
    "Core_Ridge_Logistic": "Core ridge logistic",
    "Elastic_Net_Logistic": "Elastic-net logistic",
    "Random_Forest": "Random forest",
    "Gradient_Boosting": "Gradient boosting",
}

model_colours = {
    "Core_Ridge_Logistic": "#0072B2",
    "Elastic_Net_Logistic": "#009E73",
    "Random_Forest": "#D55E00",
    "Gradient_Boosting": "#CC79A7",
}

model_line_styles = {
    "Core_Ridge_Logistic": "-",
    "Elastic_Net_Logistic": "--",
    "Random_Forest": "-.",
    "Gradient_Boosting": ":",
}


# ------------------------------------------------------------
# Create the matched two-panel figure
# ------------------------------------------------------------

figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(14, 7),
)

calibration_axis = axes[0]
decision_axis = axes[1]


# ============================================================
# Panel A: Calibration
# ============================================================

for model_name in model_order:

    model_data = (
        calibration_data.loc[
            calibration_data["Model"].eq(model_name)
        ]
        .sort_values("Mean_Predicted_Probability")
    )

    x_values = model_data[
        "Mean_Predicted_Probability"
    ].to_numpy(dtype=float)

    y_values = model_data[
        "Observed_Event_Proportion"
    ].to_numpy(dtype=float)

    brier_score = float(
        model_data["Brier_Score"].iloc[0]
    )

    calibration_slope = float(
        model_data["Calibration_Slope"].iloc[0]
    )

    calibration_axis.plot(
        x_values,
        y_values,
        color=model_colours[model_name],
        linestyle=model_line_styles[model_name],
        linewidth=2.2,
        marker="o",
        markersize=4.5,
        markerfacecolor="white",
        markeredgewidth=1.3,
        label=(
            f"{model_labels[model_name]} "
            f"(Brier = {brier_score:.3f}; "
            f"slope = {calibration_slope:.2f})"
        ),
    )


calibration_axis.plot(
    [0.0, 0.30],
    [0.0, 0.30],
    color="#555555",
    linestyle="--",
    linewidth=1.7,
    label="Perfect calibration",
)

calibration_axis.set_xlim(0.0, 0.30)
calibration_axis.set_ylim(0.0, 0.30)

calibration_axis.set_xlabel(
    "Mean predicted recurrence probability",
    fontsize=11,
)

calibration_axis.set_ylabel(
    "Observed recurrence proportion",
    fontsize=11,
)

calibration_axis.set_title(
    "A. Calibration",
    fontsize=13,
    fontweight="bold",
    pad=12,
)

calibration_axis.legend(
    loc="upper left",
    frameon=True,
    framealpha=1.0,
    edgecolor="black",
    fontsize=8,
)

calibration_axis.tick_params(
    axis="both",
    labelsize=9,
)

calibration_axis.grid(False)


# ============================================================
# Panel B: Decision-curve analysis
# ============================================================

for model_name in model_order:

    model_data = (
        decision_data.loc[
            decision_data["Strategy"].eq(model_name)
        ]
        .sort_values("Threshold_Probability")
    )

    threshold_values = model_data[
        "Threshold_Probability"
    ].to_numpy(dtype=float)

    net_benefit_values = model_data[
        "Net_Benefit"
    ].to_numpy(dtype=float)

    decision_axis.plot(
        threshold_values,
        net_benefit_values,
        color=model_colours[model_name],
        linestyle=model_line_styles[model_name],
        linewidth=2.3,
        label=model_labels[model_name],
    )


# Treat-all strategy
treat_all_data = (
    decision_data.loc[
        decision_data["Strategy"].eq("Treat all")
    ]
    .sort_values("Threshold_Probability")
)

decision_axis.plot(
    treat_all_data[
        "Threshold_Probability"
    ].to_numpy(dtype=float),
    treat_all_data[
        "Net_Benefit"
    ].to_numpy(dtype=float),
    color="#666666",
    linestyle="--",
    linewidth=1.8,
    label="Treat all",
)


# Treat-none strategy
treat_none_data = (
    decision_data.loc[
        decision_data["Strategy"].eq("Treat none")
    ]
    .sort_values("Threshold_Probability")
)

decision_axis.plot(
    treat_none_data[
        "Threshold_Probability"
    ].to_numpy(dtype=float),
    treat_none_data[
        "Net_Benefit"
    ].to_numpy(dtype=float),
    color="black",
    linestyle="-",
    linewidth=1.6,
    label="Treat none",
)


decision_axis.set_xlim(0.01, 0.30)
decision_axis.set_ylim(-0.02, 0.06)

decision_axis.set_xlabel(
    "Threshold probability",
    fontsize=11,
)

decision_axis.set_ylabel(
    "Net benefit",
    fontsize=11,
)

decision_axis.set_title(
    "B. Decision-curve analysis",
    fontsize=13,
    fontweight="bold",
    pad=12,
)

decision_axis.legend(
    loc="upper right",
    frameon=True,
    framealpha=1.0,
    edgecolor="black",
    fontsize=8,
)

decision_axis.tick_params(
    axis="both",
    labelsize=9,
)

decision_axis.grid(False)


# ------------------------------------------------------------
# Make the two plotting areas exactly the same shape
# ------------------------------------------------------------

for axis in axes:

    axis.set_box_aspect(1)

    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)


# ------------------------------------------------------------
# Final layout and saving
# ------------------------------------------------------------

figure.tight_layout(
    rect=[0.01, 0.02, 0.99, 0.98],
    w_pad=3.0,
)

combined_png_file = (
    FIGURES_DIRECTORY
    / "unweighted_calibration_and_decision_curve_matched.png"
)

combined_pdf_file = (
    FIGURES_DIRECTORY
    / "unweighted_calibration_and_decision_curve_matched.pdf"
)

figure.savefig(
    combined_png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.08,
)

figure.savefig(
    combined_pdf_file,
    bbox_inches="tight",
    pad_inches=0.08,
)

plt.show()


print("Matched two-panel figure saved:")
print(combined_png_file)
print(combined_pdf_file)